In [ ]:
!pip install arch yfinance xgboost lightgbm catboost shap numpy_financial
!pip install --upgrade dask distributed


import importlib



import os
from google.colab import userdata

# 1. Fetch the secret from the Colab Secrets tab
# 2. Inject it into the environment variables for the current session
os.environ["FRED_API_KEY"] = userdata.get('FRED_API_KEY')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.5 MB/s eta 0:00:00
  Attempting uninstall: dask
    Found existing installation: dask 2026.1.1
    Uninstalling dask-2026.1.1:
      Successfully uninstalled dask-2026.1.1
  Attempting uninstall: distributed
    Found existing installation: distributed 2026.1.1
    Uninstalling distributed-2026.1.1:
      Successfully uninstalled distributed-2026.1.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rapids-dask-dependency 26.2.0 requires dask==2026.1.1, but you have dask 2026.3.0 which is incompatible.
rapids-dask-dependency 26.2.0 requires distributed==2026.1.1, but you ha

In [ ]:
!pip install xgboost lightgbm catboost shap yfinance scikit-learn numpy pandas torch scipy arch numpy_financial
!export FRED_API_KEY="ccb67ba570ac152dc7930a216488320d"

In [ ]:
%%writefile ml_credit_engine.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import shap

# [CLOUD FIX]: Restored the elite gradient boosting trio. Linux handles OpenMP perfectly.
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import KFold
from sklearn.model_selection import TimeSeriesSplit


class NeuralCox(nn.Module):
    """
    Module 3: AlphaCredit.ai DeepHazard Neural Cox Model
    """
    def __init__(self, static_input_dim: int, time_input_dim: int):
        super(NeuralCox, self).__init__()
        self.log_baseline_hazard = nn.Parameter(torch.zeros(1))
        self.cox_linear = nn.Linear(static_input_dim, 1)
        self.lstm = nn.LSTM(time_input_dim, 64, batch_first=True)
        self.neural_head = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, X_static, X_time):
        cox_part = torch.exp(self.cox_linear(X_static))
        lstm_out, (h_n, c_n) = self.lstm(X_time)
        last_hidden = lstm_out[:, -1, :]
        neural_part = self.neural_head(last_hidden)
        baseline = F.softplus(self.log_baseline_hazard)   # strictly positive
        hazard = baseline * cox_part * torch.exp(neural_part)
        return hazard


def train_credit_super_learner(X_train: pd.DataFrame, y_train: pd.Series):
    """
    Constructs and fits an institutional Super-Learner Stacked Ensemble using strictly
    chronological TimeSeriesSplit to prevent look-ahead bias in the meta-learner.
    """
    _USE_GPU = torch.cuda.is_available()
    base_learners = [
        ('xgb', XGBClassifier(eval_metric='logloss', random_state=42, tree_method='hist', device='cuda' if _USE_GPU else 'cpu')),
        ('lgb', LGBMClassifier(verbose=-1, random_state=42, device='gpu' if _USE_GPU else 'cpu')),
        ('cat', CatBoostClassifier(verbose=0, random_state=42, task_type='GPU' if _USE_GPU else 'CPU')),
    ]

    tss = TimeSeriesSplit(n_splits=5)
    X_arr = X_train.values
    y_arr = y_train.values

    # Step 1: Generate valid out-of-fold predictions chronologically
    oof_meta = np.zeros((len(X_arr), len(base_learners)))
    valid_mask = np.zeros(len(X_arr), dtype=bool)

    for train_idx, test_idx in tss.split(X_arr):
        for col_idx, (name, model) in enumerate(base_learners):
            m = model.__class__(**model.get_params())
            m.fit(X_arr[train_idx], y_arr[train_idx])
            oof_meta[test_idx, col_idx] = m.predict_proba(X_arr[test_idx])[:, 1]
        valid_mask[test_idx] = True

    # Step 2: Train meta-learner ONLY on the valid out-of-fold predictions
    lr = LogisticRegression()
    lr.fit(oof_meta[valid_mask], y_arr[valid_mask])

    # Step 3: Train final base estimators on the FULL dataset for live inference
    final_base_estimators = []
    for name, model in base_learners:
        m = model.__class__(**model.get_params())
        m.fit(X_arr, y_arr) # SR 11-7: Model must utilise all available historical information
        final_base_estimators.append((name, m))

    # Step 4: Wrap in a custom pipeline class compatible with generate_shap_explanations
    class ChronologicalStacker:
        def __init__(self, base, meta, feature_names):
            self.named_estimators_ = {n: m for n, m in base}
            self._base = base
            self._meta = meta
            self.feature_names = feature_names

        def predict_proba(self, X):
            if isinstance(X, pd.DataFrame): X = X.values
            meta_input = np.column_stack([m.predict_proba(X)[:, 1] for _, m in self._base])
            return self._meta.predict_proba(meta_input)

    return ChronologicalStacker(final_base_estimators, lr, X_train.columns.tolist())


def generate_shap_explanations(stacked_model, X_test: pd.DataFrame) -> pd.DataFrame:
    """
    Computes an ensemble-weighted SHAP approximation using all base learners.
    """
    # 1. Extract the weights the LogisticRegression meta-learner assigned to each model
    meta_weights = stacked_model._meta.coef_[0]
    weight_xgb, weight_lgb, weight_cat = meta_weights / np.sum(np.abs(meta_weights))

    # 2. Extract models
    xgb_model = stacked_model.named_estimators_['xgb']
    lgb_model = stacked_model.named_estimators_['lgb']
    cat_model = stacked_model.named_estimators_['cat']

    # 3. Compute SHAP for all three
    shap_xgb = shap.TreeExplainer(xgb_model).shap_values(X_test.values)
    shap_lgb = shap.TreeExplainer(lgb_model).shap_values(X_test.values)
    shap_cat = shap.TreeExplainer(cat_model).shap_values(X_test.values)

    # Normalize shapes if needed (Binary classification sometimes returns lists of arrays)
    if isinstance(shap_xgb, list): shap_xgb = shap_xgb[1]
    if isinstance(shap_lgb, list): shap_lgb = shap_lgb[1]
    if isinstance(shap_cat, list): shap_cat = shap_cat[1]

    # 4. Create the weighted ensemble SHAP array
    ensemble_shap = (shap_xgb * weight_xgb) + (shap_lgb * weight_lgb) + (shap_cat * weight_cat)

    # 5. Extract Top 3 Drivers as usual
    features = X_test.columns.tolist()
    explanations = []

    for i in range(len(X_test)):
        sv = ensemble_shap[i]
        top_indices = np.argsort(np.abs(sv))[-3:][::-1]
        reasons = [f"{features[j]} ({'+' if sv[j] > 0 else ''}{sv[j]:.2f})" for j in top_indices]
        explanations.append(" | ".join(reasons))

    return pd.DataFrame({'Top_3_SHAP_Drivers': explanations}, index=X_test.index)

Writing ml_credit_engine.py


In [ ]:
%%writefile main.py
import sys
import json
import torch
import numpy as np
import pandas as pd
import warnings

import spread
import default_rate_analysis
import analysis
import volatility_trading
import pure_volatility_trading
import volatility_matrix
import dealer_markup
import bonds_EM_data
import bonds_EUR_data
import global_private_credit

import private_credit_data
import credit_ratios
import main_alpha
import ml_credit_engine
import risk_simulation_engine
import private_credit


def execute_volatility_spectrum(df_raw, df_spreads, r_daily, retail_markup, is_private, ts, hd, region_name):
    """Generates the Heatmaps for BOTH Liquid Volatility and Spread Trading"""
    thresholds = [95, 90, 80, 70, 60, 50]
    skip_cols = ["Risk_Free", "EUR_Risk_Free_10Y"]
    base_kappa = 25.0

    assets = [col for col in df_raw.columns if col not in skip_cols]
    volatility_matrix.generate_yield_tables(df_raw[assets], hd, region_name)

    all_pure_results = []
    for pct in thresholds:
        print(f"\n  [SIMULATION] Running {pct}th Percentile Volatility Threshold (PURE)...")
        results = pure_volatility_trading.analyze_strategy(
            df_raw=df_raw, r_daily=r_daily, retail_markup=retail_markup,
            percentile_override=pct, kappa_override=base_kappa, trade_size=ts, hold_days=hd
        )
        for res in results:
            res["Percentile"] = pct
            all_pure_results.append(res)

    volatility_matrix.generate_spectrum_matrices(assets, thresholds, all_pure_results, hd, f"{region_name} (Pure)")

    if df_spreads is not None and not df_spreads.empty:
        spread_pairs = list(df_spreads.columns)
        all_spread_results = []

        for pct in thresholds:
            print(f"\n  [SIMULATION] Running {pct}th Percentile Volatility Threshold (SPREAD)...")
            results = volatility_trading.analyze_strategy(
                df_spreads=df_spreads, df_raw=df_raw, r_daily=r_daily, retail_markup=retail_markup,
                percentile_override=pct, kappa_override=base_kappa, trade_size=ts, hold_days=hd
            )
            for res in results:
                res["Percentile"] = pct
                all_spread_results.append(res)

        volatility_matrix.generate_spectrum_matrices(spread_pairs, thresholds, all_spread_results, hd, f"{region_name} (Spread)")


def main():
    print("=" * 75)
    print("  GLOBAL CREDIT VOLATILITY & AI ENGINE (CLOUD INSTANCE)")
    print("=" * 75)
    print("  [1] USA Public Corporate Bonds (FRED) - FULL PIPELINE")
    print("  [2] EUR Public Corporate Bonds (ETFs) - MATRIX ONLY")
    print("  [3] Global Private Credit Pipeline (AlphaCredit Modules 1 & 2)")
    print("  [4] Emerging Markets (USD & Local) - MATRIX ONLY")
    print("  [5] Liquid ML Credit Scorecard (XGBoost + SHAP Matrix)")
    print("  [6] Private Credit Super-Learner & Risk Sim (Modules 3 & 4)")
    print("-" * 75)

    market = input("  Select Market Architecture [1/2/3/4/5/6]: ").strip()

    if market in ["1", "2", "4"]:
        ts = pure_volatility_trading.get_trade_size()
        hd = pure_volatility_trading.get_hold_days()
    elif market in ["3", "5", "6"]:
        ts = None
        hd = None
    else:
        print("  [ERROR] Invalid selection. Terminating.")
        sys.exit(1)

    is_private = False
    df_raw = pd.DataFrame()
    df_spreads = None

    if market == "1":
        region_str = "USA Public"
        new_bond_returns = spread.fetch_all_bond_returns(spread.FETCH_ORDER, spread.US_BOND_SERIES)
        merged = spread.merge_timeseries_dict("USA_bond_returns_by_grade.json", new_bond_returns)
        spreads = spread.convert_to_bps(spread.compute_spreads_pct(merged, spread.get_adjacent_pairs(spread.SPREAD_ORDER)))
        spread.save_spread_jsons(merged, spreads)

        default_rate_analysis.main()
        analysis.main()

        # --- THE ULTIMATE FIX: Call prepare_pristine_data directly to guarantee shape alignment ---
        df_raw, df_spreads = volatility_trading.prepare_pristine_data("USA_bond_returns_by_grade.json")

    elif market == "2":
        region_str = "Europe"
        df_raw = bonds_EUR_data.get_eur_market_data()
        dealer_markup.run()

    elif market == "3":
        print("\n  [ROUTING] Initiating Global Private Credit Underwriting Pipeline...")
        global_private_credit.generate_lending_matrix()

        print("\n  [ALPHACREDIT.AI] Booting Module 1: Unsmoothing Engine...")
        pc_engine = private_credit_data.PrivateCreditDataPrep()
        example_smoothed_returns = pd.Series(
            [0.015, 0.017, 0.016, 0.018, 0.019, 0.017, 0.020, 0.021, 0.019]
        )

        try:
            unsmoothed_series = private_credit.dynamic_unsmooth_private_credit(
                example_smoothed_returns, max_lags=4
            )
            print("    [MATH] MA(q) Generalized Geltner filter applied.")
        except Exception as e:
            warnings.warn(
                f"MA(q) filter failed ({e}). Falling back to AR(1).", RuntimeWarning, stacklevel=2
            )
            unsmoothed_series = pc_engine.unsmooth_returns(example_smoothed_returns)

        adj_sharpe = pc_engine.calculate_adj_sharpe(0.12, 0.04, unsmoothed_series)
        print(f"    -> Illiquidity-Adjusted Sharpe Ratio: {adj_sharpe:.2f}")


        print("\n  [ALPHACREDIT.AI] Booting Module 2: Credit Ratios Engine...")
        ratios_engine = credit_ratios.CreditRatiosEngine()
        aiy = ratios_engine.calculate_all_in_yield(r_base=0.05, f_base=0.04, s_margin=0.06, oid=0.02, f_upfront=0.01)

        print(f"    -> Deal All-In Yield (AIY): {aiy * 100:.2f}%")
        print(f"    -> Illiquidity-Adjusted Sharpe Ratio: {adj_sharpe:.2f}")
        return

    elif market == "4":
        region_str = "Emerging Markets"
        df_raw = bonds_EM_data.get_em_market_data()
        dealer_markup.run()

    elif market == "5":
        print("\n  [ROUTING] Booting Liquid ML Alpha Engine...")
        main_alpha.run()
        print("\n  [SYSTEM] Liquid ML Engine Session Terminated Safely.")
        return

    elif market == "6":
        print("\n  [ROUTING] Booting Institutional DeepHazard Neural Cox Pipeline...")
        static_dim = 12
        time_dim = 6
        seq_length = 24

        cox_model = ml_credit_engine.NeuralCox(static_input_dim=static_dim, time_input_dim=time_dim)
        X_static_tensor = torch.rand(500, static_dim)
        X_time_tensor = torch.rand(500, seq_length, time_dim)

        hazards = cox_model(X_static_tensor, X_time_tensor)
        print(f"    -> Neural Cox Forward Pass Complete. Hazards calculated for {hazards.shape[0]} deals.")

        feature_names = ["CQI", "LYGI", "LTV", "FCCR", "OAS_Z", "Sharpe_252"] + [f"Feat_{i}" for i in range(6, static_dim)]
        X_tabular = pd.DataFrame(X_static_tensor.numpy(), columns=feature_names)
        y_target = pd.Series(np.random.randint(0, 2, size=500))

        super_learner = ml_credit_engine.train_credit_super_learner(X_tabular, y_target)
        X_latest = X_tabular.tail(5).copy()
        shap_explanations = ml_credit_engine.generate_shap_explanations(super_learner, X_latest)

        X_latest['ML_Outperform_Prob'] = super_learner.predict_proba(X_latest)[:, 1] * 100

        print("\n  [ALPHACREDIT.AI] Booting Module 4: Front-End UI DataFrames & Risk Sim...")
        simulator = risk_simulation_engine.AgenticRiskSimulator()

        simulated_cf = np.array([50000, 50000, 50000, 1050000])
        pd_curve = np.array([0.02, 0.05, 0.09, 0.15])

        try:
            live_rf, _ = risk_simulation_engine.fetch_live_discount_rate(
                n_periods=len(pd_curve), rate_type="risk_free"
            )
        except RuntimeError:
            live_rf = 0.04  # Emergency fallback only
            print("    [WARNING] Risk-free fetch failed. Using 4% emergency fallback for recovery calc.")

        deal_spread = 0.0650
        avg_pd = np.mean(pd_curve)
        credit_spread = max(deal_spread - live_rf, 0.0001)
        implied_lgd = credit_spread / max(avg_pd, 0.0001)
        implied_lgd = min(implied_lgd, 1.0)
        synthetic_recovery = float(np.clip(1.0 - implied_lgd, 0.05, 0.85))

        print(f"    -> Live risk-free: {live_rf:.4%} | Credit spread: {credit_spread:.4%} | "
              f"Implied LGD: {implied_lgd:.4%} | Synthetic Recovery: {synthetic_recovery:.1%}")

        print(f"    -> Synthetic Market-Implied Recovery Rate Calculated: {synthetic_recovery * 100:.1f}%")

        ev, cvar_95, rate_meta = simulator.simulate_ev_cvar(
            cash_flows=simulated_cf,
            pd_curve=pd_curve,
            recovery_rate=synthetic_recovery,
            rate_type="credit_adjusted",
            credit_tier="HY"
        )
        print(f"    -> Monte Carlo Simulation (10k paths) Complete: Expected Value = ${ev:,.2f} | 95% CVaR = ${cvar_95:,.2f}")
        print(f"    -> Live Discount Rate Applied: {rate_meta['total_rate']*100:.2f}% (Source: {rate_meta['tenor_years']}Y Treasury + {rate_meta['credit_tier']} OAS)")

        mock_asset_names = [f"Private_Deal_{i}" for i in X_latest.index]
        mock_cqi_scores = X_latest['CQI'].values

        ml_scorecard_df = simulator.generate_ml_scorecard(
            asset_names=mock_asset_names,
            predictions=X_latest['ML_Outperform_Prob'].values,
            cqi_scores=mock_cqi_scores,
            shap_df=shap_explanations
        )

        vol_df = pd.DataFrame({
            'Pure_Bond_Return': np.random.normal(0.05, 0.12, 1000),
            'Yield_Spread_Return': np.random.normal(0.02, 0.08, 1000)
        })
        volatility_matrix_out = simulator.generate_volatility_spectrum(vol_df)

        print("\n" + "="*75)
        print(" FINAL ALPHA CREDIT ML SCORECARD (MODULE 3 & 4 INTEGRATION)")
        print("="*75)
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 150)
        print(ml_scorecard_df.to_string(index=False))

        print("\n" + "="*75)
        print(" VOLATILITY SPECTRUM MATRIX")
        print("="*75)
        print(volatility_matrix_out.to_string(index=False))

        print("\n  [SYSTEM] DeepHazard Neural Cox Pipeline execution complete.")
        return

    if df_raw.empty:
        print(f"\n  [CRITICAL] Data pipeline for {region_str} returned 0 observations. Aborting simulation.")
        return

    try:
        with open("dealer_markup.json", "r") as f:
            retail_markup = pd.Series(json.load(f)["Dealer_Multiplier"])
            retail_markup.index = pd.to_datetime(retail_markup.index)
            retail_markup = retail_markup.reindex(df_raw.index).ffill().bfill()
    except Exception:
        retail_markup = pd.Series(1.0, index=df_raw.index)

    r_daily = pd.Series(0.04, index=df_raw.index)
    execute_volatility_spectrum(df_raw, df_spreads, r_daily, retail_markup, is_private, ts, hd, region_str)

if __name__ == "__main__":
    main()

Writing main.py


In [ ]:
%%writefile private_credit_data.py
import numpy as np
import pandas as pd
from scipy.optimize import newton
import numpy_financial as npf

class PrivateCreditDataPrep:
    """Module 1: AlphaCredit.ai - NAV unsmoothing and return mechanics."""
    @staticmethod
    def calculate_moic(realized_cf: float, unrealized_value: float, total_drawn_capital: float) -> float:
        if total_drawn_capital == 0: return np.nan
        return (realized_cf + unrealized_value) / total_drawn_capital

    @staticmethod
    def calculate_irr(cash_flows: list) -> float:
        """
        Calculates the IRR with full multiple-root detection.

        Private credit cash flows (capital calls interspersed with distributions)
        can have multiple sign changes, producing multiple mathematically valid IRRs.
        This function detects that condition and refuses to return an ambiguous result,
        forcing analyst review rather than silently returning a wrong rate.
        """
        cfs = np.array(cash_flows, dtype=float)

        # Count sign changes among non-zero cash flows (Descartes' Rule prerequisite)
        nonzero_signs = np.sign(cfs[cfs != 0])
        sign_changes = int(np.sum(np.abs(np.diff(nonzero_signs)) > 0))

        if sign_changes > 1:
            # Compute ALL roots via companion matrix to enumerate valid IRRs.
            # NPV polynomial coefficients in descending degree: CF_N, ..., CF_1, CF_0
            coeffs = cfs[::-1].copy()
            roots = np.roots(coeffs)

            # Convert polynomial roots z to interest rates r: z = 1/(1+r) -> r = 1/z - 1
            valid_rates = []
            for z in roots:
                if np.abs(z.imag) < 1e-10 and z.real > 1e-12:
                    r = (1.0 / z.real) - 1.0
                    if r > -1.0:  # economically valid: rate > -100%
                        valid_rates.append(round(r, 10))

            unique_rates = sorted(set(valid_rates))

            if len(unique_rates) > 1:
                # Multiple valid IRRs: no single unambiguous answer exists.
                # Return nan to force analyst review. Do NOT silently pick one.
                import warnings
                warnings.warn(
                    f"Multiple valid IRRs detected: {[f'{r:.4%}' for r in unique_rates]}. "
                    f"Cash flow has {sign_changes} sign changes. "
                    f"Use MIRR (Modified IRR) with explicit reinvestment rate assumption. "
                    f"Returning np.nan to prevent silent misallocation.",
                    RuntimeWarning,
                    stacklevel=2
                )
                return np.nan
            elif len(unique_rates) == 1:
                return unique_rates[0]
            else:
                return np.nan

        # Single sign change: npf.irr is unambiguous and correct.
        try:
            res = npf.irr(cash_flows)
            return float(res) if not np.isnan(res) else np.nan
        except Exception:
            return np.nan

    @staticmethod
    def unsmooth_returns(returns_series: pd.Series) -> pd.Series:
      if not isinstance(returns_series, pd.Series):
          returns_series = pd.Series(returns_series)

      # Rolling expanding alpha with shift(1) — eliminates look-ahead bias.
      # Associate's approach (expanding + shift) is correct and adopted here.
      rolling_alpha = (
          returns_series
          .expanding(min_periods=6)
          .apply(lambda x: x.autocorr(lag=1), raw=False)
          .shift(1)
     )

      #  Bounds cap — omitted from associate fix ("omitted for brevity").
      # Required to prevent denominator collapse from near-unity alpha (Finding E).
      rolling_alpha_bounded = rolling_alpha.clip(lower=ALPHA_MIN, upper=ALPHA_CAP)

      # Preserve early observations where alpha is unavailable (insufficient history).
      # Associate fix drops these; this fix substitutes raw returns instead.
      use_raw_mask = rolling_alpha.isna() | (rolling_alpha.abs() < ALPHA_MIN)
      unsmoothed = returns_series.copy()
      valid_mask = ~use_raw_mask
      a = rolling_alpha_bounded[valid_mask]
      r = returns_series[valid_mask]
      r_lag = returns_series.shift(1)[valid_mask]
      unsmoothed[valid_mask] = (r - (a * r_lag)) / (1 - a)

      # Post-filter sanity check (Finding E).
      if unsmoothed.dropna().std() > returns_series.std() * 10:
          import warnings
          warnings.warn(
              "Unsmoothed series std exceeds 10x original. Returning raw series.",
              RuntimeWarning, stacklevel=2
          )
          return returns_series

      return unsmoothed.dropna()

    @staticmethod
    def calculate_adj_sharpe(portfolio_return: float, risk_free_rate: float, unsmoothed_returns: pd.Series) -> float:
        if not isinstance(unsmoothed_returns, pd.Series):
            unsmoothed_returns = pd.Series(unsmoothed_returns)
        sigma_unsmoothed = float(unsmoothed_returns.std())
        if pd.isna(sigma_unsmoothed) or abs(sigma_unsmoothed) < 1e-9: return np.nan
        return (portfolio_return - risk_free_rate) / sigma_unsmoothed

Writing private_credit_data.py


In [ ]:
%%writefile credit_ratios.py
import numpy as np
import pandas as pd

class CreditRatiosEngine:
    """Module 2: AlphaCredit.ai - Yield, Covenants, and Proprietary Alpha Indices."""

    @staticmethod
    def calculate_all_in_yield(r_base: float, f_base: float, s_margin: float, oid: float, f_upfront: float, t_expected: float = 3.0) -> float:
        r_coupon = max(r_base, f_base) + s_margin
        if t_expected == 0: return np.nan
        return r_coupon + ((oid + f_upfront) / t_expected)

    @staticmethod
    def calculate_pik_ead(initial_ead: float, r_pik: float, periods: float) -> float:
        return initial_ead * ((1 + r_pik) ** periods)

    @staticmethod
    def calculate_expected_loss(pd: float, lgd: float, ead: float) -> float:
        return pd * lgd * ead

    @staticmethod
    def calculate_fccr(ebitda: float, rent_lease: float, maint_capex: float, cash_taxes: float, cash_interest: float, scheduled_principal: float) -> float:
        modified_ebitda = ebitda + rent_lease - maint_capex - cash_taxes
        fixed_charges = cash_interest + scheduled_principal + rent_lease
        if fixed_charges <= 1e-9:
            # Fixed charges must be strictly positive — negative value indicates
            # a sign convention error in input data (principal stored as negative CF).
            raise ValueError(
                f"FCCR denominator (fixed_charges={fixed_charges:.4f}) is zero or negative. "
                f"Verify sign conventions: cash_interest={cash_interest}, "
                f"scheduled_principal={scheduled_principal}, rent_lease={rent_lease}"
            )
        # FIX: Indented to remain inside the calculate_fccr method
        return float(modified_ebitda / fixed_charges)

    @staticmethod
    def calculate_ltv(total_net_debt: float, enterprise_value: float) -> float:
        if abs(enterprise_value) < 1e-9: return np.nan
        return float(total_net_debt / enterprise_value)

    @staticmethod
    def calculate_cqi(pd_array, lgd_array, macro_corr_array, covenant_weights) -> float:
        pd_array, lgd_array = np.array(pd_array, dtype=float), np.array(lgd_array, dtype=float)
        macro_corr_array, covenant_weights = np.array(macro_corr_array, dtype=float), np.array(covenant_weights, dtype=float)

        # Input validation
        if np.any(pd_array < 0) or np.any(pd_array > 1):
            raise ValueError(f"PD values must be in [0,1]. Got range [{pd_array.min():.4f}, {pd_array.max():.4f}]")
        if np.any(covenant_weights < 0):
            raise ValueError("Covenant weights must be non-negative.")

        # Normalize weights to sum to 1
        weight_sum = covenant_weights.sum()
        if weight_sum < 1e-9:
            raise ValueError("Covenant weights sum to zero.")
        covenant_weights = covenant_weights / weight_sum

        # Compute ratio with a safe minimum to avoid log of values <= 0
        numerator = np.maximum(1.0 - pd_array, 1e-9)
        denominator = np.maximum(lgd_array * macro_corr_array, 1e-9)
        ratio = np.maximum(numerator / denominator, 1e-9)   # enforce log domain
        log_comp = np.log(ratio)
        return float(np.sum(log_comp * covenant_weights))

    @staticmethod
    def calculate_lygi(credit_demand: float, supply_penetration: float, regional_risk_premium: float, illiquidity_factor: float) -> float:
        if regional_risk_premium == 0: return np.nan
        return ((credit_demand - supply_penetration) / regional_risk_premium) * illiquidity_factor

Writing credit_ratios.py


In [ ]:
%%writefile risk_simulation_engine.py
"""
FINDING 018 — EXTENDED REMEDIATION
Dynamic Discount Rate via Live API Fetch
File: risk_simulation_engine.py

Replaces the hardcoded discount_rate=0.04 in simulate_ev_cvar() with a
live FRED-fetched, tenor-matched Treasury yield, with an optional
credit-spread adjustment for HY and private credit positions.

SR 11-7 Rationale
-----------------
A static discount rate is a model parameter that drifts out of calibration
as the rate environment changes. Between 2021 and 2023, the US risk-free
rate moved from ~0.05% to ~5.50%. A hardcoded 4% discount rate during that
period would have overstated EV by hundreds of basis points on multi-year
private credit positions. SR 11-7 §IV.A requires that model parameters
reflect current market conditions.

Discount Rate Framework (Three Tiers)
--------------------------------------
Tier 1 — Risk-Free Only (Treasury, tenor-matched):
    Appropriate for: pricing relative to sovereign benchmark,
    IG corporate bond analysis.
    Source: FRED DGS series (DGS1, DGS2, DGS5, DGS10, DGS30).
    Tenor selection: match to the weighted average life (WAL) of cash flows.

Tier 2 — Credit-Adjusted (Risk-Free + OAS):
    Appropriate for: HY bonds, private credit absolute value, hurdle rates.
    Source: FRED DGS + BAMLH0A0HYM2 (HY OAS) or BAMLC0A0CM (IG OAS).
    Formula: rate = DGS_tenor/100 + OAS/10000

Tier 3 — WACC (deal-specific):
    Appropriate for: equity-linked structures, sponsor returns analysis.
    Source: Deal-level input from underwriting model (not fetched here).

Fallback Hierarchy (NO hardcoded default)
-----------------------------------------
1. FRED DGS series (primary, most accurate, daily updates)
2. yfinance Treasury tickers (^IRX, ^FVX, ^TNX, ^TYX) — real-time backup
3. On-disk cache (.fred_cache_DGSx.pkl) — last known good value
4. RuntimeError — never silently substitute a stale or arbitrary rate
"""

import os
import pickle
import warnings
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ── FRED Series IDs for each Treasury tenor ────────────────────────────────────
FRED_TREASURY_SERIES = {
    1:  "DGS1",   # 1-Year Constant Maturity Treasury
    2:  "DGS2",   # 2-Year
    5:  "DGS5",   # 5-Year
    10: "DGS10",  # 10-Year
    30: "DGS30",  # 30-Year
}

# yfinance fallback tickers (quoted in percent, same as FRED)
YFINANCE_TREASURY_TICKERS = {
    1:  "^IRX",   # 13-week T-Bill (closest to 1Y)
    2:  "^IRX",   # use T-Bill as 2Y fallback if DGS2 fails
    5:  "^FVX",   # 5-Year
    10: "^TNX",   # 10-Year
    30: "^TYX",   # 30-Year
}

# OAS series for credit-adjusted discount rates (already used in data_engine.py)
FRED_OAS_SERIES = {
    "IG":  "BAMLC0A0CM",    # IG OAS (bps)
    "HY":  "BAMLH0A0HYM2",  # HY Master OAS (bps)
    "BB":  "BAMLH0A1HYM2",  # BB OAS (bps)
    "B":   "BAMLH0A2HYB",   # B OAS (bps)
    "CCC": "BAMLH0A3HYC",   # CCC OAS (bps)
}


def _get_retry_session() -> requests.Session:
    """Enterprise retry configuration — reuses the pattern from data_engine.py."""
    session = requests.Session()
    retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    return session


def _select_tenor(n_periods: int) -> int:
    """
    Maps a deal's number of periods to the closest standard Treasury tenor.

    Rules
    -----
    <= 1  period  -> 1-Year (DGS1)
    <= 3  periods -> 2-Year (DGS2)
    <= 7  periods -> 5-Year (DGS5)
    <= 15 periods -> 10-Year (DGS10)
    >  15 periods -> 30-Year (DGS30)

    These boundaries follow standard fixed-income benchmarking convention:
    the discount rate tenor should match the weighted average life of the
    instrument, not just the final maturity.
    """
    if n_periods <= 1:
        return 1
    elif n_periods <= 3:
        return 2
    elif n_periods <= 7:
        return 5
    elif n_periods <= 15:
        return 10
    else:
        return 30


def _fetch_fred_rate(series_id: str, api_key: str) -> float:
    """
    Fetches the latest value for a FRED series and returns it as a decimal rate.

    FRED quotes Treasury yields in percent (e.g., 4.50 means 4.50%).
    This function divides by 100 before returning.

    Returns
    -------
    float : rate as a decimal (e.g., 0.045 for 4.5%)

    Raises
    ------
    RuntimeError if the series cannot be fetched and no cache exists.
    """
    FRED_BASE_URL = "https://api.stlouisfed.org/fred/series/observations"
    cache_path = f".fred_cache_{series_id}.pkl"

    try:
        session = _get_retry_session()
        params = {
            "series_id": series_id,
            "api_key": api_key,
            "file_type": "json",
            "sort_order": "desc",
            "limit": 10,  # last 10 obs to find latest non-missing value
        }
        resp = session.get(FRED_BASE_URL, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()

        if "error_message" in data:
            raise ValueError(f"FRED error: {data['error_message']}")

        # FRED uses '.' for missing values — filter them out
        obs = [
            float(o["value"])
            for o in data.get("observations", [])
            if o.get("value", ".") != "."
        ]

        if not obs:
            raise ValueError(f"No valid observations returned for {series_id}")

        # FRED quotes in percent -> convert to decimal
        rate_decimal = obs[0] / 100.0

        # Persist to cache for fallback
        with open(cache_path, "wb") as f:
            pickle.dump(rate_decimal, f)

        return rate_decimal

    except Exception as e:
        warnings.warn(f"FRED fetch failed for {series_id}: {e}. Attempting cache.")

        if os.path.exists(cache_path):
            with open(cache_path, "rb") as f:
                cached_rate = pickle.load(f)
            warnings.warn(
                f"Using cached rate for {series_id}: {cached_rate:.4%}. "
                f"This may be stale. Verify market conditions before using EV/CVaR output."
            )
            return cached_rate

        raise RuntimeError(
            f"Cannot fetch discount rate from FRED ({series_id}) and no cache exists. "
            f"EV/CVaR simulation aborted. "
            f"Ensure FRED_API_KEY is set and network connectivity is available."
        ) from e


def _fetch_yfinance_rate(ticker: str) -> float:
    """
    Fetches a Treasury yield from Yahoo Finance as a decimal rate.
    Used as a secondary fallback when FRED is unavailable.

    yfinance quotes Treasury yields in percent — divide by 100.
    """
    try:
        import yfinance as yf
        data = yf.download(ticker, period="5d", progress=False)
        if data.empty:
            raise ValueError(f"yfinance returned empty data for {ticker}")
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        latest = float(data["Close"].dropna().iloc[-1])
        return latest / 100.0
    except Exception as e:
        raise RuntimeError(f"yfinance fallback failed for {ticker}: {e}") from e


def fetch_live_discount_rate(
    n_periods: int,
    rate_type: str = "risk_free",
    credit_tier: str = "HY",
    api_key: str = None,
) -> tuple:
    """
    Fetches a live, tenor-matched discount rate from FRED (primary) or
    yfinance (secondary), with a cache fallback.

    Parameters
    ----------
    n_periods : int
        Number of cash flow periods. Used to select the matching Treasury tenor.
        A 4-period deal selects DGS5 (5-Year Treasury).

    rate_type : str
        'risk_free'      -> Treasury yield only (Tier 1).
                           Use for IG bonds, sovereign benchmarking.
        'credit_adjusted'-> Treasury + OAS spread (Tier 2).
                           Use for HY bonds, private credit, direct lending.

    credit_tier : str
        Which OAS series to add when rate_type='credit_adjusted'.
        Options: 'IG', 'HY', 'BB', 'B', 'CCC'
        Default: 'HY' (appropriate for most private credit deals in this engine).

    api_key : str
        FRED API key. If None, reads from FRED_API_KEY environment variable.

    Returns
    -------
    tuple: (rate: float, metadata: dict)
        rate     : discount rate as a decimal (e.g., 0.045 for 4.5%)
        metadata : dict with keys 'tenor', 'risk_free_rate',
                   'oas_spread', 'total_rate', 'source', 'series_id'

    Raises
    ------
    RuntimeError if no rate can be obtained from any source and no cache exists.

    Examples
    --------
    # Tier 1 — risk-free only, 4-period private credit deal
    rate, meta = fetch_live_discount_rate(n_periods=4, rate_type='risk_free')
    # -> fetches DGS5, returns ~0.045

    # Tier 2 — credit-adjusted, HY deal
    rate, meta = fetch_live_discount_rate(n_periods=4, rate_type='credit_adjusted',
                                           credit_tier='HY')
    # -> fetches DGS5 + BAMLH0A0HYM2, returns ~0.080
    """
    if api_key is None:
        api_key = os.getenv("FRED_API_KEY")
    if not api_key:
        raise EnvironmentError(
            "FRED_API_KEY environment variable is not set. "
            "Cannot fetch live discount rate."
        )

    # ── Tenor selection ────────────────────────────────────────────────────────
    tenor = _select_tenor(n_periods)
    fred_series = FRED_TREASURY_SERIES[tenor]
    yf_ticker = YFINANCE_TREASURY_TICKERS[tenor]

    # ── Fetch risk-free rate (FRED primary, yfinance secondary) ───────────────
    try:
        risk_free_rate = _fetch_fred_rate(fred_series, api_key)
        source = f"FRED:{fred_series}"
    except RuntimeError:
        warnings.warn(f"FRED failed for {fred_series}. Trying yfinance {yf_ticker}.")
        try:
            risk_free_rate = _fetch_yfinance_rate(yf_ticker)
            source = f"yfinance:{yf_ticker}"
        except RuntimeError as e:
            raise RuntimeError(
                f"All rate sources failed for tenor={tenor}Y. "
                f"FRED series: {fred_series}, yfinance: {yf_ticker}. "
                f"Cannot proceed with simulation."
            ) from e

    # ── Tier 2: Add OAS spread if credit-adjusted ──────────────────────────────
    oas_spread_decimal = 0.0
    oas_source = None

    if rate_type == "credit_adjusted":
        oas_series_id = FRED_OAS_SERIES.get(credit_tier)
        if oas_series_id is None:
            raise ValueError(
                f"Unknown credit_tier '{credit_tier}'. "
                f"Valid options: {list(FRED_OAS_SERIES.keys())}"
            )
        try:
            # _fetch_fred_rate already returns decimal (FRED_value / 100).
            # FRED OAS series are quoted in percent (3.50 = 350 bps = 0.035 decimal).
            oas_spread_decimal = _fetch_fred_rate(oas_series_id, api_key)
            oas_source = f"FRED:{oas_series_id}"
        except RuntimeError as e:
            warnings.warn(
                f"OAS fetch failed for {oas_series_id}: {e}. "
                f"Using risk-free rate only (no spread add-on). "
                f"Discount rate will UNDERSTATE required return for credit assets."
            )

    total_rate = risk_free_rate + oas_spread_decimal

    metadata = {
        "tenor_years": tenor,
        "fred_series": fred_series,
        "risk_free_rate": risk_free_rate,
        "oas_spread_bps": oas_spread_decimal * 10000,
        "total_rate": total_rate,
        "source": source,
        "oas_source": oas_source,
        "rate_type": rate_type,
        "credit_tier": credit_tier,
    }

    return total_rate, metadata


class AgenticRiskSimulator:
    """Module 4: AlphaCredit.ai - Agentic Monte Carlo Risk Simulations."""

    @staticmethod
    def simulate_ev_cvar(
        cash_flows: np.ndarray,
        pd_curve: np.ndarray,
        recovery_rate: float,
        discount_rate: float = None,
        n_paths: int = 10000,
        rate_type: str = "risk_free",
        credit_tier: str = "HY",
        api_key: str = None,
        _force_rate: float = None,  # for testing only — never use in production
    ) -> tuple:
        """
        Monte Carlo EV and CVaR simulation with live discount rate.

        Changes from original
        ---------------------
        1. Finding 012: Marginal PD -> cumulative survival probability
           (searchsorted on survival curve prevents incorrect default timing).
        2. Finding 018: Continuous -> discrete discounting
           ((1+r)^-t instead of e^-rt, aligned with market quote convention).
        3. NEW: discount_rate is fetched live from FRED, tenor-matched to the
           number of cash flow periods. No hardcoded fallback.

        Parameters
        ----------
        cash_flows    : np.ndarray — periodic cash flows (e.g., [50000, 50000, 1050000])
        pd_curve      : np.ndarray — marginal default probability per period
        recovery_rate : float      — recovery rate on defaulted principal (e.g., 0.40)
        discount_rate : float      — if provided, overrides API fetch (for batch runs
                                     where you have already fetched the rate once).
                                     Must be a decimal (e.g., 0.045 not 4.5).
        n_paths       : int        — Monte Carlo path count (default 10,000)
        rate_type     : str        — 'risk_free' or 'credit_adjusted' (see fetch_live_discount_rate)
        credit_tier   : str        — OAS tier for credit_adjusted: 'IG','HY','BB','B','CCC'
        api_key       : str        — FRED API key (defaults to FRED_API_KEY env var)

        Returns
        -------
        tuple: (ev: float, cvar_95: float, rate_metadata: dict)
            ev           — Expected Value of the instrument (NPV-weighted across paths)
            cvar_95      — 95th percentile Conditional Value at Risk (Expected Shortfall)
            rate_metadata— Dict documenting what rate was used and its source
        """
        cash_flows = np.asarray(cash_flows, dtype=float)
        pd_curve = np.asarray(pd_curve, dtype=float)
        n_periods = len(cash_flows)

        # ── Fetch or validate discount rate ───────────────────────────────────
        if _force_rate is not None:
            # Test path only
            rate = _force_rate
            rate_metadata = {"source": "forced_test_value", "total_rate": rate}

        elif discount_rate is not None:
            # Caller pre-fetched the rate (e.g., batch processing loop)
            # Validate it is a decimal, not a percentage
            if discount_rate > 1.0:
                raise ValueError(
                    f"discount_rate={discount_rate} appears to be in percent, not decimal. "
                    f"Pass 0.045, not 4.5. FRED values must be divided by 100 before passing."
                )
            rate = discount_rate
            rate_metadata = {
                "source": "caller_provided",
                "total_rate": rate,
                "tenor_years": _select_tenor(n_periods),
            }

        else:
            # Live fetch — primary path
            rate, rate_metadata = fetch_live_discount_rate(
                n_periods=n_periods,
                rate_type=rate_type,
                credit_tier=credit_tier,
                api_key=api_key,
            )
            print(
                f"  [Discount Rate] {rate_metadata['tenor_years']}Y Treasury "
                f"({rate_metadata['fred_series']}): "
                f"{rate_metadata['risk_free_rate']:.4%}"
                + (
                    f" + OAS ({rate_metadata['credit_tier']}): "
                    f"{rate_metadata['oas_spread_bps']:.0f} bps"
                    if rate_metadata.get("oas_source")
                    else ""
                )
                + f" = {rate:.4%} (source: {rate_metadata['source']})"
            )

        # ── Finding 018: Discrete discounting ─────────────────────────────────
        # Market rates are quoted as annual percentage discrete yields.
        # (1+r)^-t is the correct convention. e^-rt systematically understates PV.
        discount_factors = (1 + rate) ** -np.arange(1, n_periods + 1)
        max_npv = np.sum(cash_flows * discount_factors)

        # ── Finding 012: Cumulative survival probability ───────────────────────
        # Convert marginal PD per period to cumulative survival function.
        # S(t) = prod_{i=0}^{t} (1 - PD_i)
        # This correctly handles non-monotone PD curves (e.g., maturity walls).
        survival_prob = np.cumprod(1.0 - pd_curve)
        # Pad with S(0)=1 (alive at inception) and S(end)=0 (guaranteed default by T+1)
        survival_padded = np.concatenate([[1.0], survival_prob, [0.0]])

        # Sample default times via inverse survival CDF
        U = np.random.rand(n_paths)
        # searchsorted on -S (descending -> ascending for searchsorted)
        default_idx = np.searchsorted(-survival_padded, -U, side="right") - 1
        default_idx = np.clip(default_idx, 0, n_periods)

        # ── Path payoff computation ────────────────────────────────────────────
        survived_mask = np.arange(n_periods) < default_idx[:, None]
        cf_received = cash_flows * survived_mask

        total_remaining_cf = np.sum(cash_flows) - np.sum(cf_received, axis=1)
        recovery_cf = np.zeros(n_paths)
        valid_default = default_idx < n_periods
        recovery_cf[valid_default] = total_remaining_cf[valid_default] * recovery_rate

        # Recovery discounted to time of default — discrete
        rec_discount = (1 + rate) ** -(default_idx + 1)
        path_npv = (
            np.sum(cf_received * discount_factors, axis=1)
            + (recovery_cf * rec_discount)
        )

        # ── EV and CVaR ────────────────────────────────────────────────────────
        ev = float(np.mean(path_npv))
        losses = max_npv - path_npv
        var_95 = float(np.percentile(losses, 95))
        tail_losses = losses[losses >= var_95]
        cvar_95 = float(np.mean(tail_losses)) if len(tail_losses) > 0 else var_95

        return ev, cvar_95, rate_metadata

    # ── Unchanged methods below ────────────────────────────────────────────────

    @staticmethod
    def generate_yield_matrix(df: pd.DataFrame) -> pd.DataFrame:
        return pd.DataFrame({
            "Asset": df.index if "Asset" not in df.columns else df["Asset"],
            "Pure Yield to Maturity (%)": df.get("YTM", np.nan),
            "Pure Yield Price Movements": df.get("Price_Movement", np.nan),
        }).reset_index(drop=True)

    @staticmethod
    def generate_volatility_spectrum(df: pd.DataFrame) -> pd.DataFrame:
        percentiles = [99, 95, 90, 80, 70]
        pure_vol = [
            np.percentile(df.get("Pure_Bond_Return", np.zeros(len(df))), p)
            for p in percentiles
        ]
        spread_vol = [
            np.percentile(df.get("Yield_Spread_Return", np.zeros(len(df))), p)
            for p in percentiles
        ]
        return pd.DataFrame({
            "Percentile Threshold": [f"{p}th" for p in percentiles],
            "Pure Bond Volatility": pure_vol,
            "Yield Spread Volatility": spread_vol,
        })

    @staticmethod
    def generate_ml_scorecard(
        asset_names: list,
        predictions: np.ndarray,
        cqi_scores: list,
        shap_df: pd.DataFrame,
    ) -> pd.DataFrame:
        drivers = shap_df["Top_3_SHAP_Drivers"].str.split(" | ", expand=True)
        for col in range(3):
            if col not in drivers.columns:
                drivers[col] = None
        return pd.DataFrame({
            "Asset": asset_names,
            "Ensemble Probability": [f"{p:.1f}%" for p in predictions],
            "CQI Score": [f"{score:.4f}" for score in cqi_scores],
            "Top_SHAP_Driver_1": drivers[0].values,
            "Top_SHAP_Driver_2": drivers[1].values,
            "Top_SHAP_Driver_3": drivers[2].values,
        })

Writing risk_simulation_engine.py


In [ ]:
%%writefile volatility_trading.py
import warnings
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from arch import arch_model

warnings.filterwarnings("ignore")

GARCH_SIGNAL_PERCENTILE: int = 90
STATIC_DEALER_MARKUP:  float = 1.00
MIN_OBS_FOR_GARCH:       int = 100
DEALER_PRICING_WINDOW:   int = 90
INVESTMENT_CAPITAL:    float = 100000.0

# --- THE FIX: Restored manual inputs ---
def get_trade_size():
    trade_size = input("Enter the Hedge Fund trade size (dollars), Note: round it to the nearest integer: ")
    while not trade_size.isdigit() or int(trade_size) < 0:
        trade_size = input("Enter the Hedge Fund trade size, Note: round it to the nearest integer: ")
    return int(trade_size) / 1000000.0

def get_hold_days():
    hold_days = input("Enter the number of hold days (positive integer): ")
    while not hold_days.isdigit() or int(hold_days) <= 0:
        hold_days = input("Please enter a valid positive integer for hold days: ")
    return int(hold_days)

def prepare_pristine_data(raw_filepath: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not os.path.exists(raw_filepath):
        raise FileNotFoundError(f"Could not find {raw_filepath}")

    with open(raw_filepath, "r") as f:
        data = json.load(f)

    df_raw = pd.DataFrame(data)
    df_raw.index = pd.to_datetime(df_raw.index)
    df_raw = df_raw.sort_index().resample("B").last().ffill()

    # --- THE FIX: Unconditional conversion to Basis Points ---
    # Removes the fragile max_val check that was failing due to 2008 CCC yield spikes
    df_raw = df_raw * 100.0

    # Initialize df_spreads with the EXACT same index length as df_raw
    df_spreads = pd.DataFrame(index=df_raw.index)

    pairs = [
        ("AA", "AAA"), ("A", "AA"), ("BBB", "A"),
        ("BB", "Fallen_Angel"),
        ("BB", "BBB"), ("B", "BB"), ("CCC", "B")
    ]

    for riskier, safer in pairs:
        if safer in df_raw.columns and riskier in df_raw.columns:
            df_spreads[f"{riskier} - {safer}"] = df_raw[riskier] - df_raw[safer]

    return df_raw, df_spreads.dropna(axis=1, how='all')

def get_treasury_rates(df_spreads: pd.DataFrame) -> pd.Series:
    try:
        start = df_spreads.index.min().strftime("%Y-%m-%d")
        end   = df_spreads.index.max().strftime("%Y-%m-%d")
        raw   = yf.download("^IRX", start=start, end=end, progress=False)
        rates = raw["Close"] / 100.0
        return rates.squeeze() if isinstance(rates, pd.DataFrame) else rates
    except Exception:
        return pd.Series(0.04, index=df_spreads.index)

def get_dealer_volatility(series: pd.Series, window: int = DEALER_PRICING_WINDOW) -> pd.Series:
    returns = series.diff().dropna()
    dealer_vol = returns.rolling(window).std().shift(1)
    return dealer_vol.reindex(series.index).ffill().bfill()

def fit_garch_volatility(series: pd.Series, label: str = "", min_obs: int = MIN_OBS_FOR_GARCH) -> pd.Series:
    returns = series.diff().dropna()
    if len(returns) < min_obs: return returns.rolling(DEALER_PRICING_WINDOW).std().reindex(series.index).ffill().bfill()
    try:
        mdl = arch_model(returns, vol="GARCH", p=1, q=1, mean="AR", lags=1, dist="Normal", rescale=True)
        res = mdl.fit(disp="off", show_warning=False)
        params = res.params
        alpha, beta = params.get('alpha[1]', 0), params.get('beta[1]', 0)
        if alpha + beta >= 1.0: raise ValueError(f"GARCH non-stationary for {label}: alpha+beta={alpha+beta:.4f} >= 1.")
        return res.conditional_volatility.reindex(series.index).ffill().bfill()
    except Exception:
        return returns.rolling(DEALER_PRICING_WINDOW).std().reindex(series.index).ffill().bfill()

def calc_ou_straddle(dealer_vol_series: pd.Series, r_series: pd.Series, markup_series: pd.Series, T_days: int, kappa: float) -> pd.Series:
    T = T_days / 252.0
    annualized_normal_vol = dealer_vol_series * np.sqrt(252)
    KAPPA_FLOOR = 1e-6
    if kappa < KAPPA_FLOOR: ou_variance = (annualized_normal_vol ** 2) * T
    else: ou_variance = (annualized_normal_vol ** 2 / (2 * kappa)) * (1 - np.exp(-2 * kappa * T))
    safe_ou_variance = np.where(ou_variance > 0, ou_variance, 1e-12)
    effective_vol = np.sqrt(safe_ou_variance / T)
    priced_vol = effective_vol * markup_series
    discount_factor = np.exp(-r_series * T)
    sqrt_2_over_pi = np.sqrt(2 / np.pi)
    straddle_premium = discount_factor * priced_vol * np.sqrt(T) * sqrt_2_over_pi
    return straddle_premium

def calculate_dynamic_pb_markup(retail_markup_series: pd.Series, daily_vol_bps: pd.Series, trade_size_millions: float = 50.0) -> pd.Series:
    base_discount = 0.05
    volume_discount = np.log10(max(1.0, trade_size_millions)) * 0.05
    annualized_vol_bps = daily_vol_bps * np.sqrt(252)
    safe_vol_threshold = 100.0
    illiquidity_penalty = np.maximum(0.0, (annualized_vol_bps - safe_vol_threshold) / 1000.0)
    total_discount = base_discount + volume_discount - illiquidity_penalty
    total_discount = np.clip(total_discount, 0.0, 0.25)
    hf_multiplier = 1.0 - total_discount
    return retail_markup_series * hf_multiplier

def calculate_dynamic_execution_friction(base_spread_bps: float, macro_vol_proxy: pd.Series, shock_percentile: float = 90.0, growth_rate: float = 0.08) -> pd.Series:
    threshold_vol = np.percentile(macro_vol_proxy.dropna(), shock_percentile)
    excess_vol = np.maximum(0, macro_vol_proxy - threshold_vol)
    dynamic_friction_bps = base_spread_bps * np.exp(growth_rate * excess_vol)
    return pd.Series(dynamic_friction_bps, index=macro_vol_proxy.index)

def analyze_strategy(df_spreads: pd.DataFrame, df_raw: pd.DataFrame, r_daily: pd.Series, retail_markup: pd.Series, percentile_override: float = 90.0, kappa_override: float = 25.0, trade_size: float = 50.0, hold_days: int = 5) -> list[dict]:
    target_pairs = ["AA - AAA", "A - AA", "BBB - A", "BB - Fallen_Angel", "BB - BBB", "B - BB", "CCC - B"]
    pairs = [p for p in target_pairs if p in df_spreads.columns]
    results = []
    ts = trade_size

    for pair in pairs:
        print(f"\n{'─' * 60}\n  Processing Spread: {pair}")
        if "Fallen_Angel" in pair or "B - CCC" in pair or "BB - B" in pair:
            current_kappa = 40.0 if kappa_override == 25.0 else kappa_override
            pair_base_retail_markup = retail_markup * 1.20
            print(f"  [Distressed Rules Applied] Kappa: {current_kappa} | Liquidity Premium: +20%")
        else:
            current_kappa = kappa_override
            pair_base_retail_markup = retail_markup
            print(f"  [Standard Rules Applied] Kappa: {current_kappa}")

        spread_garch_vol = fit_garch_volatility(df_spreads[pair], label=f"{pair} spread")
        vol_threshold = float(np.percentile(spread_garch_vol.dropna(), percentile_override))

        shock_days = spread_garch_vol >= vol_threshold
        actual_freq = shock_days.sum() / max(len(shock_days), 1)

        spread_dealer_vol = get_dealer_volatility(df_spreads[pair])

        pair_hf_markup = calculate_dynamic_pb_markup(retail_markup_series=pair_base_retail_markup, daily_vol_bps=spread_dealer_vol, trade_size_millions= ts)

        gross_hd = (df_spreads[pair].shift(-hold_days) - df_spreads[pair]).abs()

        valid_shock_gross = gross_hd[shock_days].dropna()
        valid_norm_gross = gross_hd[~shock_days].dropna()
        gross_shock = float(valid_shock_gross.mean()) if not valid_shock_gross.empty else 0.0
        gross_normal = float(valid_norm_gross.mean()) if not valid_norm_gross.empty else 0.0

        hf_straddle_cost = calc_ou_straddle(spread_dealer_vol, r_daily, pair_hf_markup, T_days=hold_days, kappa=current_kappa)

        valid_hf_shock_fee = hf_straddle_cost[shock_days].dropna()
        valid_hf_norm_fee = hf_straddle_cost[~shock_days].dropna()
        hf_fee_shock = float(valid_hf_shock_fee.mean()) if not valid_hf_shock_fee.empty else 0.0
        hf_fee_normal = float(valid_hf_norm_fee.mean()) if not valid_hf_norm_fee.empty else 0.0

        friction_penalty = calculate_dynamic_execution_friction(base_spread_bps=1.0, macro_vol_proxy=spread_dealer_vol, shock_percentile=90.0)

        valid_friction = friction_penalty[shock_days].dropna()
        avg_shock_penalty = float(valid_friction.mean()) if not valid_friction.empty else 0.0

        hf_net_shock = gross_shock - hf_fee_shock - avg_shock_penalty
        hf_net_normal = gross_normal - hf_fee_normal
        hf_mult = hf_net_shock / abs(hf_net_normal) if hf_net_normal else 0.0

        ret_fee_shock, ret_fee_normal, ret_net_shock, ret_net_normal, ret_mult = 0, 0, 0, 0, 0
        ret_gross_shock = gross_shock
        ret_gross_normal = gross_normal

        try:
            leg1, leg2 = pair.split(" - ")
            if leg1 in df_raw.columns and leg2 in df_raw.columns:
                l1_dealer_vol = get_dealer_volatility(df_raw[leg1])
                l2_dealer_vol = get_dealer_volatility(df_raw[leg2])
                l1_straddle = calc_ou_straddle(l1_dealer_vol, r_daily, pair_base_retail_markup, T_days=hold_days, kappa=current_kappa)
                l2_straddle = calc_ou_straddle(l2_dealer_vol, r_daily, pair_base_retail_markup, T_days=hold_days, kappa=current_kappa)
                ret_total_cost = l1_straddle + l2_straddle

                valid_ret_shock_fee = ret_total_cost[shock_days].dropna()
                valid_ret_norm_fee = ret_total_cost[~shock_days].dropna()
                ret_fee_shock = float(valid_ret_shock_fee.mean()) if not valid_ret_shock_fee.empty else 0.0
                ret_fee_normal = float(valid_ret_norm_fee.mean()) if not valid_ret_norm_fee.empty else 0.0

                ret_net_shock = ret_gross_shock - ret_fee_shock - avg_shock_penalty
                ret_net_normal = ret_gross_normal - ret_fee_normal
                ret_mult = ret_net_shock / abs(ret_net_normal) if ret_net_normal else 0.0
        except ValueError:
            pass

        results.append({
            "Pair": pair, "Freq": actual_freq, "VolThreshold": vol_threshold,
            "HF_Gross_Shock": gross_shock, "HF_Gross_Normal": gross_normal,
            "HF_Fee_Shock": hf_fee_shock, "HF_Fee_Normal": hf_fee_normal,
            "HF_Net_Shock": hf_net_shock, "HF_Net_Normal": hf_net_normal, "HF_Mult": hf_mult,
            "Ret_Gross_Shock": ret_gross_shock, "Ret_Gross_Normal": ret_gross_normal,
            "Ret_Fee_Shock": ret_fee_shock, "Ret_Fee_Normal": ret_fee_normal,
            "Ret_Net_Shock": ret_net_shock, "Ret_Net_Normal": ret_net_normal, "Ret_Mult": ret_mult,
        })
    return results

def plot_summary_table(results: list[dict], investment: float = INVESTMENT_CAPITAL, hold_days = 5) -> None:
    columns = [
        "Pair", "Market\nEnvironment",
        "Gross\nReturn (%)", f"Gross Payout\n(${investment/1000:.0f}k)",
        "HF Net\nReturn (%)", f"HF Net Profit\n(${investment/1000:.0f}k)",
        "Retail Net\nReturn (%)", f"Retail Net Profit\n(${investment/1000:.0f}k)"
    ]

    def fmt_pct(val): return "N/A" if pd.isna(val) else f"{val / 100.0:+.2f}%"
    def fmt_usd(val): return "N/A" if pd.isna(val) else f"${val * (investment / 10000.0):+,.0f}"
    def get_color(val):
        if pd.isna(val): return "#FFFFFF"
        return "#E6F4EA" if val > 0 else "#FCE8E6"

    cell_text = []
    colors = []

    for r in results:
        freq_shock = r['Freq'] * 100
        freq_norm  = (1 - r['Freq']) * 100
        row_shock = [
            r['Pair'], f"Shock (Top {freq_shock:.0f}%)",
            fmt_pct(r['HF_Gross_Shock']), fmt_usd(r['HF_Gross_Shock']),
            fmt_pct(r['HF_Net_Shock']),   fmt_usd(r['HF_Net_Shock']),
            fmt_pct(r['Ret_Net_Shock']),  fmt_usd(r['Ret_Net_Shock'])
        ]
        cell_text.append(row_shock)
        colors.append([
            "w", "#f8d7da", "#F0F8FF", "#F0F8FF",
            get_color(r['HF_Net_Shock']), get_color(r['HF_Net_Shock']),
            get_color(r['Ret_Net_Shock']), get_color(r['Ret_Net_Shock'])
        ])

        row_norm = [
            "", f"Normal (Bottom {freq_norm:.0f}%)",
            fmt_pct(r['HF_Gross_Normal']), fmt_usd(r['HF_Gross_Normal']),
            fmt_pct(r['HF_Net_Normal']),   fmt_usd(r['HF_Net_Normal']),
            fmt_pct(r['Ret_Net_Normal']),  fmt_usd(r['Ret_Net_Normal'])
        ]
        cell_text.append(row_norm)
        colors.append([
            "w", "#e2e3e5", "#F0F8FF", "#F0F8FF",
            get_color(r['HF_Net_Normal']), get_color(r['HF_Net_Normal']),
            get_color(r['Ret_Net_Normal']), get_color(r['Ret_Net_Normal'])
        ])

    fig, ax = plt.subplots(figsize=(13, 8))
    ax.axis('off')
    ax.axis('tight')

    table = ax.table(cellText=cell_text, cellColours=colors, colLabels=columns, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.8)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('#1D3557')
        elif col == 0 and row % 2 != 0:
            cell.set_text_props(weight='bold')
        elif col == 1:
            cell.set_text_props(style='italic')

    plt.title(f"Post-Shock vs Normal Returns per ${investment:,.0f} Invested" + ' and held for ' + str(hold_days) + ' days', fontweight="bold", fontsize=15, pad=20)
    plt.tight_layout()
    plt.savefig("volatility_summary_table.png", dpi=150, bbox_inches="tight")
    plt.show()

def plot_scenario(results: list[dict], mode: str, hold_days = 5) -> None:
    assets = [r.get("Asset", r.get("Pair")) for r in results]
    x, width = np.arange(len(assets)), 0.35

    fig, ax = plt.subplots(figsize=(16, 7))

    normal_fee_vals = None
    if mode == "Gross":
        normal_vals = [r.get("HF_Gross_Normal", 0) if not np.isnan(r.get("HF_Gross_Normal", 0)) else 0 for r in results]
        shock_vals = [r.get("HF_Gross_Shock", 0) if not np.isnan(r.get("HF_Gross_Shock", 0)) else 0 for r in results]
        title, filename = "TIER 1: Theoretical Gross Payout", f"spread_{mode.lower()}_payout.png"
        shock_colors, normal_colors = ["#E63946"] * len(shock_vals), ["#457B9D"] * len(normal_vals)
    elif mode == "HedgeFund":
        normal_vals = [r.get("HF_Net_Normal", 0) if not np.isnan(r.get("HF_Net_Normal", 0)) else 0 for r in results]
        shock_vals = [r.get("HF_Net_Shock", 0) if not np.isnan(r.get("HF_Net_Shock", 0)) else 0 for r in results]
        title, filename = "TIER 2: Hedge Fund Net Returns", f"spread_{mode.lower()}_returns.png"
        shock_colors = ["#2A9D8F" if v > 0 else "#E63946" for v in shock_vals]
        normal_colors = ["#A8DADC" if v > 0 else "#F4A261" for v in normal_vals]
        normal_fee_vals = [r.get("HF_Fee_Normal", 0) for r in results]
    elif mode == "Retail":
        normal_vals = [r.get("Ret_Net_Normal", 0) if not np.isnan(r.get("Ret_Net_Normal", 0)) else 0 for r in results]
        shock_vals = [r.get("Ret_Net_Shock", 0) if not np.isnan(r.get("Ret_Net_Shock", 0)) else 0 for r in results]
        title, filename = "TIER 3: Retail Net Returns", f"spread_{mode.lower()}_returns.png"
        shock_colors = ["#2A9D8F" if v > 0 else "#E63946" for v in shock_vals]
        normal_colors = ["#A8DADC" if v > 0 else "#F4A261" for v in normal_vals]
        normal_fee_vals = [r.get("Ret_Fee_Normal", 0) for r in results]

    ax.bar(x - width / 2, normal_vals, width, label="Normal Day", color=normal_colors, alpha=0.6)
    ax.bar(x + width / 2, shock_vals, width, label="Post-Shock Trade", color=shock_colors)

    all_plotted_vals = [v for v in normal_vals + shock_vals if not np.isnan(v)]
    if all_plotted_vals:
        min_y, max_y = min(all_plotted_vals), max(all_plotted_vals)
        max_abs = max(abs(min_y), abs(max_y))
        if max_abs < 10.0: max_abs = 10.0
        limit = max_abs * 1.40
        ax.set_ylim(-limit, limit)

    ax.set_ylabel("Net Profit (bps)", fontweight="bold")
    ax.set_title(title + ' after being held for ' + str(hold_days) + ' days' , fontsize=15, fontweight="bold", pad=20)
    ax.set_xticks(x)
    labels = ["★ " + a if "Fallen_Angel" in a else a for a in assets]
    ax.set_xticklabels(labels, fontweight="bold", fontsize=11)

    ax.axhline(0, color="black", linewidth=1.5, zorder=3)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="upper left")

    for i, r in enumerate(results):
        fee_shock = r.get('HF_Fee_Shock') if mode == "HedgeFund" else r.get('Ret_Fee_Shock')
        y_val = shock_vals[i]
        va_algn = "bottom" if y_val >= 0 else "top"
        pixel_offset = 10 if y_val >= 0 else -10

        if mode == "Gross":
            normal_divisor = max(abs(r.get('HF_Gross_Normal', 1)), 0.0001)
            text = f"Gross\nMult: {(r['HF_Gross_Shock'] / normal_divisor):.2f}x" if r.get('HF_Gross_Normal') else ""
        else:
            mult = r['HF_Mult'] if mode == "HedgeFund" else r['Ret_Mult']
            text = f"Fee (Shock): {fee_shock:.1f} bps\nMult: {mult:.2f}x"

        ax.annotate(text, xy=(x[i] + width / 2, y_val), xytext=(4, pixel_offset), textcoords="offset points", ha="left", va=va_algn, fontsize=8, bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", pad=1.5))

        if normal_fee_vals is not None and mode != "Gross":
            norm_fee = normal_fee_vals[i]
            if not np.isnan(norm_fee):
                y_norm = normal_vals[i]
                va_algn_norm = "bottom" if y_norm >= 0 else "top"
                pixel_offset_norm = 10 if y_norm >= 0 else -10
                norm_text = f"Fee (Normal): {norm_fee:.1f} bps"
                ax.annotate(norm_text, xy=(x[i] - width / 2, y_norm), xytext=(-4, pixel_offset_norm), textcoords="offset points", ha="right", va=va_algn_norm, fontsize=8, bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", pad=1.5))

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()

def run(trade_size: float = 50.0, hold_days: int = 5) -> None:
    # Not used directly in main.py, but left intact for standalone runs
    pass

Writing volatility_trading.py


In [ ]:
%%writefile volatility_matrix.py
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

def plot_volatility_matrix(matrix_df: pd.DataFrame, base_title: str, hold_days: int, region: str):
    if matrix_df.empty or matrix_df.isnull().all().all():
        print(f"  [WARNING] Skipping plot '{base_title}': No valid data.")
        return

    plt.figure(figsize=(16, 10))
    sns.heatmap(matrix_df, annot=True, fmt=".5f", cmap="RdYlGn", center=0,
                cbar_kws={'label': 'Net Profit / Return (bps)'},
                annot_kws={"size": 8})

    full_title = f"{region} - {base_title} ({hold_days}-Day Hold)"
    plt.title(full_title, fontsize=16, fontweight="bold", pad=20)
    plt.ylabel("Asset Class", fontweight="bold")
    plt.xlabel("Volatility Percentile Traded", fontweight="bold")
    plt.tight_layout()

    filename = f"{region.replace(' ', '_').lower()}_{base_title.replace(' ', '_').lower()}.png"
    plt.savefig(filename, dpi=150)
    plt.show()
    plt.close()


def generate_spectrum_matrices(assets: list, percentiles: list, strategy_results: list, hold_days: int, region: str):
    gross_df = pd.DataFrame(index=assets, columns=[f"{p}th" for p in percentiles], dtype=float)
    hf_net_df = pd.DataFrame(index=assets, columns=[f"{p}th" for p in percentiles], dtype=float)
    retail_net_df = pd.DataFrame(index=assets, columns=[f"{p}th" for p in percentiles], dtype=float)

    for r in strategy_results:
        # --- THE FIX: Support both Pure 'Asset' and Spread 'Pair' keys ---
        asset = r.get("Asset", r.get("Pair"))
        if asset not in assets:
            continue
        pct_label = f"{r['Percentile']}th"
        gross_df.at[asset, pct_label] = r.get("HF_Gross_Shock", 0)
        hf_net_df.at[asset, pct_label] = r.get("HF_Net_Shock", 0)
        retail_net_df.at[asset, pct_label] = r.get("Ret_Net_Shock", 0)

    plot_volatility_matrix(gross_df, "Tier 1 - Theoretical Gross Payout", hold_days, region)
    plot_volatility_matrix(hf_net_df, "Tier 2 - Hedge Fund Net Return", hold_days, region)
    plot_volatility_matrix(retail_net_df, "Tier 3 - Retail Net Return", hold_days, region)


def generate_yield_tables(df_assets: pd.DataFrame, hold_days: int, region: str):
    print(f"\n  [SYSTEM] Generating Yield & Carry Tables for {region}...")

    latest_yields = df_assets.iloc[-1]

    t1_data = []
    for asset, yld in latest_yields.items():
        t1_data.append([asset.replace("_", " "), f"{yld:.3f}%"])

    fig, ax = plt.subplots(figsize=(10, len(t1_data) * 0.5 + 2))
    ax.axis('tight')
    ax.axis('off')
    plt.title(f"{region} - Current Market Yields (Latest Annualized)", fontweight="bold", fontsize=14, loc="center", pad=20)

    table1 = ax.table(cellText=t1_data, colLabels=["Asset Class", "Current Annual Yield (%)"], loc='center', cellLoc='center')
    table1.auto_set_font_size(False)
    table1.set_fontsize(12)
    table1.scale(1.0, 2.0)

    for (row, col), cell in table1.get_celld().items():
        if row == 0:
            cell.set_facecolor('#1D3557')
            cell.set_text_props(weight='bold', color='white')

    plt.tight_layout()
    t1_filename = f"{region.replace(' ', '_').lower()}_current_yields.png"
    plt.savefig(t1_filename, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

    t2_data = []
    for asset, yld in latest_yields.items():
        prorated_pct = yld * (hold_days / 365.0)
        prorated_bps = prorated_pct * 100
        t2_data.append([asset.replace("_", " "), f"{yld:.3f}%", str(hold_days), f"{prorated_pct:.4f}%", f"{prorated_bps:.2f} bps"])

    fig2, ax2 = plt.subplots(figsize=(14, len(t2_data) * 0.5 + 2))
    ax2.axis('tight')
    ax2.axis('off')
    plt.title(f"{region} - Expected Baseline Carry Return ({hold_days}-Day Hold)", fontweight="bold", fontsize=14, loc="center", pad=20)

    cols2 = ["Asset Class", "Annual Yield", "Hold Period (Days)", "Expected Return (%)", "Expected Return (bps)"]
    table2 = ax2.table(cellText=t2_data, colLabels=cols2, loc='center', cellLoc='center')
    table2.auto_set_font_size(False)
    table2.set_fontsize(12)
    table2.scale(1.0, 2.0)

    for (row, col), cell in table2.get_celld().items():
        if row == 0:
            cell.set_facecolor('#1D3557')
            cell.set_text_props(weight='bold', color='white')

    plt.tight_layout()
    t2_filename = f"{region.replace(' ', '_').lower()}_{hold_days}day_carry_return.png"
    plt.savefig(t2_filename, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

Writing volatility_matrix.py


In [ ]:
%%writefile spread.py
import os
import json
import requests
import yfinance as yf
import pandas as pd

FRED_API_KEY = os.getenv("FRED_API_KEY")
if not FRED_API_KEY:
    raise EnvironmentError("FRED_API_KEY environment variable is not set.")
FRED_BASE_URL = "https://api.stlouisfed.org/fred/series/observations"

FETCH_ORDER = ["AAA", "AA", "A", "BBB", "Fallen_Angel", "BB", "B", "CCC"]
SPREAD_ORDER = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]

US_BOND_SERIES = {
    "AAA": "BAMLC0A1CAAA", "AA":  "BAMLC0A2CAA", "A":   "BAMLC0A3CA",
    "BBB": "BAMLC0A4CBBB", "Fallen_Angel": "ANGL_ETF_PROXY",
    "BB":  "BAMLH0A0HYM2", "B":   "BAMLH0A2HYB", "CCC": "BAMLH0A3HYC",
}

def fetch_fred_series(series_id: str) -> dict[str, float]:
    params = {"series_id": series_id, "api_key": FRED_API_KEY, "file_type": "json", "sort_order": "asc"}
    response = requests.get(FRED_BASE_URL, params=params, timeout=30)
    if not response.ok: response.raise_for_status()
    raw = response.json()
    return { obs["date"]: float(obs["value"]) for obs in raw.get("observations", []) if obs.get("value", ".") != "." }

def fetch_all_bond_returns(rating_order: list, series_map: dict) -> dict:
    bond_returns = {}
    for rating in rating_order:
        series_id = series_map.get(rating)
        if rating == "Fallen_Angel":
            print(f"  [API] Fetching Fallen Angel Proxy (ANGL ETF) via yfinance...")
            try:
                angl_ticker = yf.Ticker("ANGL")
                hist_data = angl_ticker.history(start="2016-01-01")
                ttm_dividends = hist_data["Dividends"].rolling(window=252).sum().dropna()
                yield_proxy = ((ttm_dividends / hist_data["Close"]) * 100).dropna()
                bond_returns[rating] = {str(date.date()): float(val) for date, val in yield_proxy.items()}
                print(f"  OK   {rating:<12} (Dynamic ANGL Yield) -> {len(bond_returns[rating])} obs")
            except Exception as e:
                print(f"  FAIL {rating:<12} (ANGL Proxy) -> {e}")
            continue
        try:
            data = fetch_fred_series(series_id)
            if data:
                bond_returns[rating] = data
                print(f"  OK   {rating:<12} ({series_id}) -> {len(data)} obs")
        except requests.RequestException as e:
            print(f"  ERROR {rating:<12} ({series_id}) -> {e}")
    return bond_returns

def get_adjacent_pairs(rating_order: list[str]) -> list[tuple[str, str]]:
    return list(zip(rating_order, rating_order[1:]))

def compute_spreads_pct(bond_returns: dict, adjacent_pairs: list) -> dict:
    spreads_pct = {}
    available = [r for r in SPREAD_ORDER if r in bond_returns]

    for lower, higher in get_adjacent_pairs(available):
        key = f"{higher} - {lower}"
        common_dates = sorted(set(bond_returns[lower]) & set(bond_returns[higher]))
        spreads_pct[key] = {
            date: bond_returns[higher][date] - bond_returns[lower][date]
            for date in common_dates
        }

    # --- THE FIX: Explicitly append the missing Fallen Angel pair ---
    if "Fallen_Angel" in bond_returns and "BB" in bond_returns:
        common_dates = sorted(set(bond_returns["BB"]) & set(bond_returns["Fallen_Angel"]))
        spreads_pct["BB - Fallen_Angel"] = {
            date: bond_returns["BB"][date] - bond_returns["Fallen_Angel"][date]
            for date in common_dates
        }
    # ----------------------------------------------------------------

    return spreads_pct

def convert_to_bps(spreads_pct: dict) -> dict:
    return { pair: {date: round(val * 100, 2) for date, val in series.items()} for pair, series in spreads_pct.items() }

def merge_timeseries_dict(filepath: str, new_data: dict) -> dict:
    old_data = {}
    if os.path.exists(filepath):
        try:
            with open(filepath, "r") as f: old_data = json.load(f)
        except Exception: pass
    merged_data = {}
    all_keys = set(old_data.keys()).union(set(new_data.keys()))
    for key in all_keys:
        merged_data[key] = {}
        if key in old_data: merged_data[key].update(old_data[key])
        if key in new_data: merged_data[key].update(new_data[key])
        merged_data[key] = dict(sorted(merged_data[key].items()))
    return merged_data

def save_spread_jsons(merged_returns: dict, spreads_bps: dict):
    with open("USA_bond_returns_by_grade.json", "w") as f: json.dump(merged_returns, f, indent=2)
    with open("USA_yield_spread_by_bond_grade.json", "w") as f: json.dump(spreads_bps, f, indent=2)

def print_spread_table(spreads_bps: dict):
    latest_date = list(list(spreads_bps.values())[-1].keys())[-1]
    print(f"\nLatest USA yield spreads as of {latest_date}:")
    print(f"{'Pair':<18} {'Date':<12} {'Spread (bps)':>14}")
    print(f"{'-'*18} {'-'*12} {'-'*14}")
    for pair, series in spreads_bps.items():
        date = list(series.keys())[-1]
        val  = series[date]
        print(f"{pair:<18} {date:<12} {val:>14.2f}")

def main():
    new_bond_returns = fetch_all_bond_returns(FETCH_ORDER, US_BOND_SERIES)
    merged_bond_returns = merge_timeseries_dict("USA_bond_returns_by_grade.json", new_bond_returns)
    adjacent_pairs = get_adjacent_pairs(SPREAD_ORDER)
    spreads_pct = compute_spreads_pct(merged_bond_returns, adjacent_pairs)
    spreads_bps = convert_to_bps(spreads_pct)
    save_spread_jsons(merged_bond_returns, spreads_bps)
    print_spread_table(spreads_bps)

if __name__ == "__main__":
    main()

Writing spread.py


In [ ]:
%%writefile pure_yield_forecast.py
import json
import math
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore")

from statsmodels.tsa.stattools import adfuller, grangercausalitytests
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import kpss as kpss_test



# -----------------------------------------------------------------------------
# 1. DATA PREPARATION
# -----------------------------------------------------------------------------
def load_and_prep_yields(filepath: str, include_fallen_angels: bool = False) -> pd.DataFrame:
    with open(filepath, "r") as f:
        data = json.load(f)

    df = pd.DataFrame(data)
    df.index = pd.to_datetime(df.index)
    df = df * 100

    hierarchy = ["AAA", "AA", "A", "BBB", "Fallen_Angel", "BB", "B", "CCC"]
    if not include_fallen_angels:
        hierarchy.remove("Fallen_Angel")

    ordered_cols = [col for col in hierarchy if col in df.columns]
    df = df[ordered_cols]

    df = df.resample("B").last().ffill()
    df = df.dropna()

    # FIX: Inject infinitesimal white-noise jitter to prevent singular covariance
    # matrices in the VAR estimator caused by zero-variance forward-filled periods.
    # 1e-8 bps is financially immaterial but keeps the matrix positive-definite.
    # CRITICAL: Use a LOCAL RandomState — never set the global np.random.seed()
    # in a data prep function, as it poisons downstream Monte Carlo reproducibility.
    _local_rng = np.random.RandomState(seed=42)
    jitter = _local_rng.normal(0, 1e-8, df.shape)
    df = df + jitter

    return df


# -----------------------------------------------------------------------------
# 2. THE MATH PIPELINE (Reusing your established logic)
# -----------------------------------------------------------------------------
def ensure_stationarity(df: pd.DataFrame) -> tuple:
    adf_results, differenced, was_differenced = {}, {}, {}
    for col in df.columns:
        series = df[col].dropna()

        adf_stat, adf_p, *_ = adfuller(series, autolag="AIC")
        try:
            _, kpss_p, _, _ = kpss_test(series, regression='c', nlags='auto')
        except Exception:
            kpss_p = 0.05  # conservative: assume non-stationary on KPSS failure

        # ADF H0: non-stationary. KPSS H0: stationary.
        # Stationary confirmed only when BOTH agree.
        adf_stationary = adf_p < 0.05      # ADF rejects non-stationarity
        kpss_stationary = kpss_p > 0.05   # KPSS fails to reject stationarity

        is_stationary = adf_stationary and kpss_stationary
        was_differenced[col] = not is_stationary
        differenced[col] = series if is_stationary else series.diff().dropna()

    stat_df = pd.DataFrame(differenced).dropna()
    stat_df.index.freq = pd.tseries.frequencies.to_offset("B")
    return stat_df, was_differenced


def select_lag_order(stat_df: pd.DataFrame, max_lags: int) -> int:
    model = VAR(stat_df)
    lag_result = model.select_order(maxlags=max_lags)
    return max(1, lag_result.aic)


def forecast_and_irf(fitted_model, stat_df: pd.DataFrame, original_df: pd.DataFrame, was_diff: dict,
                     horizon: int) -> dict:
    cols = list(stat_df.columns)
    lag_order = fitted_model.k_ar
    last_obs = stat_df.values[-lag_order:]

    raw_fc = fitted_model.forecast(last_obs, steps=horizon)
    future_dates = pd.bdate_range(start=original_df.index[-1], periods=horizon + 1)[1:]
    forecast_df = pd.DataFrame(raw_fc, index=future_dates, columns=cols)

    # Un-difference back to pure bps levels
    forecast_levels = {}
    for col in cols:
        if was_diff.get(col, False):
            forecast_levels[col] = original_df[col].iloc[-1] + forecast_df[col].cumsum()
        else:
            forecast_levels[col] = forecast_df[col]

    forecast_levels_df = pd.DataFrame(forecast_levels, index=future_dates)
    irf = fitted_model.irf(periods=horizon)

    return {"forecast_df": forecast_levels_df, "irf": irf, "future_dates": future_dates, "cols": cols}


# -----------------------------------------------------------------------------
# 3. CHARTING (Adapted for 8 assets instead of 6 pairs)
# -----------------------------------------------------------------------------
def plot_pure_forecast(original_df: pd.DataFrame, forecast_results: dict):
    cols, forecast_df, future_dates = forecast_results["cols"], forecast_results["forecast_df"], forecast_results[
        "future_dates"]
    n = len(cols)

    # Expanded color palette to handle 8 assets
    colors = ["#1D3557", "#457B9D", "#2A9D8F", "#E9C46A", "#F4A261", "#E63946", "#D62828", "#6A040F"]

    fig, axes = plt.subplots(n, 1, figsize=(15, 3.5 * n), sharex=False)

    for ax, col, color in zip(axes, cols, colors):
        hist_vals = original_df[col][original_df.index >= (original_df.index[-1] - pd.DateOffset(months=3))]
        fc_vals = forecast_df[col]

        ax.plot(hist_vals.index, hist_vals.values, color=color, linewidth=1.5, label="Historical (3 months)")
        ax.axvline(original_df.index[-1], color="black", linewidth=1.5, linestyle="--", alpha=0.5, label="Today")
        ax.axvspan(future_dates[0], future_dates[-1], alpha=0.10, color="#FF0000")

        ax.plot(fc_vals.index, fc_vals.values, color="#FF0000", linewidth=2.5, linestyle="--",
                marker="o", markersize=4, markerfacecolor="white", label="VAR Forecast")

        ax.set_title(f"{col} Pure Bond Yield (bps)", fontsize=10, fontweight="bold")
        ax.set_ylabel("Yield (bps)")
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8, loc="upper left")

        ax.xaxis.set_major_locator(mdates.DayLocator(interval=5))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %Y"))
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)

    fig.suptitle("VAR Forecast — Pure Bond Yields (bps)", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("pure_yield_forecast.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_pure_irf(forecast_results: dict):
    cols, irf, future_dates = forecast_results["cols"], forecast_results["irf"], forecast_results["future_dates"]
    n = len(cols)
    colors = ["#1D3557", "#457B9D", "#2A9D8F", "#E9C46A", "#F4A261", "#E63946", "#D62828", "#6A040F"]
    irf_dates = [future_dates[0] - pd.tseries.offsets.BDay(1)] + list(future_dates)

    configs = [
        {"data": irf.irfs, "img_file": "irf_pure_yield_matrix_1bps.png", "title": "IRF: 1 bps Absolute Yield Shock"},
        {"data": irf.orth_irfs, "img_file": "irf_pure_yield_matrix_stddev.png", "title": "IRF: 1 Std-Dev Yield Shock"}
    ]

    for config in configs:
        fig, axes = plt.subplots(n, n, figsize=(3 * n, 2.5 * n), sharex=False)
        for shock_idx in range(n):
            for resp_idx in range(n):
                ax = axes[resp_idx, shock_idx]
                response = config["data"][:, resp_idx, shock_idx]
                color = colors[resp_idx]

                ax.plot(irf_dates, response, color=color, linewidth=1.5)
                ax.fill_between(irf_dates, response, 0, where=(response > 0), alpha=0.2, color=color)
                ax.fill_between(irf_dates, response, 0, where=(response < 0), alpha=0.2, color="#AAAAAA")
                ax.axhline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.4)

                if resp_idx == 0: ax.set_title(f"Shock to {cols[shock_idx]}", fontsize=10, fontweight="bold")
                if shock_idx == 0: ax.set_ylabel(f"Resp: {cols[resp_idx]}\n(bps change)", fontsize=8, fontweight="bold")

                ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
                ax.xaxis.set_major_locator(mdates.DayLocator(interval=max(1, len(irf_dates) // 3)))
                ax.tick_params(axis='x', rotation=45, labelsize=7)
                ax.grid(True, alpha=0.2)

        fig.suptitle(config["title"], fontsize=16, fontweight="bold", y=1.02)
        plt.tight_layout(h_pad=1.0, w_pad=0.5)
        plt.savefig(config["img_file"], dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)


# -----------------------------------------------------------------------------
# 4. EXECUTION
# -----------------------------------------------------------------------------
def main():
    print("\n" + "=" * 65)
    print("  VAR PIPELINE — Pure Bond Yield Forecasting")
    print("=" * 65)

    # NOTE: Set include_fallen_angels=True if you want them, but be aware it truncates historical data to ~2016
    df = load_and_prep_yields("USA_bond_returns_by_grade.json", include_fallen_angels=False)

    T = len(df)
    dynamic_max_lags = math.floor(12 * ((T / 100) ** 0.25)) if T > 0 else 12
    horizon = 10  # Change this to whatever forecast horizon you want

    stat_df, was_diff = ensure_stationarity(df)
    lag = select_lag_order(stat_df, dynamic_max_lags)
    model = VAR(stat_df).fit(lag)

    results = forecast_and_irf(model, stat_df, df, was_diff, horizon)

    print("\n  [Rendering] Plotting Historic vs Forecast Yields...")
    plot_pure_forecast(df, results)

    print("  [Rendering] Plotting IRF Contagion Matrices...")
    plot_pure_irf(results)

    print("  [Success] Pure Yield analysis complete. Images saved.")


if __name__ == "__main__":
    main()


Writing pure_yield_forecast.py


In [ ]:
%%writefile private_credit.py
import pandas as pd
from statsmodels.tsa.stattools import acf, adfuller, kpss
import numpy as np
import warnings

# Calibrated kappa for illiquid private credit (slow mean reversion)
PRIVATE_CREDIT_OU_KAPPA = 0.002


def dynamic_unsmooth_private_credit(series: pd.Series, max_lags: int = 4) -> pd.Series:
    """
    Dynamically identifies the smoothing window q using the Autocorrelation Function (ACF)
    and applies an MA(q) generalized Geltner filter.
    """
    # Calculate ACF and 95% confidence intervals
    acf_vals, confint = acf(series.dropna(), nlags=max_lags, alpha=0.05)

    significant_lags = []
    # Iterate through lags to find statistically significant autocorrelation
    for i in range(1, len(acf_vals)):
        # If the lower bound is > 0, the autocorrelation is significant
        if confint[i][0] > 0:
            significant_lags.append((i, acf_vals[i]))
        else:
            break  # Stop at the first insignificant lag

    if not significant_lags:
        print("    [MATH] No significant smoothing detected. Skipping filter.")
        return series

    q = len(significant_lags)
    rho_sum = sum(val for _, val in significant_lags)

    # Cap rho_sum to prevent division by zero or negative variance explosions
    rho_sum = min(rho_sum, 0.99)
    print(f"    [MATH] MA({q}) Smoothing Detected. Applying Generalized Filter (\u03a3\u03c1: {rho_sum:.4f}).")

    unsmoothed = series.copy()

    # Apply generalized MA(q) unsmoothing:
    # R_true = (R_obs - Sum(rho * R_past)) / (1 - Sum(rho))
    for i in range(q, len(series)):
        weighted_past = sum(rho * series.iloc[i - lag] for lag, rho in significant_lags)
        unsmoothed.iloc[i] = (series.iloc[i] - weighted_past) / (1 - rho_sum)

    return unsmoothed.dropna()


def get_frac_weights(d: float, size: int) -> np.ndarray:
    w = [1.]
    for k in range(1, size):
        w.append(-w[-1] / k * (d - k + 1))
    # FIX: Return as a standard 1D array instead of reshaping to 2D
    return np.array(w[::-1])


def frac_diff(series: pd.Series, d: float, thres: float = 1e-4) -> pd.Series:
    w = get_frac_weights(d, len(series))
    w = w[np.abs(w) >= thres]
    df_diff = pd.Series(index=series.index, dtype=float)

    for i in range(len(w), len(series)):
        # FIX: w is 1D, so np.dot returns a scalar directly. Removed the [0] at the end.
        df_diff.iloc[i] = np.dot(w, series.iloc[i - len(w):i])

    return df_diff.dropna()


def enforce_stationarity(series: pd.Series, max_d: float = 1.0, step: float = 0.1) -> pd.Series:
    """
    Circuit breaker: Iteratively applies fractional differencing until the series
    passes both ADF (H0: Non-stationary) and KPSS (H0: Stationary) tests.
    """
    warnings.simplefilter("ignore")  # Ignore KPSS p-value interpolation warnings

    for d in np.arange(0.0, max_d + step, step):
        test_series = frac_diff(series, d) if d > 0 else series.dropna()

        if len(test_series) < 20:
            break

        adf_p = adfuller(test_series, autolag='AIC')[1]
        kpss_p = kpss(test_series, regression='c', nlags='auto')[1]

        # Circuit breaker criteria: ADF < 0.05 AND KPSS > 0.05
        if adf_p < 0.05 and kpss_p > 0.05:
            print(f"    [STATIONARITY] Strict Stationarity achieved at d={d:.1f}")
            return test_series

    print("    [WARNING] Strict stationarity failed. Defaulting to 1st difference (d=1.0).")
    return series.diff().dropna()


def prepare_private_credit_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Runs the unsmoothing engine across all provided asset columns.
    """
    unsmoothed_df = pd.DataFrame(index=df.index)
    for col in df.columns:
        if col != "Risk_Free":
            print(f"  Processing Private Credit Asset: {col}")
            unsmoothed_df[col] = dynamic_unsmooth_private_credit(df[col])
            unsmoothed_df[col] = enforce_stationarity(unsmoothed_df[col])
        else:
            unsmoothed_df[col] = df[col]

    return unsmoothed_df.dropna()

Writing private_credit.py


In [ ]:
%%writefile ml_engine.py
import logging
import numpy as np
import pandas as pd
import shap
import torch
import yfinance as yf
from sklearn.ensemble import IsolationForest

# Ensemble Imports
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit

logger = logging.getLogger(__name__)

# Institutional ETF Mapping
ETF_NAMES = {
    "SHY": "1-3 Year Treasury (SHY)",
    "TLT": "20+ Year Treasury (TLT)",
    "LQD": "Inv. Grade Corporate (LQD)",
    "HYG": "High Yield Corporate (HYG)",
    "ANGL": "Fallen Angels (ANGL)",
    "PFF": "Preferred Stock (PFF)"
}


def format_feature_name(name: str) -> str:
    """Translates Python variable names into readable institutional terminology."""
    name = name.replace("OAS_Z", "OAS Z-Score")
    name = name.replace("Term_Spread", "Term Spread")
    name = name.replace("Liquidity_Proxy", "Liquidity Proxy")
    for w in [21, 63, 252]:
        name = name.replace(f"Sharpe_{w}", f"{w}-Day Sharpe Ratio")
        name = name.replace(f"Sortino_{w}", f"{w}-Day Sortino Ratio")
        name = name.replace(f"Calmar_{w}", f"{w}-Day Calmar Ratio")
    return name


def run_ml_pipeline(dataset: dict) -> pd.DataFrame:
    logger.info("Initializing Liquid Super-Learner: XGBoost + LightGBM + CatBoost Stack...")

    if not dataset:
        logger.error("Dataset is empty. Aborting ML Pipeline.")
        return pd.DataFrame()

    all_data = []
    for ticker, df in dataset.items():
        temp = df.copy()
        temp['Asset'] = ticker
        all_data.append(temp)

    master_df = pd.concat(all_data)

    # -------------------------------------------------------------------------
    # Calculate 21-Day ROI (Momentum Baseline)
    # -------------------------------------------------------------------------
    master_df['Trailing_21D_ROI'] = master_df.groupby('Asset')['Close'].pct_change(periods=21) * 100

    exclude_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Ret', 'Target', 'Asset', 'Trailing_21D_ROI']
    features = [c for c in master_df.columns if c not in exclude_cols]

    train_df = master_df.dropna(subset=['Target']).copy()
    inference_df = master_df[master_df['Target'].isna()].groupby('Asset').tail(1).copy()

    if inference_df.empty:
        logger.warning("No inference data found (Missing NaN targets).")
        return pd.DataFrame()

    X_train = train_df[features]
    y_train = train_df['Target']
    X_latest = inference_df[features]

    # Anomaly Detection
    iso = IsolationForest(contamination=0.02, random_state=42)
    train_df['Anomaly'] = iso.fit_predict(X_train)
    clean_train = train_df[train_df['Anomaly'] == 1]

    # -------------------------------------------------------------------------
    # SUPER-LEARNER ENSEMBLE ARCHITECTURE (REMEDIATED SR 11-7)
    # -------------------------------------------------------------------------
    _USE_GPU = torch.cuda.is_available()

    base_learners = [
        ('xgb', XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.05,
                              random_state=42, eval_metric='logloss',
                              tree_method='hist', device='cuda' if _USE_GPU else 'cpu')),
        ('lgb', LGBMClassifier(n_estimators=150, max_depth=4, learning_rate=0.05,
                               verbose=-1, random_state=42,
                               device='gpu' if _USE_GPU else 'cpu')),
        ('cat', CatBoostClassifier(iterations=150, depth=4, learning_rate=0.05,
                                   verbose=0, random_state=42,
                                   task_type='GPU' if _USE_GPU else 'CPU'))
    ]

    X_train_arr = clean_train[features].values
    y_train_arr = clean_train['Target'].values

    tss = TimeSeriesSplit(n_splits=5)
    oof_meta = np.zeros((len(X_train_arr), len(base_learners)))
    valid_mask = np.zeros(len(X_train_arr), dtype=bool)

    # Generate valid OOF predictions strictly chronologically
    for train_idx, test_idx in tss.split(X_train_arr):
        for col_idx, (name, model) in enumerate(base_learners):
            m = model.__class__(**model.get_params())
            m.fit(X_train_arr[train_idx], y_train_arr[train_idx])
            oof_meta[test_idx, col_idx] = m.predict_proba(X_train_arr[test_idx])[:, 1]
        valid_mask[test_idx] = True

    # Fit the meta-learner ONLY on chronologically valid OOF rows
    lr_meta = LogisticRegression()
    lr_meta.fit(oof_meta[valid_mask], y_train_arr[valid_mask])

    # Train final base estimators on the FULL dataset for live inference
    final_base_estimators = []
    for name, model in base_learners:
        m = model.__class__(**model.get_params())
        m.fit(X_train_arr, y_train_arr)
        final_base_estimators.append((name, m))

    # Wrap in custom pipeline class compatible with downstream SHAP logic
    class ChronologicalStacker:
        def __init__(self, base, meta):
            self.named_estimators_ = {n: m for n, m in base}
            self._base = base
            self._meta = meta

        def predict_proba(self, X):
            if isinstance(X, pd.DataFrame): X = X.values
            meta_input = np.column_stack([m.predict_proba(X)[:, 1] for _, m in self._base])
            return self._meta.predict_proba(meta_input)

    stacked_model = ChronologicalStacker(final_base_estimators, lr_meta)

    # Predict using the full, untainted stack
    inference_df['ML_Score'] = stacked_model.predict_proba(X_latest)[:, 1] * 100
    inference_df['Anomaly'] = iso.predict(X_latest)

    # -------------------------------------------------------------------------
    # SHAP EXTRACTION (Ensemble-Weighted Approximation)
    # -------------------------------------------------------------------------
    meta_weights = stacked_model._meta.coef_[0]
    weight_xgb, weight_lgb, weight_cat = meta_weights / np.sum(np.abs(meta_weights))

    xgb_layer = stacked_model.named_estimators_['xgb']
    lgb_layer = stacked_model.named_estimators_['lgb']
    cat_layer = stacked_model.named_estimators_['cat']

    shap_xgb = shap.TreeExplainer(xgb_layer).shap_values(X_latest)
    shap_lgb = shap.TreeExplainer(lgb_layer).shap_values(X_latest)
    shap_cat = shap.TreeExplainer(cat_layer).shap_values(X_latest)

    if isinstance(shap_xgb, list): shap_xgb = shap_xgb[1]
    elif hasattr(shap_xgb, 'values'): shap_xgb = shap_xgb.values

    if isinstance(shap_lgb, list): shap_lgb = shap_lgb[1]
    elif hasattr(shap_lgb, 'values'): shap_lgb = shap_lgb.values

    if isinstance(shap_cat, list): shap_cat = shap_cat[1]
    elif hasattr(shap_cat, 'values'): shap_cat = shap_cat.values

    ensemble_shap = (shap_xgb * weight_xgb) + (shap_lgb * weight_lgb) + (shap_cat * weight_cat)

    profit_reasons = []
    risk_reasons = []

    for i in range(len(X_latest)):
        sv = ensemble_shap[i]

        pos_indices = np.where(sv > 0)[0]
        pos_sorted = pos_indices[np.argsort(sv[pos_indices])][::-1][:3]
        pos_str = " | ".join([f"{format_feature_name(features[j])} (+{sv[j]:.2f})" for j in pos_sorted]) if len(pos_sorted) > 0 else "None"
        profit_reasons.append(pos_str)

        neg_indices = np.where(sv < 0)[0]
        neg_sorted = neg_indices[np.argsort(sv[neg_indices])][:3]
        neg_str = " | ".join([f"{format_feature_name(features[j])} ({sv[j]:.2f})" for j in neg_sorted]) if len(neg_sorted) > 0 else "None"
        risk_reasons.append(neg_str)

    inference_df['Profit_Drivers'] = profit_reasons
    inference_df['Risk_Drivers'] = risk_reasons

    # --- THE RESTORED MISSING CODE ---
    inference_df['Asset_Name'] = inference_df['Asset'].map(ETF_NAMES).fillna(inference_df['Asset'])

    # -------------------------------------------------------------------------
    # Fetch Live ETF Yields
    # -------------------------------------------------------------------------
    logger.info("Fetching real-time dividend yields for scorecard...")
    yields = []
    for t in inference_df['Asset']:
        try:
            ticker_obj = yf.Ticker(t)
            yld = ticker_obj.info.get('yield')
            if yld is None:
                yld = ticker_obj.info.get('dividendYield', 0.0)
            yields.append((yld or 0.0) * 100)
        except Exception as e:
            logger.warning(f"Could not fetch yield for {t}: {e}")
            yields.append(0.0)

    inference_df['Current_Yield'] = yields

    return inference_df

Writing ml_engine.py


In [ ]:
%%writefile main_alpha.py
import matplotlib.pyplot as plt
import seaborn as sns
import data_engine
import ml_engine
import textwrap


def plot_ml_scorecard(scorecard_df):
    """Outputs the Main ML Scorecard with Yield, ROI, and split Profit/Risk SHAP values."""
    print("\n[UI] Rendering Machine Learning Scorecard...")

    df = scorecard_df.copy()
    df['ML_Score_Str'] = df['ML_Score'].apply(lambda x: f"{x:.1f}%")
    df['ROI_Str'] = df['Trailing_21D_ROI'].apply(lambda x: f"{x:+.2f}%")
    df['Status'] = df['Anomaly'].apply(lambda x: "SAFE" if x == 1 else "ANOMALY (DO NOT TRADE)")

    # NEW: Format the Yield String (Handles missing data gracefully)
    df['Yield_Str'] = df['Current_Yield'].apply(lambda x: f"{x:.2f}%" if x > 0 else "N/A")

    # Textwrap prevents the image from breaking horizontally
    df['Profit_Drivers'] = df['Profit_Drivers'].apply(lambda x: "\n".join(textwrap.wrap(x, width=32)))
    df['Risk_Drivers'] = df['Risk_Drivers'].apply(lambda x: "\n".join(textwrap.wrap(x, width=32)))

    # Added 'Yield_Str' to the table_data array
    table_data = df[['Asset_Name', 'ML_Score_Str', 'Status', 'Yield_Str', 'ROI_Str', 'Profit_Drivers',
                     'Risk_Drivers']].values.tolist()

    # Added "Current\nYield" to the headers
    columns = ["Credit Asset", "Win\nProbability", "System\nStatus", "Current\nYield", "21-Day ROI\n(Momentum)",
               "Top 3 Profit Drivers\n(Positive SHAP)", "Top 3 Risk Drivers\n(Negative SHAP)"]

    # Expanded canvas width from 26 to 28 to give the new column space
    fig, ax = plt.subplots(figsize=(28, 10))
    ax.axis('tight')
    ax.axis('off')

    plt.title("Institutional ML Credit Alpha Scorecard (21-Day Forward Horizon)", fontweight="bold", fontsize=18,
              pad=30)

    table = ax.table(cellText=table_data, colLabels=columns, cellLoc='center', bbox=[0, 0, 1, 0.85])
    table.auto_set_font_size(False)
    table.set_fontsize(11)

    # MATHEMATICAL REBALANCE: Sum exactly equals 1.00
    col_widths = {0: 0.16, 1: 0.07, 2: 0.07, 3: 0.07, 4: 0.09, 5: 0.27, 6: 0.27}

    for (row, col), cell in table.get_celld().items():
        cell.set_width(col_widths[col])

        if row == 0:
            cell.set_facecolor('#1D3557')
            cell.set_text_props(weight='bold', color='white')
        elif col == 1:
            val = float(table_data[row - 1][1].replace('%', ''))
            cell.set_facecolor('#E6F4EA') if val > 55 else cell.set_facecolor('#FCE8E6')
            cell.set_text_props(weight='bold')
        elif col == 2 and table_data[row - 1][2] != "SAFE":
            cell.set_facecolor('#FFCCCB')
            cell.set_text_props(weight='bold')
        elif col == 3:  # NEW: Yield Column Background
            cell.set_facecolor('#F8F9FA')
            cell.set_text_props(weight='bold')
        elif col == 4:  # Color code ROI momentum (Shifted to col 4)
            val = float(table_data[row - 1][4].replace('%', ''))
            cell.set_text_props(color='green' if val > 0 else 'red', weight='bold')
        elif col == 5:  # Profit Drivers Background (Shifted to col 5)
            cell.set_facecolor('#F6FDF8')
            cell.set_text_props(ha='left')
        elif col == 6:  # Risk Drivers Background (Shifted to col 6)
            cell.set_facecolor('#FFF9F9')
            cell.set_text_props(ha='left')

    plt.tight_layout()
    plt.savefig("ML_Credit_Scorecard.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(" -> Saved 'ML_Credit_Scorecard.png'")

def plot_individual_asset_metrics(scorecard_df):
    """Outputs a deeply comprehensive ratio table for each of the 6 Credit ETFs."""
    print("\n[UI] Rendering Comprehensive Asset Profiles (This generates 6 images)...")

    # The complete list of quantitative features engineered by the engine
    metrics = [
        "Liquidity_Proxy", "Term_Spread", "OAS_Z",
        "Sharpe_21", "Sharpe_63", "Sharpe_252",
        "Sortino_21", "Sortino_63", "Sortino_252",
        "Calmar_21", "Calmar_63", "Calmar_252"
    ]

    for index, row in scorecard_df.iterrows():
        asset_full = row['Asset_Name']
        ticker = row['Asset']

        table_data = []

        for m in metrics:
            if m in row:
                val = row[m]
                nice_name = ml_engine.format_feature_name(m)
                table_data.append([nice_name, f"{val:.4f}"])

        fig, ax = plt.subplots(figsize=(8, len(table_data) * 0.4 + 2))
        ax.axis('tight')
        ax.axis('off')
        plt.title(f"Quantitative Ratio Profile: {asset_full}", fontweight="bold", fontsize=14, pad=20)

        table = ax.table(cellText=table_data, colLabels=["Quantitative Metric", "Current Computed Value"], loc='center',
                         cellLoc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(11)
        table.scale(1, 2.0)

        for (r, c), cell in table.get_celld().items():
            if r == 0:
                cell.set_facecolor('#1D3557')
                cell.set_text_props(weight='bold', color='white')
            elif c == 0:
                cell.set_text_props(weight='bold', ha='left')
                cell.set_facecolor('#F8F9FA')

        plt.tight_layout()
        filename = f"{ticker}_Comprehensive_Profile.png"
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        plt.show()
        print(f" -> Saved '{filename}'")


def plot_vol_vs_spread(dataset):
    print("\n[UI] Rendering Pure Price Volatility vs Yield Spread Volatility...")
    tlt = dataset['TLT']['Ret'].rolling(21).std() * (252 ** 0.5) * 100
    hyg = dataset['HYG']['Ret'].rolling(21).std() * (252 ** 0.5) * 100

    plt.figure(figsize=(14, 6))
    plt.plot(tlt.tail(252), label="Pure Duration Vol (TLT 20Y+)", color="blue")
    plt.plot(hyg.tail(252), label="Credit Spread Vol (HYG Junk)", color="red")
    plt.title("Volatility Regime: Pure Duration vs. High Yield Spreads", fontweight="bold")
    plt.ylabel("Annualized Volatility (%)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig("Vol_vs_Spread.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(" -> Saved 'Vol_vs_Spread.png'")


def run():
    """Callable entry point for the Master Pipeline (main.py)"""
    print("=" * 70)
    print("  INSTITUTIONAL ML CREDIT ALPHA ENGINE (XGBoost + SHAP) ")
    print("=" * 70)

    dataset = data_engine.generate_master_dataset()
    if not dataset:
        print("[CRITICAL] Dataset failed to build. Check network and FRED API key.")
        return

    scorecard = ml_engine.run_ml_pipeline(dataset)

    while True:
        print("\n[DASHBOARD CONTROLS]")
        print("  [1] Output ML Predictive Scorecard (With Profit/Risk Drivers & ROI)")
        print("  [2] Output Comprehensive Asset Profiles (All Quantitative Ratios)")
        print("  [3] Output Volatility Regime (Duration vs Spread Vol)")
        print("  [4] Return to Main System (Exit Module)")
        choice = input("Select Module [1/2/3/4]: ").strip()

        if choice == '1':
            if scorecard.empty:
                print("\n  [ERROR] Scorecard data is missing. Cannot render table.")
            else:
                plot_ml_scorecard(scorecard)
        elif choice == '2':
            if scorecard.empty:
                print("\n  [ERROR] Scorecard data is missing.")
            else:
                plot_individual_asset_metrics(scorecard)
        elif choice == '3':
            plot_vol_vs_spread(dataset)
        elif choice == '4':
            print("Shutting down ML environment.")
            break
        else:
            print("Invalid input.")


if __name__ == "__main__":
    run()

Writing main_alpha.py


In [ ]:
%%writefile global_private_credit.py
import pandas as pd
import numpy as np
import yfinance as yf
import seaborn as sns
import matplotlib.pyplot as plt
import textwrap

# -----------------------------------------------------------------------------
# DIRECT LENDING ASSUMPTIONS & CONSTANTS
# -----------------------------------------------------------------------------
PRIVATE_CREDIT_ILLIQUIDITY_PREMIUM = 0.0200
# REMOVED: SENIOR_SECURED_LGD = 0.40 (Now dynamically calculated)

def calculate_implied_recovery(spread: float, pd: float, liquidity_premium: float) -> float:
    """
    Extracts the market-implied recovery rate from the credit spread.
    Formula: Recovery = 1 - [(Total Spread - Liquidity Premium) / PD]
    """
    # 1. Isolate the pure default risk component (Floor at 10 bps to prevent div-by-zero)
    pure_credit_spread = max(spread - liquidity_premium, 0.0010)

    # 2. Calculate implied Loss Given Default (LGD)
    implied_lgd = pure_credit_spread / max(pd, 0.0001)

    # 3. Calculate Recovery and cap it between 5% (wipeout) and 85% (historical max for senior secured)
    implied_recovery = 1.0 - implied_lgd
    return float(np.clip(implied_recovery, 0.05, 0.85))

# UPGRADE: Added "Spread" (Credit Risk Premium) to dynamically price each tier
BORROWER_PROFILES = {
    95: {"Rating": "BBB to BBB-", "Spread": 0.0150, "PD": 0.0010, "Leverage": "< 2.5x", "ICR": "> 4.0x",
         "FCCR": "> 3.0x", "LTV": "< 40%",
         "Profile": "Dominant market leader. Massive, predictable FCF. Top-tier sponsor willing to inject capital."},
    80: {"Rating": "BB+ to BB", "Spread": 0.0300, "PD": 0.0075, "Leverage": "2.5x - 4.0x", "ICR": "2.5x - 4.0x",
         "FCCR": "2.0x - 3.0x", "LTV": "40% - 55%",
         "Profile": "Established top-3 player. Strong FCF covering all debt. Sponsor backing with >50% equity."},
    60: {"Rating": "B+ to B", "Spread": 0.0450, "PD": 0.0250, "Leverage": "4.0x - 5.5x", "ICR": "1.7x - 2.5x",
         "FCCR": "1.2x - 2.0x", "LTV": "55% - 70%",
         "Profile": "Core Middle Market. Vulnerable to margin compression. Standard PE buyout structure."},
    40: {"Rating": "B-", "Spread": 0.0650, "PD": 0.0500, "Leverage": "5.5x - 7.0x", "ICR": "1.2x - 1.7x",
         "FCCR": "1.0x - 1.2x", "LTV": "70% - 85%",
         "Profile": "Highly cyclical or concentrated. Tight FCF. Aggressive LBO with thin equity (<30%)."},
    20: {"Rating": "CCC+ to CCC", "Spread": 0.0950, "PD": 0.1200, "Leverage": "7.0x - 8.5x", "ICR": "1.0x - 1.2x",
         "FCCR": "0.8x - 1.0x", "LTV": "85% - 95%",
         "Profile": "Declining market share. Flat/negative FCF. Sponsor hesitant to inject more capital."},
    10: {"Rating": "CCC- to CC", "Spread": 0.1400, "PD": 0.2000, "Leverage": "8.5x+", "ICR": "< 1.0x", "FCCR": "< 0.8x",
         "LTV": "95% - 100%",
         "Profile": "Burning cash. Cannot cover interest expense. Sponsor has abandoned the asset."},
    5: {"Rating": "C to D", "Spread": 0.1800, "PD": 0.3500, "Leverage": "Meaningless", "ICR": "< 0.5x",
        "FCCR": "< 0.5x", "LTV": "> 100%",
        "Profile": "Fundamentally broken. Relying on revolvers to survive. Preparing for restructuring."}
}


def fetch_regional_base_rates() -> dict:
    """
    Instead of fetching gross yields, we fetch the Risk-Free / Base Rates.
    We proxy this by taking the ETF Yield and stripping out the average B-rated spread (450 bps).
    """
    print("\n  [API] Fetching Global Base Lending Rates (yfinance)...")
    proxies = {
        "USA (Broad)": "JNK",
        "Europe (Aggregate)": "IHYG.L",
        "Emerging Markets": "EMHY",
        "United Kingdom": "IS15.L",
        "Global (Aggregate)": "HYG"
    }
    regional_base_rates = {}
    for region, ticker in proxies.items():
        try:
            t = yf.Ticker(ticker)
            hist = t.history(period="1y")
            if not hist.empty:
                ttm_div = hist["Dividends"].sum()
                current_price = hist["Close"].iloc[-1]
                public_yield = ttm_div / current_price

                # Extract the Base Rate by subtracting the average B-rated spread (450 bps)
                base_rate = max(public_yield - 0.0450, 0.02)
                regional_base_rates[region] = base_rate
                print(f"    OK   {region}: {base_rate * 100:.2f}% Implied Base Rate")
            else:
                regional_base_rates[region] = 0.04
        except Exception as e:
            print(f"    FAIL {region}: {e}")
            regional_base_rates[region] = 0.04

    base_eur = regional_base_rates.get("Europe (Aggregate)", 0.03)
    regional_base_rates["Germany (Premium)"] = base_eur - 0.010
    regional_base_rates["France (Core)"] = base_eur - 0.005
    regional_base_rates["Spain (Peripheral)"] = base_eur + 0.005
    regional_base_rates["Italy (Peripheral)"] = base_eur + 0.010

    return regional_base_rates


def generate_lending_matrix():
    regional_base_rates = fetch_regional_base_rates()
    percentiles = sorted(list(BORROWER_PROFILES.keys()), reverse=True)

    df_net_yield = pd.DataFrame(index=list(regional_base_rates.keys()),
                                columns=[f"Top {p}%" for p in percentiles], dtype=float)

    print("\n  [MATH] Calculating Risk-Adjusted Gross Yields and Net Yields...")

    # 1. HEATMAP MATH: Unique Gross Yields per Cell
    for region, base_rate in regional_base_rates.items():
        for p in percentiles:
            prof = BORROWER_PROFILES[p]
            specific_gross_yield = base_rate + PRIVATE_CREDIT_ILLIQUIDITY_PREMIUM + prof["Spread"]

            # THE FIX: Synthetic dynamic expected loss
            dynamic_recovery = calculate_implied_recovery(prof["Spread"], prof["PD"], PRIVATE_CREDIT_ILLIQUIDITY_PREMIUM)
            dynamic_lgd = 1.0 - dynamic_recovery
            expected_loss = prof["PD"] * dynamic_lgd

            net_yield = specific_gross_yield - expected_loss
            df_net_yield.at[region, f"Top {p}%"] = net_yield * 100

    # RENDER 1: HEATMAP
    plt.figure(figsize=(14, 8))
    sns.heatmap(df_net_yield, annot=True, fmt=".2f", cmap="RdYlGn", center=df_net_yield.values.mean(),
                cbar_kws={'label': 'Net Expected Yield (%)'}, annot_kws={"size": 10})
    plt.title("Global Private Credit: Risk-Adjusted Net Lending Yields (%)", fontsize=16, fontweight="bold", pad=20)
    plt.ylabel("Geographic Lending Market", fontweight="bold")
    plt.xlabel("Borrower Quality Percentile", fontweight="bold")
    plt.tight_layout()
    plt.savefig("global_private_credit_net_yields.png", dpi=150)
    plt.show()
    plt.close()

    # RENDER 2: THE COMPREHENSIVE UNDERWRITING TABLE
    global_base = regional_base_rates.get("Global (Aggregate)", 0.04)

    table_data = []
    columns = ["Percentile", "Implied\nPublic Rating", "Leverage\n(Debt/EBITDA)",
               "Interest\nCoverage", "Fixed Charge\nCoverage", "Loan-to-\nValue",
               "Dynamic\nGross Yield", "Expected\nLoss (EL)", "Dynamic\nNet Yield",
               "Borrower & Sponsor Profile"]

    for p in percentiles:
        prof = BORROWER_PROFILES[p]

        # DYNAMIC GROSS YIELD FOR THE TABLE
        dynamic_gross = global_base + PRIVATE_CREDIT_ILLIQUIDITY_PREMIUM + prof["Spread"]

        # THE FIX: Synthetic dynamic expected loss applied to the table generation
        dynamic_recovery = calculate_implied_recovery(prof["Spread"], prof["PD"], PRIVATE_CREDIT_ILLIQUIDITY_PREMIUM)
        dynamic_lgd = 1.0 - dynamic_recovery
        el = prof["PD"] * dynamic_lgd

        net_y = dynamic_gross - el
        wrapped_profile = "\n".join(textwrap.wrap(prof["Profile"], width=65))

        # RE-ADDED: Append the newly calculated dynamic variables to the matrix
        table_data.append([
            f"Top {p}%", prof["Rating"], prof["Leverage"], prof["ICR"],
            prof["FCCR"], prof["LTV"],
            f"{dynamic_gross * 100:.2f}%",
            f"{el * 100:.2f}%",
            f"{net_y * 100:.2f}%",
            wrapped_profile
        ])

    fig, ax = plt.subplots(figsize=(26, 12))
    ax.axis('tight')
    ax.axis('off')

    # FIX: Updated header text to reflect the dynamic LGD rather than the deleted constant
    header_text = (
        "GLOBAL PRIVATE CREDIT UNDERWRITING PROFILE & CORRELATION MATRIX\n"
        f"Dynamic Benchmark: Global Aggregate Implied Base Rate | Model Assumptions: Market-Implied LGD, "
        f"Illiquidity Premium = {PRIVATE_CREDIT_ILLIQUIDITY_PREMIUM * 10000:.0f} bps\n"
        "Formula: Gross Yield = Base Rate + Illiquidity Premium + Risk Spread  |  Net Yield = Gross Yield - (PD × LGD)"
    )
    plt.title(header_text, fontweight="bold", fontsize=16, loc="left", pad=20, color="#1D3557")

    table = ax.table(cellText=table_data, colLabels=columns, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(12)

    col_widths = {0: 0.06, 1: 0.08, 2: 0.08, 3: 0.08, 4: 0.08, 5: 0.08, 6: 0.07, 7: 0.07, 8: 0.08, 9: 0.32}

    for (row, col), cell in table.get_celld().items():
        cell.set_width(col_widths[col])
        if row == 0:
            cell.set_facecolor('#1D3557')
            cell.set_text_props(weight='bold', color='white')
        else:
            if col == 8:  # Dynamic Net Yield Column
                val = float(table_data[row - 1][8].replace('%', ''))
                if val > 8.0:
                    cell.set_facecolor('#E6F4EA')
                elif val > 4.0:
                    cell.set_facecolor('#FFF3CD')
                else:
                    cell.set_facecolor('#FCE8E6')
                cell.set_text_props(weight='bold')
            elif col == 9:
                cell.set_text_props(ha='left')

    table.scale(1.0, 5.0)
    plt.tight_layout()
    plt.savefig("private_credit_borrower_profile_table.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

    print("  [SUCCESS] Rendered global_private_credit_net_yields.png")
    print("  [SUCCESS] Rendered private_credit_borrower_profile_table.png")

    return df_net_yield


if __name__ == "__main__":
    generate_lending_matrix()

Writing global_private_credit.py


In [ ]:
%%writefile forecast.py
import json
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
warnings.filterwarnings("ignore")

from statsmodels.tsa.stattools  import adfuller, grangercausalitytests
from statsmodels.tsa.api        import VAR
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM

def build_dataframe(diff_over_time: dict) -> pd.DataFrame:
    frames = {}
    hierarchy = ["AA - AAA", "A - AA", "BBB - A", "BB - Fallen_Angel", "BB - BBB", "B - BB", "CCC - B"]

    if not diff_over_time:
        return pd.DataFrame()

    for pair, series in diff_over_time.items():
        s = pd.Series(series)
        s.index = pd.to_datetime(s.index)
        frames[pair] = s

    df = pd.DataFrame(frames).sort_index()
    ordered_cols = [col for col in hierarchy if col in df.columns]
    df = df[ordered_cols]

    if df.empty:
        return df

    df = df.resample("B").last().ffill()
    df = df.dropna()
    return df

def ensure_stationarity(df: pd.DataFrame) -> tuple:
    print("\n" + "=" * 65)
    print("STEP 1 — ADF Stationarity Test")
    print("=" * 65)
    print(f"  {'Pair':<18} {'ADF stat':>10} {'p-value':>10}  Result")
    print(f"  {'─'*18} {'─'*10} {'─'*10}  {'─'*28}")

    adf_results, differenced, was_differenced = {}, {}, {}
    for col in df.columns:
        series               = df[col].dropna()
        adf_stat, p, *_      = adfuller(series, autolag="AIC")
        is_stationary        = p < 0.05
        was_differenced[col] = not is_stationary
        differenced[col]     = series if is_stationary else series.diff().dropna()
        result               = "Stationary" if is_stationary else "Non-stationary → differenced"
        adf_results[col]     = {"adf_stat": round(adf_stat, 4), "p_value" : round(p, 6), "differenced": was_differenced[col]}
        print(f"  {col:<18} {adf_stat:>10.4f} {p:>10.6f}  {result}")

    stationary_df = pd.DataFrame(differenced).dropna()
    stationary_df.index.freq = pd.tseries.frequencies.to_offset("B")
    return stationary_df, adf_results, was_differenced

def select_lag_order(stationary_df: pd.DataFrame, max_lags: int = 12) -> int:
    model = VAR(stationary_df)
    lag_result = model.select_order(maxlags=max_lags)
    optimal = lag_result.selected_orders.get('aic', 1)
    optimal = int(max(1, optimal))
    return optimal

def fit_var_model(stationary_df: pd.DataFrame, lag_order: int):
    fitted = VAR(stationary_df).fit(lag_order)
    return fitted

def run_granger_causality(stationary_df: pd.DataFrame, lag_order: int) -> dict:
    cols, results = list(stationary_df.columns), {}
    for i in range(len(cols) - 1):
        col_a, col_b = cols[i], cols[i + 1]
        for causing, target in [(col_a, col_b), (col_b, col_a)]:
            label = f"{causing[:8]} → {target[:8]}"
            try:
                test_data = stationary_df[[target, causing]].dropna()
                gc        = grangercausalitytests(test_data, maxlag=lag_order, verbose=False)
                p         = gc[lag_order][0]["ssr_ftest"][1]
                results[f"{causing} → {target}"] = {"p_value": round(p, 6)}
            except Exception as e:
                pass
    return results

def forecast_and_irf(fitted_model, stationary_df: pd.DataFrame, original_df: pd.DataFrame, was_differenced: dict, horizon: int = 12) -> dict:
    cols = list(stationary_df.columns)
    lag_order = fitted_model.k_ar
    last_obs  = stationary_df.values[-lag_order:]
    raw_fc    = fitted_model.forecast(last_obs, steps=horizon)

    last_date    = original_df.index[-1]
    future_dates = pd.bdate_range(start=last_date, periods=horizon + 1)[1:]
    forecast_df = pd.DataFrame(raw_fc, index=future_dates, columns=cols)

    forecast_levels = {}
    for col in cols:
        if was_differenced.get(col, False):
            last_level           = original_df[col].iloc[-1]
            forecast_levels[col] = last_level + forecast_df[col].cumsum()
        else:
            forecast_levels[col] = forecast_df[col]

    forecast_levels_df = pd.DataFrame(forecast_levels, index=future_dates)
    irf = fitted_model.irf(periods=horizon)
    return {"forecast_df" : forecast_levels_df, "irf" : irf, "future_dates" : future_dates, "cols" : cols}

def save_future_projections(forecast_levels_df: pd.DataFrame):
    out = {}
    for col in forecast_levels_df.columns:
        out[col] = {str(date.date()): round(val, 4) for date, val in forecast_levels_df[col].items()}
    with open("future_projections.json", "w") as f:
        json.dump(out, f, indent=2)

def plot_forecast(original_df: pd.DataFrame, forecast_results: dict):
    cols         = forecast_results["cols"]
    forecast_df  = forecast_results["forecast_df"]
    future_dates = forecast_results["future_dates"]
    n            = len(cols)

    hist_colors = ["#E63946", "#F4A261", "#2A9D8F", "#8338EC", "#457B9D", "#D4AF37", "#06D6A0"]
    fig, axes = plt.subplots(n, 1, figsize=(15, 3.8 * n), sharex=False)
    if n == 1: axes = [axes]

    for ax, col, hist_color in zip(axes, cols, hist_colors):
        last_date = original_df.index[-1]
        one_month_ago = last_date - pd.DateOffset(months=1)
        hist_vals     = original_df[col][original_df.index >= one_month_ago]
        fc_vals      = forecast_df[col]

        ax.plot(hist_vals.index, hist_vals.values, color=hist_color, linewidth=1.5, label="Historical (1 month)", zorder=3)
        ax.axvline(last_date, color="white", linewidth=1.5, linestyle="--", alpha=0.8, label="Today", zorder=4)
        ax.axvspan(future_dates[0], future_dates[-1], alpha=0.10, color="#FF0000", zorder=1, label="Forecast window")
        ax.plot(fc_vals.index, fc_vals.values, color="#FF0000", linewidth=2.5, linestyle="--", marker="o", markersize=5, markerfacecolor="white", markeredgecolor="#FF0000", markeredgewidth=1.5, label="VAR Forecast", zorder=5)

        ax.axhline(0, color="white", linewidth=0.5, linestyle=":", alpha=0.4)
        ax.set_title(f"{col}  —  Yield Spread minus Expected Loss (bps)", fontsize=9, fontweight="bold")
        ax.set_ylabel("bps")
        ax.legend(fontsize=7, loc="upper left")
        ax.grid(True, alpha=0.25)

        all_d = list(hist_vals.index) + list(future_dates)
        ax.set_xlim(all_d[0], future_dates[-1])
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b %Y"))
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=7)

    fig.suptitle("VAR Forecast — Yield Spread minus Expected Loss (bps)\nSolid = Historical  |  Red dashed ● = Future Projection", fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("forecast_spread_minus_el.png", dpi=150, bbox_inches="tight")
    plt.show()

def plot_irf(forecast_results: dict):
    cols = forecast_results["cols"]
    irf = forecast_results["irf"]
    future_dates = forecast_results["future_dates"]
    n = len(cols)
    colors = ["#E63946", "#F4A261", "#2A9D8F", "#8338EC", "#457B9D", "#D4AF37", "#06D6A0"]

    try:
        today = future_dates[0] - pd.tseries.offsets.BDay(1)
        irf_dates = [today] + list(future_dates)
        if len(irf_dates) != irf.irfs.shape[0]: irf_dates = range(irf.irfs.shape[0])
    except Exception:
        irf_dates = range(irf.irfs.shape[0])

    configs = [
        {"data": irf.irfs, "json_file": "irf_contagion_matrix_1bps.json", "img_file": "irf_spread_minus_el_matrix_1bps.png", "title": "Impulse Response Function (IRF) for a 1 bps shock — Full Contagion Matrix"},
        {"data": irf.orth_irfs, "json_file": "irf_contagion_matrix_stddev.json", "img_file": "irf_spread_minus_el_matrix_stddev.png", "title": "Impulse Response Function (IRF) for a 1 Std-Dev shock — Full Contagion Matrix"}
    ]

    for config in configs:
        irf_data = config["data"]
        irf_export_dict = {}
        for shock_idx in range(n):
            shock_name = cols[shock_idx]
            irf_export_dict[f"Shock_to_{shock_name}"] = {}
            for resp_idx in range(n):
                resp_name = cols[resp_idx]
                response_array = irf_data[:, resp_idx, shock_idx]
                timeline_data = {}
                for i, val in enumerate(response_array):
                    date_str = irf_dates[i].strftime("%Y-%m-%d") if isinstance(irf_dates[i], pd.Timestamp) else f"Period_{i}"
                    timeline_data[date_str] = round(float(val), 4)
                irf_export_dict[f"Shock_to_{shock_name}"][f"Response_of_{resp_name}"] = timeline_data

        with open(config["json_file"], "w") as f: json.dump(irf_export_dict, f, indent=2)

        fig, axes = plt.subplots(n, n, figsize=(3.5 * n, 3.2 * n), sharex=False)
        if n == 1: axes = np.array([[axes]])

        for shock_idx in range(n):
            for resp_idx in range(n):
                ax = axes[resp_idx, shock_idx]
                response = irf_data[:, resp_idx, shock_idx]
                color = colors[resp_idx % len(colors)]

                ax.plot(irf_dates, response, color=color, linewidth=1.8)
                where_up = np.array([v > 0 for v in response])
                where_down = np.array([v < 0 for v in response])
                ax.fill_between(irf_dates, response, 0, where=where_up, alpha=0.18, color=color)
                ax.fill_between(irf_dates, response, 0, where=where_down, alpha=0.18, color="#AAAAAA")
                ax.axhline(0, color="white", linewidth=0.8, linestyle="--", alpha=0.6)

                if resp_idx == 0: ax.set_title(f"Shock to {cols[shock_idx]}", fontsize=11, fontweight="bold")
                if shock_idx == 0: ax.set_ylabel(f"Resp: {cols[resp_idx]}\n(bps change)", fontsize=10, fontweight="bold")

                if isinstance(irf_dates[0], pd.Timestamp):
                    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
                    step = max(1, len(irf_dates) // 4)
                    ax.xaxis.set_major_locator(mdates.DayLocator(interval=step))
                    ax.tick_params(axis='x', rotation=45, labelsize=8)
                else:
                    ax.set_xlabel("Periods after shock", fontsize=9)
                ax.grid(True, alpha=0.25)

        fig.suptitle(config["title"], fontsize=16, fontweight="bold", y=1.02)
        plt.tight_layout(h_pad=1.5, w_pad=1.0)
        plt.savefig(config["img_file"], dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(fig)

def determine_cointegration_rank(df: pd.DataFrame, k_ar_diff: int = 1) -> int:
    try:
        result = coint_johansen(df.values, det_order=0, k_ar_diff=k_ar_diff)
    except Exception as e:
        return 0

    K = df.shape[1]
    rank = 0
    for i in range(K):
        trace_stat, critical_val = result.lr1[i], result.cvt[i, 1]
        reject = trace_stat > critical_val
        if reject: rank += 1
        if not reject: break
    return rank

def run_johansen_var_pipeline(diff_over_time: dict, horizon: int = 12, max_lags: int = 12) -> dict:
    print("\n" + "=" * 65)
    print("  JOHANSEN-AUGMENTED VAR PIPELINE")
    print("=" * 65)

    original_df = build_dataframe(diff_over_time)

    if original_df.empty:
        raise ValueError("Data pipeline returned an empty DataFrame. Check FRED API.")

    print(f"\n  DataFrame: {original_df.shape[0]:,} rows x {original_df.shape[1]} columns")
    print(f"  Date range: {original_df.index[0].date()} -> {original_df.index[-1].date()}")
    print(f"  Pairs: {list(original_df.columns)}")

    if original_df.shape[1] < 2:
        return {}

    from statsmodels.tsa.stattools import kpss as kpss_test
    was_differenced, all_nonstationary = {}, True
    for col in original_df.columns:
        series = original_df[col].dropna()
        adf_p = adfuller(series, autolag="AIC")[1]
        try:
            kpss_p = kpss_test(series, regression='c', nlags='auto')[1]
        except Exception: kpss_p = 0.01

        is_stationary = (adf_p < 0.05) and (kpss_p > 0.05)
        was_differenced[col] = not is_stationary
        if is_stationary: all_nonstationary = False

    diff_df = original_df.diff().dropna()
    diff_df.index.freq = pd.tseries.frequencies.to_offset("B")
    lag_for_var = select_lag_order(diff_df, max_lags=max_lags)
    k_ar_diff = max(1, lag_for_var - 1)

    if all_nonstationary:
        rank = determine_cointegration_rank(original_df, k_ar_diff=k_ar_diff)
    else:
        rank = 0

    K = original_df.shape[1]

    if rank == 0:
        stationary_df = diff_df
        fitted_model = fit_var_model(stationary_df, lag_for_var)
        granger_results = run_granger_causality(stationary_df, lag_for_var)
        forecast_results = forecast_and_irf(fitted_model, stationary_df, original_df, was_differenced={col: True for col in original_df.columns}, horizon=horizon)

    elif rank < K:
        vecm_model = VECM(original_df, k_ar_diff=k_ar_diff, coint_rank=rank, deterministic='n')
        vecm_fit = vecm_model.fit()

        last_date = original_df.index[-1]
        future_dates = pd.bdate_range(start=last_date, periods=horizon + 1)[1:]
        raw_forecast = vecm_fit.predict(steps=horizon)
        forecast_df = pd.DataFrame(raw_forecast, index=future_dates, columns=original_df.columns)
        irf = vecm_fit.irf(periods=horizon)
        forecast_results = {"forecast_df": forecast_df, "irf": irf, "future_dates": future_dates, "cols": list(original_df.columns)}
        save_future_projections(forecast_df)
        plot_forecast(original_df, forecast_results)
        plot_irf(forecast_results)
        return {"rank": rank, "vecm_fit": vecm_fit, "forecast": forecast_df}

    else:
        fitted_model = fit_var_model(original_df, lag_for_var)
        granger_results = run_granger_causality(original_df, lag_for_var)
        forecast_results = forecast_and_irf(fitted_model, original_df, original_df, was_differenced={col: False for col in original_df.columns}, horizon=horizon)

    save_future_projections(forecast_results["forecast_df"])
    plot_forecast(original_df, forecast_results)
    plot_irf(forecast_results)

    return {"rank": rank, "lag_order": lag_for_var, "granger": granger_results if rank != 1 else {}, "forecast": forecast_results["forecast_df"], "irf": forecast_results["irf"]}

Writing forecast.py


In [ ]:
%%writefile default_rate_analysis.py
import certifi
import os
import json
import requests

FRED_API_KEY  = os.getenv("FRED_API_KEY", "ccb67ba570ac152dc7930a216488320d")
FRED_BASE_URL = "https://api.stlouisfed.org/fred/series/observations"
RATING_ORDER  = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]

FRED_OAS_SERIES = {
    "AAA": ["BAMLC0A1CAAA", "BAMLC0A0CM"],
    "AA" : ["BAMLC0A2CAA"],
    "A"  : ["BAMLC0A3CA"],
    "BBB": ["BAMLC0A4CBBB"],
    "BB" : ["BAMLH0A1HYM2", "BAMLH0A1BB", "BAMLH0A0HYM2"],
    "B"  : ["BAMLH0A2HYM2", "BAMLH0A2B",  "BAMLH0A0HYM2"],
    "CCC": ["BAMLH0A3HYM2", "BAMLH0A3CCC","BAMLH0A0HYM2"],
}
FRED_SEARCH_URL = "https://api.stlouisfed.org/fred/series/search"
FRED_DR_SERIES = {"IG_proxy" : "DRBLACBS", "HY_proxy" : "DRCCLACBS"}
PUBLISHED_LGD = {"AAA": 0.400, "AA" : 0.400, "A"  : 0.414, "BBB": 0.435, "BB" : 0.519, "B"  : 0.621, "CCC": 0.682}
DR_MAPPING = {"AAA": "IG_proxy", "AA" : "IG_proxy", "A"  : "IG_proxy", "BBB": "IG_proxy", "BB" : "HY_proxy", "B"  : "HY_proxy", "CCC": "HY_proxy"}
HY_SEARCH_QUERIES = {
    "BB" : "ICE BofA BB US High Yield Index Option-Adjusted Spread",
    "B"  : "ICE BofA Single-B US High Yield Index Option-Adjusted Spread",
    "CCC": "ICE BofA CCC US High Yield Index Option-Adjusted Spread",
}

def fred_fetch(series_id: str) -> list[dict]:
    resp = requests.get(FRED_BASE_URL, params={"series_id": series_id, "api_key": FRED_API_KEY, "file_type": "json", "sort_order": "asc"}, verify=certifi.where(), timeout=30)
    data = resp.json()
    if "error_message" in data: return []
    return [{"date": o["date"], "value": float(o["value"])} for o in data.get("observations", []) if o.get("value", ".") != "."]

def search_fred_series(query):
    try:
        resp = requests.get(FRED_SEARCH_URL, params={"search_text": query, "api_key": FRED_API_KEY, "file_type": "json", "limit": 20, "order_by": "popularity", "sort_order": "desc"}, verify=certifi.where(), timeout=30)
        for s in resp.json().get("seriess", []):
            sid, title = s.get("id", ""), s.get("title", "").lower()
            if "baml" in sid.lower() and "option-adjusted" in title: return sid
    except Exception: pass
    return None

def resolve_hy_series_ids(oas_series: dict) -> dict:
    for rating, query in HY_SEARCH_QUERIES.items():
        found = search_fred_series(query)
        if found and found not in oas_series[rating]: oas_series[rating].insert(0, found)
    return oas_series

def fetch_dr_proxies(dr_series: dict) -> dict:
    fred_dr_data = {}
    for proxy_name, series_id in dr_series.items():
        obs = fred_fetch(series_id)
        if obs:
            fred_dr_data[proxy_name] = obs
            print(f"  OK   {proxy_name} ({series_id}) -> {len(obs)} obs")
    return fred_dr_data

def build_raw_api_json(oas_series: dict, fred_dr_data: dict, dr_mapping: dict, lgd: dict) -> dict:
    raw_api_json = {}
    for rating in RATING_ORDER:
        obs_list, used_id = [], None
        for sid in oas_series[rating]:
            obs_list = fred_fetch(sid)
            if obs_list:
                used_id = sid
                break
        if not obs_list: continue
        proxy_key  = dr_mapping[rating]
        dr_obs     = fred_dr_data.get(proxy_key, [])
        latest_dr  = (dr_obs[-1]["value"] / 100) if dr_obs else 0.0
        raw_api_json[rating] = [{"date": o["date"], "default_rate": latest_dr, "lgd": lgd[rating]} for o in obs_list]
        print(f"  OK   {rating} ({used_id}) -> {len(obs_list)} entries DR={latest_dr*100:.4f}%  LGD={lgd[rating]*100:.1f}%")
    return raw_api_json

def filter_latest_entries(raw_api_json: dict) -> dict:
    latest = {}
    for rating, entries in raw_api_json.items():
        grouped = {}
        for each in entries: grouped.setdefault(each["date"], []).append(each)
        latest_date = max(grouped.keys())
        latest[rating] = grouped[latest_date]
    return latest

def compute_averages(latest_entries: dict) -> dict:
    averages = {}
    for rating, entries in latest_entries.items():
        n = len(entries)
        averages[rating] = {
            "avg_default_rate": round(sum(each["default_rate"] for each in entries) / n, 8),
            "avg_lgd"         : round(sum(each["lgd"]          for each in entries) / n, 6),
        }
    return averages

def compute_expected_loss(averages: dict) -> dict:
    return { rating: round(v["avg_default_rate"] * v["avg_lgd"], 8) for rating, v in averages.items() }

def compute_el_diff(expected_loss: dict, rating_order: list) -> dict:
    available = [r for r in rating_order if r in expected_loss]
    pairs     = list(zip(available, available[1:]))
    return {
        # THE FIX: Restored to match spread.py perfectly!
        f"{riskier} - {safer}": round(expected_loss[riskier] - expected_loss[safer], 8)
        for safer, riskier in pairs
    }

def merge_list_of_dicts(filepath: str, new_data: dict) -> dict:
    if os.path.exists(filepath):
        try:
            with open(filepath, "r") as f: old_data = json.load(f)
        except Exception: old_data = {}
    else: old_data = {}
    merged_data, all_keys = {}, set(old_data.keys()).union(set(new_data.keys()))
    for key in all_keys:
        date_map = {}
        if key in old_data:
            for entry in old_data[key]: date_map[entry["date"]] = entry
        if key in new_data:
            for entry in new_data[key]: date_map[entry["date"]] = entry
        merged_data[key] = [date_map[d] for d in sorted(date_map.keys())]
    return merged_data

def save_default_rate_jsons(raw_api_json, latest_entries, averages, expected_loss, el_diff):
    merged_raw_api = merge_list_of_dicts("default_rate_and_lgd_by_grade.json", raw_api_json)
    files = [
        ("default_rate_and_lgd_by_grade.json", merged_raw_api),
        ("latest_by_rating.json", latest_entries),
        ("averages_by_grade.json", averages),
        ("expected_loss_by_grade.json", expected_loss),
        ("el_diff_adjacent_grades.json", el_diff),
    ]
    for fname, data in files:
        with open(fname, "w") as f: json.dump(data, f, indent=2)

def print_el_tables(averages: dict, expected_loss: dict, el_diff: dict):
    print(f"\n{'=' * 65}\nAverage DR, LGD and Expected Loss by grade\n{'=' * 65}")
    print(f"  {'Rating':<6}  {'DR':>10}  {'LGD':>8}  {'EL':>12}\n  {'─' * 6}  {'─' * 10}  {'─' * 8}  {'─' * 12}")
    for rating in RATING_ORDER:
        if rating in averages:
            dr, lgd, el = averages[rating]["avg_default_rate"], averages[rating]["avg_lgd"], expected_loss[rating]
            print(f"  {rating:<6}  {dr * 100:>9.4f}%  {lgd * 100:>7.1f}%  {el * 100:>11.6f}%")

    print(f"\n{'=' * 65}\nEL difference between adjacent grades\n{'=' * 65}")
    print(f"  {'Pair':<14}  {'Diff (%)':>12}  {'Diff (bps)':>12}\n  {'─' * 14}  {'─' * 12}  {'─' * 12}")
    for pair, diff in el_diff.items():
        print(f"  {pair:<14}  {diff * 100:>11.6f}%  {diff * 100 * 100:>11.4f}")

def main():
    updated_oas_series = resolve_hy_series_ids(FRED_OAS_SERIES)
    fred_dr_data = fetch_dr_proxies(FRED_DR_SERIES)
    raw_api_json = build_raw_api_json(updated_oas_series, fred_dr_data, DR_MAPPING, PUBLISHED_LGD)
    latest_entries = filter_latest_entries(raw_api_json)
    averages = compute_averages(latest_entries)
    expected_loss = compute_expected_loss(averages)
    el_diff = compute_el_diff(expected_loss, RATING_ORDER)
    save_default_rate_jsons(raw_api_json, latest_entries, averages, expected_loss, el_diff)
    print_el_tables(averages, expected_loss, el_diff)

if __name__ == "__main__":
    main()

Writing default_rate_analysis.py


In [ ]:
%%writefile dealer_markup.py
import pandas as pd
import numpy as np
import yfinance as yf
import json
import os


def load_timeseries_data(filepath: str) -> pd.DataFrame:
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Could not find {filepath}")

    with open(filepath, "r") as f:
        data = json.load(f)

    df = pd.DataFrame(data)
    df.index = pd.to_datetime(df.index)
    df = df.sort_index().resample("B").last().ffill().dropna()
    return df


def calculate_realized_volatility(series: pd.Series, lookback_window: int = 21, trading_days: int = 252) -> pd.Series:
    """Calculates Normal (absolute basis point) Realized Volatility."""

    # TNX is quoted in percent (e.g., 4.50). Convert it directly to basis points (450).
    series_bps = series * 100

    # Calculate daily absolute changes in basis points (NOT log returns)
    returns_bps = series_bps.diff().dropna()

    # Calculate daily standard deviation and annualize it
    daily_volatility = returns_bps.rolling(window=lookback_window).std()
    annualized_rv = daily_volatility * np.sqrt(trading_days)

    return annualized_rv.dropna()


def fetch_macro_volatility_proxy(start_date: str, end_date: str) -> pd.DataFrame:
    print("  [API] Fetching ICE BofA MOVE Index (IV) and 10-Year Treasury Yield (RV)...")
    tickers = ["^MOVE", "^TNX"]
    data = yf.download(tickers, start=start_date, end=end_date, progress=False)["Close"]

    if data.empty:
        raise ValueError("Failed to fetch macro volatility data from Yahoo Finance.")
    return data.ffill().dropna()


def calculate_dealer_markup(df_macro: pd.DataFrame, lookback: int = 21) -> pd.Series:
    print("  [Math] Calculating dynamic Volatility Risk Premium (Dealer Markup)...")

    # 1. MOVE is already quoted in annualized basis points (e.g., 120 means 120 bps)
    implied_volatility = df_macro["^MOVE"]

    # 2. Calculate RV in annualized basis points
    realized_volatility = calculate_realized_volatility(df_macro["^TNX"], lookback_window=lookback)

    # 3. Align timelines
    aligned_df = pd.concat([implied_volatility, realized_volatility], axis=1).dropna()
    aligned_df.columns = ["IV", "RV"]

    # 4. Calculate Multiplier (Apples to Apples: Bps IV / Bps RV)
    safe_rv = np.maximum(aligned_df["RV"], 0.0001)
    raw_markup = aligned_df["IV"] / safe_rv

    # 5. Apply 5% Dealer Floor and smooth
    dynamic_markup = np.maximum(raw_markup, 1.05)
    smoothed_markup = dynamic_markup.rolling(window=5).mean().bfill()

    return smoothed_markup


def run():
    bond_filepath = "USA_bond_returns_by_grade.json"
    markup_filepath = "dealer_markup.json"

    try:
        df_bonds = load_timeseries_data(bond_filepath)
        start_date = df_bonds.index.min().strftime('%Y-%m-%d')
        end_date = df_bonds.index.max().strftime('%Y-%m-%d')

        df_macro = fetch_macro_volatility_proxy(start_date, end_date)
        daily_markup = calculate_dealer_markup(df_macro, lookback=21)
        final_markup = daily_markup.reindex(df_bonds.index).ffill().bfill()

        print("\n" + "=" * 50)
        print(" LATEST DYNAMIC DEALER MARKUPS")
        print("=" * 50)
        print(f"  {'Date':<15} | {'Dealer Multiplier':<15}")
        print("-" * 50)

        for date, markup in final_markup.tail(5).items():
            print(f"  {date.strftime('%Y-%m-%d'):<15} | {markup:>6.2f}x")

        print("-" * 50)
        print(f"  Historical Average Markup: {final_markup.mean():.2f}x")

        final_markup.index = final_markup.index.strftime('%Y-%m-%d')
        with open(markup_filepath, "w") as f:
            json.dump({"Dealer_Multiplier": final_markup.to_dict()}, f, indent=2)


    except Exception as e:
        print(f"[ERROR] {e}")

def main():
    run()



if __name__ == "__main__":
    main()

Writing dealer_markup.py


In [ ]:
%%writefile data_engine.py
import os
import logging
import requests
import numpy as np
import pandas as pd
import yfinance as yf
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s] - %(message)s')
logger = logging.getLogger(__name__)

FRED_API_KEY = os.getenv("FRED_API_KEY", "ccb67ba570ac152dc7930a216488320d")

def get_secure_session():
    """Enterprise API configuration: Auto-retries on 429 (Rate Limit) and 500s."""
    session = requests.Session()
    retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

def build_features(df):
    """Engineers Institutional Ratios with Safe Rolling Windows."""
    df = df.copy()
    df['Ret'] = df['Adj Close'].pct_change()
    df['Liquidity_Proxy'] = (df['High'] - df['Low']) / df['Close']

    windows = [21, 63, 252]
    for w in windows:
        # min_periods=1 prevents pandas from filling the top of the dataset with NaNs
        roll_ret = df['Ret'].rolling(w, min_periods=1)
        std_dev = roll_ret.std().fillna(0)
        df[f'Sharpe_{w}'] = (roll_ret.mean() / (std_dev + 1e-9)) * np.sqrt(252)

        downside = df['Ret'].copy()
        downside[downside > 0] = 0
        down_std = downside.rolling(w, min_periods=1).std().fillna(0)
        df[f'Sortino_{w}'] = (roll_ret.mean() / (down_std + 1e-9)) * np.sqrt(252)

        roll_max = df['Adj Close'].rolling(w, min_periods=1).max()
        max_dd = ((df['Adj Close'] / roll_max) - 1.0).rolling(w, min_periods=1).min()

        # Safely handle the shift calculation
        shifted_close = df['Adj Close'].shift(w).bfill()
        annual_ret = (df['Adj Close'] / shifted_close) ** (252 / w) - 1
        df[f'Calmar_{w}'] = annual_ret / (abs(max_dd) + 1e-9)

    return df

def generate_master_dataset():
    """Compiles Market Proxies & Macro Anchors into a secure, ML-ready panel."""
    logger.info("Fetching ETFs and Macro Anchors...")
    session = get_secure_session()

    def secure_fred(series_id):
        url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={FRED_API_KEY}&file_type=json"
        try:
            resp = session.get(url, timeout=10)
            resp.raise_for_status()
            data = resp.json()
            if "error_message" in data:
                return pd.Series(dtype=float)

            df = pd.DataFrame(data['observations'])[['date', 'value']]
            df['date'] = pd.to_datetime(df['date'])
            df['value'] = pd.to_numeric(df['value'], errors='coerce')
            return df.set_index('date')['value'].ffill().dropna()
        except Exception as e:
            logger.error(f"FRED Fetch Error ({series_id}): {e}")
            return pd.Series(dtype=float)

    term_spread = (secure_fred("DGS10") - secure_fred("DGS2")).ffill()
    oas = secure_fred("BAMLC0A0CM")

    dataset = {}
    tickers = ['SHY', 'TLT', 'LQD', 'HYG', 'ANGL', 'PFF']

    for t in tickers:
        logger.info(f"Processing {t}...")
        df = yf.download(t, start="2015-01-01", progress=False)
        if df.empty:
            continue

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if 'Adj Close' not in df.columns:
            df['Adj Close'] = df['Close']

        # Strip timezones so yfinance dates match FRED dates
        if df.index.tz is not None:
            df.index = df.index.tz_localize(None)

        df = build_features(df)
        df = df.iloc[252:].copy()

        # --- THE FIX: Eradicate OAS_Z NaNs ---
        # 1. Reindex, backfill, forward-fill, and fillna(0) to guarantee 0 missing dates
        df['Term_Spread'] = term_spread.reindex(df.index).bfill().ffill().fillna(0)
        oas_reindexed = oas.reindex(df.index).bfill().ffill().fillna(0)

        # 2. Use min_periods=1 to stop pandas from injecting 62 NaNs into the math
        roll_mean = oas_reindexed.rolling(63, min_periods=1).mean()
        roll_std = oas_reindexed.rolling(63, min_periods=1).std().fillna(0)

        df['OAS_Z'] = (oas_reindexed - roll_mean) / (roll_std + 1e-9)

        # Build Target
        future_close = df['Adj Close'].shift(-21)
        df['Target'] = (future_close > df['Adj Close']).astype(float)
        df.loc[future_close.isna(), 'Target'] = np.nan

        feature_cols = [c for c in df.columns if c not in ['Target']]

        # Gate Check
        nan_pct = df[feature_cols].isnull().mean()
        critical_nan = nan_pct[nan_pct > 0.05]
        if not critical_nan.empty:
            raise RuntimeError(
                f"Feature NaN threshold exceeded for {t}. "
                f"Columns above 5% NaN: {critical_nan.to_dict()}"
            )

        df = df.dropna(subset=feature_cols)
        dataset[t] = df

    logger.info("Master dataset generation complete.")
    return dataset

Writing data_engine.py


In [ ]:
%%writefile bonds_EUR_data.py
import pandas as pd
import numpy as np
import yfinance as yf
import spread as us_spread


def fetch_etf_yield(ticker: str) -> pd.Series:
    """
    Calculates dynamic Trailing 12-Month (TTM) yield for an ETF.
    """
    print(f"  [API] Fetching proxy {ticker} via yfinance...")
    try:
        t = yf.Ticker(ticker)
        hist_data = t.history(start="2016-01-01")
        if hist_data.empty:
            print(f"  [FAIL] No data found for {ticker}")
            return pd.Series(dtype=float)

        # Calculate TTM Yield: (252-day rolling sum of dividends / daily close price)
        ttm_dividends = hist_data["Dividends"].rolling(window=252).sum().bfill()

        # --- THE FIX: We imported numpy so np.nan safely prevents zero-division ---
        safe_close = hist_data["Close"].replace(0, np.nan)
        yield_proxy = (ttm_dividends / safe_close) * 100

        # Ensure the index is a timezone-naive DatetimeIndex to prevent merge errors
        if yield_proxy.index.tz is not None:
            yield_proxy.index = yield_proxy.index.tz_localize(None)

        return yield_proxy.dropna()
    except Exception as e:
        print(f"  [ERROR] {ticker}: {e}")
        return pd.Series(dtype=float)


def get_eur_market_data() -> pd.DataFrame:
    """
    Fetches European Proxies (yfinance) and Risk-Free Rates (FRED).
    """
    print("  [API] Fetching European Proxies (ETFs for OAS, FRED for Risk-Free)...")
    data = {}

    # 1. Fetch Euro IG Proxy (iShares Core € Corp Bond UCITS ETF)
    data["EUR_IG_OAS"] = fetch_etf_yield("IEAC.L")

    # 2. Fetch Euro HY Proxy (iShares € High Yield Corp Bond UCITS ETF)
    data["EUR_HY_OAS"] = fetch_etf_yield("IHYG.L")

    # 3. Fetch Risk-Free Rate via FRED
    print("  [API] Fetching DGS10 for Risk-Free Rate...")
    try:
        raw_fred = us_spread.fetch_fred_series("DGS10")
        s = pd.Series(raw_fred)

        # Explicitly convert the text strings into a DatetimeIndex
        s.index = pd.to_datetime(s.index)

        data["EUR_Risk_Free_10Y"] = s
        print(f"  OK   EUR_Risk_Free_10Y (DGS10) -> {len(s)} obs")
    except Exception as e:
        print(f"  [ERROR] DGS10: {e}")

    # Assemble DataFrame
    df = pd.DataFrame(data).ffill().dropna()

    if df.empty:
        print("  [CRITICAL] European dataframe is empty. Data fetch failed.")
        return df

    # Standardize to Business Days
    df = df.resample("B").last().ffill()

    # --- THE FIX: Unconditionally convert to Basis Points so the engine prices it correctly ---
    df = df * 100.0

    print(f"  [SUCCESS] Assembled European Data: {len(df)} overlapping observations.")

    return df

Writing bonds_EUR_data.py


In [ ]:
%%writefile bonds_EM_data.py
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import textwrap
import spread as us_spread

# -----------------------------------------------------------------------------
# EMERGING MARKETS ASSET PROFILES
# -----------------------------------------------------------------------------
EM_PROFILES = {
    "EM_USD_Sovereign": {
        "Ticker": "EMB", "Rating": "BB+ to BBB-", "Currency": "USD (No FX Risk)",
        "Vol_Regime": "Moderate", "Countries": "Mexico, Saudi Arabia, Turkey, Indonesia, UAE, Brazil.",
        "Profile": "Emerging market government debt issued in US Dollars. Highly sensitive to US Federal Reserve interest rate hikes (Original Sin). Considered the safest EM tier."
    },
    "EM_Corporate": {
        "Ticker": "CEMB", "Rating": "BB to BB+", "Currency": "USD (No FX Risk)",
        "Vol_Regime": "Moderate-High", "Countries": "Colombia, Brazil, Israel, Mexico, UAE, Chile, Macao.",
        "Profile": "EM Corporate debt in USD. Driven by both global macroeconomic sentiment and local corporate default cycles. Tends to track US High Yield closely."
    },
    "EM_High_Yield": {
        "Ticker": "EMHY", "Rating": "B to CCC", "Currency": "USD (No FX Risk)",
        "Vol_Regime": "High (Kappa = 40)", "Countries": "Turkey, Brazil, Colombia, Mexico, Argentina, South Africa.",
        "Profile": "The 'Junk' tier of EM debt. Extremely sensitive to global liquidity vacuums and commodity price crashes. Experiences violent sell-offs and aggressive snap-backs."
    },
    "EM_Local_Currency": {
        "Ticker": "LEMB", "Rating": "BB to B", "Currency": "Local (Extreme FX Risk)",
        "Vol_Regime": "Extreme (Kappa = 40)", "Countries": "Brazil, Mexico, Indonesia, South Africa, Malaysia, Poland.",
        "Profile": "Government debt priced in native currencies (e.g., Lira, Real). Introduces massive Foreign Exchange (FX) volatility. Returns can be entirely wiped out by currency devaluation."
    },
    "Risk_Free": {
        "Ticker": "DGS10", "Rating": "AAA", "Currency": "USD",
        "Vol_Regime": "Baseline", "Countries": "United States of America.",
        "Profile": "US 10-Year Treasury Yield. The ultimate global safe-haven and the core benchmark against which all Emerging Market risk premiums are measured."
    }
}


def fetch_etf_yield(ticker: str) -> pd.Series:
    """Calculates dynamic Trailing 12-Month (TTM) yield for an Emerging Market ETF."""
    print(f"  [API] Fetching EM proxy {ticker} via yfinance...")
    try:
        t = yf.Ticker(ticker)
        hist_data = t.history(start="2016-01-01")
        if hist_data.empty:
            print(f"  [FAIL] No data found for {ticker}")
            return pd.Series(dtype=float)

        ttm_dividends = hist_data["Dividends"].rolling(window=252).sum().bfill()

        # --- THE FIX: Safe division ---
        safe_close = hist_data["Close"].replace(0, np.nan)
        yield_proxy = (ttm_dividends / safe_close) * 100

        if yield_proxy.index.tz is not None:
            yield_proxy.index = yield_proxy.index.tz_localize(None)

        return yield_proxy.dropna()
    except Exception as e:
        print(f"  [ERROR] {ticker}: {e}")
        return pd.Series(dtype=float)


def generate_em_profile_table(latest_yields: pd.Series):
    """Generates a high-quality Matplotlib table detailing the EM Asset characteristics."""
    print("\n  [SYSTEM] Generating Emerging Markets Profile Table...")

    table_data = []
    columns = ["Asset Class", "Benchmark\nETF", "Implied\nRating", "Currency\nExposure",
               "Volatility\nRegime", "Latest Dynamic\nYield", "Major Country\nExposures",
               "Macro Sensitivities & Asset Profile"]

    for asset in EM_PROFILES.keys():
        if asset in latest_yields:
            prof = EM_PROFILES[asset]
            wrapped_profile = "\n".join(textwrap.wrap(prof["Profile"], width=55))
            wrapped_countries = "\n".join(textwrap.wrap(prof["Countries"], width=25))

            # --- THE FIX: The yields array is in BPS, so we divide by 100 to show % on the table ---
            current_yield = latest_yields[asset] / 100.0

            table_data.append([
                asset.replace("_", " "),
                prof["Ticker"],
                prof["Rating"],
                prof["Currency"],
                prof["Vol_Regime"],
                f"{current_yield:.2f}%",
                wrapped_countries,
                wrapped_profile
            ])

    fig, ax = plt.subplots(figsize=(28, 9))
    ax.axis('tight')
    ax.axis('off')

    header_text = (
        "EMERGING MARKETS (EM) DEBT: ASSET PROFILE & RISK MATRIX\n"
        "Model Dynamics: EMHY and Local Currency debt trigger a 'Kappa = 40.0' mean-reversion override due to violent snap-back characteristics.\n"
        "Yields shown are real-time dynamic TTM proxies (Dividends/Price) for ETFs, and raw rate for US 10-Year Treasury."
    )
    plt.title(header_text, fontweight="bold", fontsize=15, loc="left", pad=20, color="#1D3557")

    table = ax.table(cellText=table_data, colLabels=columns, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(12)

    col_widths = {0: 0.10, 1: 0.05, 2: 0.06, 3: 0.10, 4: 0.10, 5: 0.08, 6: 0.16, 7: 0.35}

    for (row, col), cell in table.get_celld().items():
        cell.set_width(col_widths[col])
        if row == 0:
            cell.set_facecolor('#1D3557')
            cell.set_text_props(weight='bold', color='white')
        else:
            if col == 5:
                cell.set_facecolor('#F8F9FA')
                cell.set_text_props(weight='bold')
            elif col == 6 or col == 7:
                cell.set_text_props(ha='left')

    table.scale(1.0, 4.5)
    plt.tight_layout()
    plt.savefig("emerging_markets_profile_table.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print("  [SUCCESS] Rendered emerging_markets_profile_table.png")


def get_em_market_data() -> pd.DataFrame:
    """Fetches Emerging Market Proxies (yfinance) and USD Risk-Free Rates (FRED)."""
    print("  [API] Fetching Emerging Market Proxies (ETFs for OAS, FRED for Risk-Free)...")
    data = {}

    data["EM_USD_Sovereign"] = fetch_etf_yield("EMB")
    data["EM_Corporate"] = fetch_etf_yield("CEMB")
    data["EM_High_Yield"] = fetch_etf_yield("EMHY")
    data["EM_Local_Currency"] = fetch_etf_yield("LEMB")

    print("  [API] Fetching DGS10 for USD Risk-Free Rate...")
    try:
        raw_fred = us_spread.fetch_fred_series("DGS10")
        s = pd.Series(raw_fred)
        s.index = pd.to_datetime(s.index)
        data["Risk_Free"] = s
        print(f"  OK   Risk_Free (DGS10) -> {len(s)} obs")
    except Exception as e:
        print(f"  [ERROR] DGS10: {e}")

    df = pd.DataFrame(data).ffill().dropna()

    if df.empty:
        print("  [CRITICAL] EM dataframe is empty. Data fetch failed.")
        return df

    df = df.resample("B").last().ffill()

    # --- THE FIX: Unconditionally convert to Basis Points ---
    df = df * 100.0

    print(f"  [SUCCESS] Assembled EM Data: {len(df)} overlapping observations.")

    latest_yields = df.iloc[-1]
    generate_em_profile_table(latest_yields)

    return df

Writing bonds_EM_data.py


In [ ]:
%%writefile analysis.py
import json
import matplotlib.pyplot as plt
import forecast

def load_inputs(spread_path: str, el_path: str) -> tuple[dict, dict]:
    with open(spread_path) as f:
        yield_spreads = json.load(f)
    with open(el_path) as f:
        el_diffs = json.load(f)
    return yield_spreads, el_diffs

def compute_diff_over_time(yield_spreads: dict, el_diffs: dict) -> dict:
    diff_over_time = {}
    for pair in yield_spreads:
        if pair not in el_diffs:
            continue
        el_bps = el_diffs[pair] * 100 * 100
        diff_over_time[pair] = {
            date: round(ys_bps - el_bps, 4)
            for date, ys_bps in yield_spreads[pair].items()
        }
    return diff_over_time

def save_diff_json(diff_over_time: dict, path: str = "spread-el_grades.json"):
    with open(path, "w") as f:
        json.dump(diff_over_time, f, indent=2)

def print_diff_table(diff_over_time: dict, yield_spreads: dict, el_diffs: dict):
    print(f"\n{'Pair':<18} {'Date':<13} {'Yield Spread':>14} "
          f"{'EL (bps)':>10} {'Difference':>12}")
    print(f"{'-'*18} {'-'*13} {'-'*14} {'-'*10} {'-'*12}")
    for pair, series in diff_over_time.items():
        date     = list(series.keys())[-1]
        diff_val = series[date]
        ys_val   = list(yield_spreads[pair].values())[-1]
        el_bps   = el_diffs[pair] * 100 * 100
        print(f"{pair:<18} {date:<13} {ys_val:>14.2f} {el_bps:>10.2f} {diff_val:>12.2f}")

def plot_diff(diff_over_time: dict):
    # THE FIX: Added Fallen Angel to visual hierarchy
    hierarchy = ["AA - AAA", "A - AA", "BBB - A", "BB - Fallen_Angel", "BB - BBB", "B - BB", "CCC - B"]
    pairs = [pair for pair in hierarchy if pair in diff_over_time]

    n = len(pairs)
    if n == 0:
        print("No data available to plot.")
        return

    colors = ["#E63946", "#F4A261", "#2A9D8F", "#8338EC", "#457B9D", "#D4AF37", "#06D6A0"]
    fig, axes = plt.subplots(n, 1, figsize=(14, 3 * n), sharex=False)
    if n == 1:
        axes = [axes]

    for ax, pair, color in zip(axes, pairs, colors):
        dates = list(diff_over_time[pair].keys())
        values = list(diff_over_time[pair].values())
        ax.plot(dates, values, color=color, linewidth=1.4, label=pair)
        ax.axhline(0, color="white", linewidth=0.7, linestyle="--", alpha=0.5)
        ax.fill_between(dates, values, 0, where=[v > 0 for v in values], alpha=0.15, color=color, label="Excess yield")
        ax.set_title(f"{pair}  —  Yield Spread minus Expected Loss (bps)", fontsize=10, fontweight="bold")
        ax.set_ylabel("bps")
        step = max(1, len(dates) // 12)
        ax.set_xticks(range(0, len(dates), step))
        ax.set_xticklabels([dates[i] for i in range(0, len(dates), step)], rotation=35, ha="right", fontsize=7)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle("Yield Spread minus Expected Loss — USA Adjacent Bond Grades", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("spread_minus_el.png", dpi=150, bbox_inches="tight")
    plt.show()

def main():
    spread_path = "USA_yield_spread_by_bond_grade.json"
    el_path = "el_diff_adjacent_grades.json"
    yield_spreads, el_diffs = load_inputs(spread_path, el_path)
    diff_over_time = compute_diff_over_time(yield_spreads, el_diffs)
    save_diff_json(diff_over_time)
    print_diff_table(diff_over_time, yield_spreads, el_diffs)
    plot_diff(diff_over_time)

    print("\n  [FORECAST] Launching Johansen-Augmented VAR Pipeline...")
    try:
        forecast_results = forecast.run_johansen_var_pipeline(diff_over_time=diff_over_time, horizon=12, max_lags=12)
        print(f"  [FORECAST] Pipeline complete. Cointegration rank: {forecast_results.get('rank', 'N/A')}")
    except Exception as e:
        print(f"  [FORECAST ERROR] VAR pipeline failed: {e}")
        raise

if __name__ == "__main__":
    main()

Writing analysis.py


In [ ]:

%%writefile dealer_markup.json
{
  "Dealer_Multiplier": {
    "2012-04-13": 1.4189517995896839,
    "2012-04-16": 1.4189517995896839,
    "2012-04-17": 1.4189517995896839,
    "2012-04-18": 1.4189517995896839,
    "2012-04-19": 1.4189517995896839,
    "2012-04-20": 1.4189517995896839,
    "2012-04-23": 1.4189517995896839,
    "2012-04-24": 1.4189517995896839,
    "2012-04-25": 1.4189517995896839,
    "2012-04-26": 1.4189517995896839,
    "2012-04-27": 1.4189517995896839,
    "2012-04-30": 1.4189517995896839,
    "2012-05-01": 1.4189517995896839,
    "2012-05-02": 1.4189517995896839,
    "2012-05-03": 1.4189517995896839,
    "2012-05-04": 1.4189517995896839,
    "2012-05-07": 1.4189517995896839,
    "2012-05-08": 1.4189517995896839,
    "2012-05-09": 1.4189517995896839,
    "2012-05-10": 1.4189517995896839,
    "2012-05-11": 1.4189517995896839,
    "2012-05-14": 1.4189517995896839,
    "2012-05-15": 1.4189517995896839,
    "2012-05-16": 1.4189517995896839,
    "2012-05-17": 1.4189517995896839,
    "2012-05-18": 1.4189517995896839,
    "2012-05-21": 1.4069052868190401,
    "2012-05-22": 1.3884158984683705,
    "2012-05-23": 1.3540432461281475,
    "2012-05-24": 1.3152774719028013,
    "2012-05-25": 1.269918517727371,
    "2012-05-28": 1.269918517727371,
    "2012-05-29": 1.2315990958110838,
    "2012-05-30": 1.1904752254275632,
    "2012-05-31": 1.1639996052539368,
    "2012-06-01": 1.137716089926829,
    "2012-06-04": 1.1181495420216307,
    "2012-06-05": 1.0974038117435667,
    "2012-06-06": 1.0917817243735592,
    "2012-06-07": 1.0742749452751856,
    "2012-06-08": 1.0643223548027412,
    "2012-06-11": 1.0531952010539443,
    "2012-06-12": 1.05,
    "2012-06-13": 1.05,
    "2012-06-14": 1.05,
    "2012-06-15": 1.0532752929206175,
    "2012-06-18": 1.0532752929206175,
    "2012-06-19": 1.0532752929206177,
    "2012-06-20": 1.0532752929206177,
    "2012-06-21": 1.0532752929206175,
    "2012-06-22": 1.05,
    "2012-06-25": 1.05,
    "2012-06-26": 1.05,
    "2012-06-27": 1.05,
    "2012-06-28": 1.05,
    "2012-06-29": 1.05,
    "2012-07-02": 1.05,
    "2012-07-03": 1.05,
    "2012-07-04": 1.05,
    "2012-07-05": 1.05,
    "2012-07-06": 1.05,
    "2012-07-09": 1.05,
    "2012-07-10": 1.05,
    "2012-07-11": 1.05,
    "2012-07-12": 1.05,
    "2012-07-13": 1.05,
    "2012-07-16": 1.05,
    "2012-07-17": 1.05,
    "2012-07-18": 1.05,
    "2012-07-19": 1.05,
    "2012-07-20": 1.05,
    "2012-07-23": 1.05,
    "2012-07-24": 1.05,
    "2012-07-25": 1.05,
    "2012-07-26": 1.05,
    "2012-07-27": 1.05,
    "2012-07-30": 1.05,
    "2012-07-31": 1.05,
    "2012-08-01": 1.0514424400079445,
    "2012-08-02": 1.0514424400079445,
    "2012-08-03": 1.0514424400079443,
    "2012-08-06": 1.0514424400079445,
    "2012-08-07": 1.0514424400079445,
    "2012-08-08": 1.05,
    "2012-08-09": 1.05,
    "2012-08-10": 1.05,
    "2012-08-13": 1.05,
    "2012-08-14": 1.05,
    "2012-08-15": 1.05,
    "2012-08-16": 1.05,
    "2012-08-17": 1.05,
    "2012-08-20": 1.05,
    "2012-08-21": 1.05,
    "2012-08-22": 1.05,
    "2012-08-23": 1.05,
    "2012-08-24": 1.05,
    "2012-08-27": 1.05,
    "2012-08-28": 1.05,
    "2012-08-29": 1.05,
    "2012-08-30": 1.05,
    "2012-08-31": 1.05,
    "2012-09-03": 1.05,
    "2012-09-04": 1.05,
    "2012-09-05": 1.05,
    "2012-09-06": 1.05,
    "2012-09-07": 1.05,
    "2012-09-10": 1.05,
    "2012-09-11": 1.05,
    "2012-09-12": 1.05,
    "2012-09-13": 1.05,
    "2012-09-14": 1.05,
    "2012-09-17": 1.05,
    "2012-09-18": 1.05,
    "2012-09-19": 1.05,
    "2012-09-20": 1.05,
    "2012-09-21": 1.05,
    "2012-09-24": 1.05,
    "2012-09-25": 1.05,
    "2012-09-26": 1.05,
    "2012-09-27": 1.05,
    "2012-09-28": 1.05,
    "2012-10-01": 1.05,
    "2012-10-02": 1.05,
    "2012-10-03": 1.05,
    "2012-10-04": 1.05,
    "2012-10-05": 1.05,
    "2012-10-08": 1.05,
    "2012-10-09": 1.05,
    "2012-10-10": 1.05,
    "2012-10-11": 1.05,
    "2012-10-12": 1.05,
    "2012-10-15": 1.1106850916845945,
    "2012-10-16": 1.1538473335181758,
    "2012-10-17": 1.1708869733081921,
    "2012-10-18": 1.1919881204609162,
    "2012-10-19": 1.1986686433576597,
    "2012-10-22": 1.1469002842457372,
    "2012-10-23": 1.1120071111843037,
    "2012-10-24": 1.1027104897793336,
    "2012-10-25": 1.121472030339979,
    "2012-10-26": 1.126890811195696,
    "2012-10-29": 1.126890811195696,
    "2012-10-30": 1.126890811195696,
    "2012-10-31": 1.1179740786230241,
    "2012-11-01": 1.109705009850876,
    "2012-11-02": 1.1019619914658298,
    "2012-11-05": 1.0620993037524604,
    "2012-11-06": 1.05,
    "2012-11-07": 1.05,
    "2012-11-08": 1.05,
    "2012-11-09": 1.05,
    "2012-11-12": 1.05,
    "2012-11-13": 1.05,
    "2012-11-14": 1.05,
    "2012-11-15": 1.05,
    "2012-11-16": 1.05,
    "2012-11-19": 1.05,
    "2012-11-20": 1.05,
    "2012-11-21": 1.05,
    "2012-11-22": 1.05,
    "2012-11-23": 1.05,
    "2012-11-26": 1.05,
    "2012-11-27": 1.05,
    "2012-11-28": 1.05,
    "2012-11-29": 1.05,
    "2012-11-30": 1.05,
    "2012-12-03": 1.05,
    "2012-12-04": 1.05,
    "2012-12-05": 1.05,
    "2012-12-06": 1.0534507920276093,
    "2012-12-07": 1.12921801700243,
    "2012-12-10": 1.1979709107910828,
    "2012-12-11": 1.2552469545701068,
    "2012-12-12": 1.3019393893435791,
    "2012-12-13": 1.3460276340227595,
    "2012-12-14": 1.308200949229065,
    "2012-12-17": 1.2787453841362124,
    "2012-12-18": 1.2695387596597052,
    "2012-12-19": 1.2453106885347243,
    "2012-12-20": 1.21425956538141,
    "2012-12-21": 1.1853117394172072,
    "2012-12-24": 1.152012121207758,
    "2012-12-25": 1.152012121207758,
    "2012-12-26": 1.112085698292968,
    "2012-12-27": 1.0910102591900581,
    "2012-12-28": 1.0874570359565836,
    "2012-12-31": 1.093639671687756,
    "2013-01-01": 1.093639671687756,
    "2013-01-02": 1.0876419612014054,
    "2013-01-03": 1.0794989648136784,
    "2013-01-04": 1.0781100402680965,
    "2013-01-07": 1.0651753499480958,
    "2013-01-08": 1.05,
    "2013-01-09": 1.05,
    "2013-01-10": 1.05,
    "2013-01-11": 1.05,
    "2013-01-14": 1.05,
    "2013-01-15": 1.05,
    "2013-01-16": 1.05,
    "2013-01-17": 1.05,
    "2013-01-18": 1.05,
    "2013-01-21": 1.05,
    "2013-01-22": 1.0573477925693178,
    "2013-01-23": 1.057347792569318,
    "2013-01-24": 1.057347792569318,
    "2013-01-25": 1.0573477925693178,
    "2013-01-28": 1.0573477925693178,
    "2013-01-29": 1.05,
    "2013-01-30": 1.0596881306217945,
    "2013-01-31": 1.0664945896267113,
    "2013-02-01": 1.0933686208816684,
    "2013-02-04": 1.1187958676101324,
    "2013-02-05": 1.1436251703497242,
    "2013-02-06": 1.1408163737320927,
    "2013-02-07": 1.1342117108320298,
    "2013-02-08": 1.1154622488303825,
    "2013-02-11": 1.0986941046662675,
    "2013-02-12": 1.0794688701616812,
    "2013-02-13": 1.0835092485707494,
    "2013-02-14": 1.086758861453358,
    "2013-02-15": 1.083122290884014,
    "2013-02-18": 1.083122290884014,
    "2013-02-19": 1.0844580172158722,
    "2013-02-20": 1.094246209519648,
    "2013-02-21": 1.09288044741586,
    "2013-02-22": 1.0941674629803966,
    "2013-02-25": 1.0896794642964307,
    "2013-02-26": 1.1068537387000426,
    "2013-02-27": 1.1214018563689536,
    "2013-02-28": 1.1326696721095246,
    "2013-03-01": 1.1435323067611876,
    "2013-03-04": 1.1506626563122644,
    "2013-03-05": 1.1435890411762253,
    "2013-03-06": 1.127131928500491,
    "2013-03-07": 1.1299903745973034,
    "2013-03-08": 1.1208958002320413,
    "2013-03-11": 1.1137654506809644,
    "2013-03-12": 1.093669962517185,
    "2013-03-13": 1.080186696985227,
    "2013-03-14": 1.0565064848384,
    "2013-03-15": 1.05,
    "2013-03-18": 1.05,
    "2013-03-19": 1.05,
    "2013-03-20": 1.05,
    "2013-03-21": 1.05,
    "2013-03-22": 1.05,
    "2013-03-25": 1.05,
    "2013-03-26": 1.0643097509742845,
    "2013-03-27": 1.0645929349195176,
    "2013-03-28": 1.0725891066768825,
    "2013-03-29": 1.0725891066768825,
    "2013-04-01": 1.0767269871740102,
    "2013-04-02": 1.0834864103525752,
    "2013-04-03": 1.0751526081117715,
    "2013-04-04": 1.0748694241665384,
    "2013-04-05": 1.0668732524091737,
    "2013-04-08": 1.0627353719120458,
    "2013-04-09": 1.067823992057826,
    "2013-04-10": 1.0618480433243453,
    "2013-04-11": 1.0618480433243453,
    "2013-04-12": 1.0618480433243453,
    "2013-04-15": 1.0618480433243451,
    "2013-04-16": 1.05,
    "2013-04-17": 1.05,
    "2013-04-18": 1.05,
    "2013-04-19": 1.05,
    "2013-04-22": 1.05,
    "2013-04-23": 1.05,
    "2013-04-24": 1.05,
    "2013-04-25": 1.05,
    "2013-04-26": 1.05,
    "2013-04-29": 1.05,
    "2013-04-30": 1.05,
    "2013-05-01": 1.05,
    "2013-05-02": 1.05,
    "2013-05-03": 1.05,
    "2013-05-06": 1.05,
    "2013-05-07": 1.05,
    "2013-05-08": 1.05,
    "2013-05-09": 1.05,
    "2013-05-10": 1.05,
    "2013-05-13": 1.05,
    "2013-05-14": 1.05,
    "2013-05-15": 1.05,
    "2013-05-16": 1.05,
    "2013-05-17": 1.05,
    "2013-05-20": 1.05,
    "2013-05-21": 1.05,
    "2013-05-22": 1.05,
    "2013-05-23": 1.05,
    "2013-05-24": 1.05,
    "2013-05-27": 1.05,
    "2013-05-28": 1.05,
    "2013-05-29": 1.05,
    "2013-05-30": 1.05,
    "2013-05-31": 1.05,
    "2013-06-03": 1.05,
    "2013-06-04": 1.0610928460297242,
    "2013-06-05": 1.067361600348076,
    "2013-06-06": 1.0729263149521606,
    "2013-06-07": 1.0729263149521606,
    "2013-06-10": 1.0729263149521606,
    "2013-06-11": 1.0618885538827485,
    "2013-06-12": 1.0556197995643968,
    "2013-06-13": 1.0500550849603116,
    "2013-06-14": 1.0500550849603116,
    "2013-06-17": 1.0500550849603119,
    "2013-06-18": 1.0563176597599353,
    "2013-06-19": 1.056317659759935,
    "2013-06-20": 1.0601918496398903,
    "2013-06-21": 1.0782017440369844,
    "2013-06-24": 1.1134564636694981,
    "2013-06-25": 1.138978417823585,
    "2013-06-26": 1.1689750864785766,
    "2013-06-27": 1.1718823437883361,
    "2013-06-28": 1.166341923845035,
    "2013-07-01": 1.1478775479800432,
    "2013-07-02": 1.1341690247897018,
    "2013-07-03": 1.1402220414006863,
    "2013-07-04": 1.1402220414006863,
    "2013-07-05": 1.138546348223777,
    "2013-07-08": 1.1260768737699842,
    "2013-07-09": 1.1092865300024624,
    "2013-07-10": 1.0911554392787814,
    "2013-07-11": 1.0551057540128055,
    "2013-07-12": 1.05,
    "2013-07-15": 1.05,
    "2013-07-16": 1.05,
    "2013-07-17": 1.05,
    "2013-07-18": 1.05,
    "2013-07-19": 1.05,
    "2013-07-22": 1.05,
    "2013-07-23": 1.05,
    "2013-07-24": 1.05,
    "2013-07-25": 1.05,
    "2013-07-26": 1.05,
    "2013-07-29": 1.05,
    "2013-07-30": 1.05,
    "2013-07-31": 1.05,
    "2013-08-01": 1.05,
    "2013-08-02": 1.05,
    "2013-08-05": 1.05,
    "2013-08-06": 1.05,
    "2013-08-07": 1.05,
    "2013-08-08": 1.05,
    "2013-08-09": 1.05,
    "2013-08-12": 1.05,
    "2013-08-13": 1.05,
    "2013-08-14": 1.05,
    "2013-08-15": 1.05,
    "2013-08-16": 1.0539626315532733,
    "2013-08-19": 1.072540591613166,
    "2013-08-20": 1.072540591613166,
    "2013-08-21": 1.0762914663575283,
    "2013-08-22": 1.0825961453183823,
    "2013-08-23": 1.0786335137651089,
    "2013-08-26": 1.060055553705216,
    "2013-08-27": 1.0600555537052159,
    "2013-08-28": 1.056304678960854,
    "2013-08-29": 1.05,
    "2013-08-30": 1.062617890324143,
    "2013-09-02": 1.062617890324143,
    "2013-09-03": 1.0941159091819257,
    "2013-09-04": 1.1242511637616057,
    "2013-09-05": 1.1584207595916987,
    "2013-09-06": 1.1798602788707266,
    "2013-09-09": 1.1742778884040401,
    "2013-09-10": 1.1497689063633254,
    "2013-09-11": 1.1196336517836456,
    "2013-09-12": 1.0902937787952223,
    "2013-09-13": 1.0729370579530229,
    "2013-09-16": 1.0666949054721315,
    "2013-09-17": 1.0597058686550633,
    "2013-09-18": 1.0597058686550633,
    "2013-09-19": 1.0548761458133942,
    "2013-09-20": 1.0507933473765652,
    "2013-09-23": 1.05,
    "2013-09-24": 1.05,
    "2013-09-25": 1.05,
    "2013-09-26": 1.05,
    "2013-09-27": 1.05,
    "2013-09-30": 1.05,
    "2013-10-01": 1.05,
    "2013-10-02": 1.0513667134980396,
    "2013-10-03": 1.0562627393456965,
    "2013-10-04": 1.082676512457317,
    "2013-10-07": 1.1047937474134613,
    "2013-10-08": 1.1308447022790589,
    "2013-10-09": 1.1719224426696953,
    "2013-10-10": 1.2012711627643589,
    "2013-10-11": 1.1997513665293964,
    "2013-10-14": 1.201266807862256,
    "2013-10-15": 1.191099249573786,
    "2013-10-16": 1.1523185039276254,
    "2013-10-17": 1.1621174359251618,
    "2013-10-18": 1.2021965591004746,
    "2013-10-21": 1.2359696784729381,
    "2013-10-22": 1.2200862818958105,
    "2013-10-23": 1.2164225736532952,
    "2013-10-24": 1.172378895713438,
    "2013-10-25": 1.1074057956614671,
    "2013-10-28": 1.05,
    "2013-10-29": 1.05,
    "2013-10-30": 1.05,
    "2013-10-31": 1.0503242592022761,
    "2013-11-01": 1.055123123884846,
    "2013-11-04": 1.055123123884846,
    "2013-11-05": 1.055123123884846,
    "2013-11-06": 1.055123123884846,
    "2013-11-07": 1.05479886468257,
    "2013-11-08": 1.05,
    "2013-11-11": 1.05,
    "2013-11-12": 1.05,
    "2013-11-13": 1.05,
    "2013-11-14": 1.05,
    "2013-11-15": 1.05,
    "2013-11-18": 1.05,
    "2013-11-19": 1.05,
    "2013-11-20": 1.05,
    "2013-11-21": 1.05,
    "2013-11-22": 1.05,
    "2013-11-25": 1.05,
    "2013-11-26": 1.05,
    "2013-11-27": 1.05,
    "2013-11-28": 1.05,
    "2013-11-29": 1.05,
    "2013-12-02": 1.05,
    "2013-12-03": 1.05,
    "2013-12-04": 1.05,
    "2013-12-05": 1.05,
    "2013-12-06": 1.05,
    "2013-12-09": 1.05,
    "2013-12-10": 1.0701698180550612,
    "2013-12-11": 1.0833454814395826,
    "2013-12-12": 1.1014873341332074,
    "2013-12-13": 1.1270697281066204,
    "2013-12-16": 1.1474823335159943,
    "2013-12-17": 1.1367780447290188,
    "2013-12-18": 1.123602381344497,
    "2013-12-19": 1.1125679781030036,
    "2013-12-20": 1.1050540278155134,
    "2013-12-23": 1.0945383127484731,
    "2013-12-24": 1.098918458340951,
    "2013-12-25": 1.098918458340951,
    "2013-12-26": 1.1113161092833426,
    "2013-12-27": 1.127532130184949,
    "2013-12-30": 1.1391678740643276,
    "2013-12-31": 1.1679550166607853,
    "2014-01-01": 1.1679550166607853,
    "2014-01-02": 1.1909611413373615,
    "2014-01-03": 1.2187809751400867,
    "2014-01-06": 1.23346450031536,
    "2014-01-07": 1.2527807792258456,
    "2014-01-08": 1.2569334384762003,
    "2014-01-09": 1.2563889653042017,
    "2014-01-10": 1.2161714805590849,
    "2014-01-13": 1.178164485030074,
    "2014-01-14": 1.1291440185542871,
    "2014-01-15": 1.0863073263651413,
    "2014-01-16": 1.05,
    "2014-01-17": 1.05,
    "2014-01-20": 1.05,
    "2014-01-21": 1.05,
    "2014-01-22": 1.05,
    "2014-01-23": 1.05,
    "2014-01-24": 1.05,
    "2014-01-27": 1.05,
    "2014-01-28": 1.05,
    "2014-01-29": 1.05,
    "2014-01-30": 1.05,
    "2014-01-31": 1.05,
    "2014-02-03": 1.05,
    "2014-02-04": 1.05,
    "2014-02-05": 1.05,
    "2014-02-06": 1.05,
    "2014-02-07": 1.05,
    "2014-02-10": 1.05,
    "2014-02-11": 1.05,
    "2014-02-12": 1.05,
    "2014-02-13": 1.05,
    "2014-02-14": 1.05,
    "2014-02-17": 1.05,
    "2014-02-18": 1.05,
    "2014-02-19": 1.05,
    "2014-02-20": 1.05,
    "2014-02-21": 1.05,
    "2014-02-24": 1.05,
    "2014-02-25": 1.05,
    "2014-02-26": 1.05,
    "2014-02-27": 1.05,
    "2014-02-28": 1.05,
    "2014-03-03": 1.05,
    "2014-03-04": 1.05,
    "2014-03-05": 1.0509442390167747,
    "2014-03-06": 1.0537319426592586,
    "2014-03-07": 1.0537319426592586,
    "2014-03-10": 1.0537319426592586,
    "2014-03-11": 1.0537319426592586,
    "2014-03-12": 1.0527877036424838,
    "2014-03-13": 1.05,
    "2014-03-14": 1.05,
    "2014-03-17": 1.05,
    "2014-03-18": 1.05,
    "2014-03-19": 1.05,
    "2014-03-20": 1.05,
    "2014-03-21": 1.05,
    "2014-03-24": 1.05,
    "2014-03-25": 1.05,
    "2014-03-26": 1.05,
    "2014-03-27": 1.05,
    "2014-03-28": 1.05,
    "2014-03-31": 1.05,
    "2014-04-01": 1.05,
    "2014-04-02": 1.0537844107286216,
    "2014-04-03": 1.0537844107286216,
    "2014-04-04": 1.0537844107286216,
    "2014-04-07": 1.0537844107286214,
    "2014-04-08": 1.0537844107286216,
    "2014-04-09": 1.05,
    "2014-04-10": 1.05,
    "2014-04-11": 1.05,
    "2014-04-14": 1.05,
    "2014-04-15": 1.05,
    "2014-04-16": 1.05,
    "2014-04-17": 1.0561334270488052,
    "2014-04-18": 1.0561334270488052,
    "2014-04-21": 1.0561334270488052,
    "2014-04-22": 1.0561334270488054,
    "2014-04-23": 1.0561334270488052,
    "2014-04-24": 1.0561334270488052,
    "2014-04-25": 1.05,
    "2014-04-28": 1.0572572350807503,
    "2014-04-29": 1.0737548926135119,
    "2014-04-30": 1.0778067457778022,
    "2014-05-01": 1.0833458740944617,
    "2014-05-02": 1.0887582072409052,
    "2014-05-05": 1.0826886256709152,
    "2014-05-06": 1.0984661607573178,
    "2014-05-07": 1.125105368332973,
    "2014-05-08": 1.1490898605614737,
    "2014-05-09": 1.164701579682238,
    "2014-05-12": 1.1977237356980475,
    "2014-05-13": 1.189665248035047,
    "2014-05-14": 1.1589741872951014,
    "2014-05-15": 1.129450566749941,
    "2014-05-16": 1.1097326161382202,
    "2014-05-19": 1.118434038019221,
    "2014-05-20": 1.143271955929071,
    "2014-05-21": 1.179911712014191,
    "2014-05-22": 1.2154513405086003,
    "2014-05-23": 1.249177053232584,
    "2014-05-26": 1.249177053232584,
    "2014-05-27": 1.2360930540304875,
    "2014-05-28": 1.1870384311644737,
    "2014-05-29": 1.1548532226983217,
    "2014-05-30": 1.13577417380935,
    "2014-06-02": 1.100742359429879,
    "2014-06-03": 1.0709151272244053,
    "2014-06-04": 1.0721553851804426,
    "2014-06-05": 1.0688065318362197,
    "2014-06-06": 1.0523459522307825,
    "2014-06-09": 1.0523459522307825,
    "2014-06-10": 1.0523459522307825,
    "2014-06-11": 1.0511056942747454,
    "2014-06-12": 1.05,
    "2014-06-13": 1.05,
    "2014-06-16": 1.054467482395361,
    "2014-06-17": 1.054467482395361,
    "2014-06-18": 1.054467482395361,
    "2014-06-19": 1.0544674823953608,
    "2014-06-20": 1.054467482395361,
    "2014-06-23": 1.05,
    "2014-06-24": 1.05,
    "2014-06-25": 1.05,
    "2014-06-26": 1.05,
    "2014-06-27": 1.05,
    "2014-06-30": 1.05,
    "2014-07-01": 1.0559719243404373,
    "2014-07-02": 1.0832775865394946,
    "2014-07-03": 1.0986801492790068,
    "2014-07-04": 1.0986801492790068,
    "2014-07-07": 1.10406066930056,
    "2014-07-08": 1.10406066930056,
    "2014-07-09": 1.0980887449601227,
    "2014-07-10": 1.0707830827610654,
    "2014-07-11": 1.055380520021553,
    "2014-07-14": 1.0509405958770845,
    "2014-07-15": 1.0523568292717416,
    "2014-07-16": 1.0523568292717416,
    "2014-07-17": 1.0595488731410867,
    "2014-07-18": 1.0715836655516289,
    "2014-07-21": 1.0817810184586703,
    "2014-07-22": 1.091802122300037,
    "2014-07-23": 1.1016640543065148,
    "2014-07-24": 1.0990209024447706,
    "2014-07-25": 1.0882508211163693,
    "2014-07-28": 1.0787401006750654,
    "2014-07-29": 1.0673027634390417,
    "2014-07-30": 1.057440831432564,
    "2014-07-31": 1.052891939424963,
    "2014-08-01": 1.0516272283428223,
    "2014-08-04": 1.05,
    "2014-08-05": 1.05,
    "2014-08-06": 1.0570182882043286,
    "2014-08-07": 1.064835468269634,
    "2014-08-08": 1.0838901595550263,
    "2014-08-11": 1.0947234219845199,
    "2014-08-12": 1.1041342508127296,
    "2014-08-13": 1.1020955547433229,
    "2014-08-14": 1.098062563285005,
    "2014-08-15": 1.0937513207513447,
    "2014-08-18": 1.0928860360303538,
    "2014-08-19": 1.092603087466632,
    "2014-08-20": 1.097310484518569,
    "2014-08-21": 1.0935262959115815,
    "2014-08-22": 1.087189455814523,
    "2014-08-25": 1.0811214173453099,
    "2014-08-26": 1.0760511865565814,
    "2014-08-27": 1.0718871357300916,
    "2014-08-28": 1.1623719543413789,
    "2014-08-29": 1.2452769118296605,
    "2014-09-01": 1.2452769118296605,
    "2014-09-02": 1.290627896267805,
    "2014-09-03": 1.351404890276699,
    "2014-09-04": 1.3988475514578167,
    "2014-09-05": 1.341820786563111,
    "2014-09-08": 1.300168640358749,
    "2014-09-09": 1.3022387102910558,
    "2014-09-10": 1.2896397826581316,
    "2014-09-11": 1.2889146441611092,
    "2014-09-12": 1.2907394345296155,
    "2014-09-15": 1.2753100042591832,
    "2014-09-16": 1.2872977119143116,
    "2014-09-17": 1.285495130495105,
    "2014-09-18": 1.2804027156457476,
    "2014-09-19": 1.2680005921564341,
    "2014-09-22": 1.2490478922625698,
    "2014-09-23": 1.1920103431338342,
    "2014-09-24": 1.1504947828133818,
    "2014-09-25": 1.1033467366182748,
    "2014-09-26": 1.0972723593901659,
    "2014-09-29": 1.0850314321755548,
    "2014-09-30": 1.0879495332142832,
    "2014-10-01": 1.0790319591022128,
    "2014-10-02": 1.0790319591022128,
    "2014-10-03": 1.0622256157345475,
    "2014-10-06": 1.0591892531748621,
    "2014-10-07": 1.05,
    "2014-10-08": 1.05,
    "2014-10-09": 1.05,
    "2014-10-10": 1.05,
    "2014-10-13": 1.0775625678449166,
    "2014-10-14": 1.1082997187722428,
    "2014-10-15": 1.186806453068185,
    "2014-10-16": 1.2152495181081895,
    "2014-10-17": 1.2199870383106712,
    "2014-10-20": 1.1924244704657547,
    "2014-10-21": 1.1616873195384285,
    "2014-10-22": 1.0831805852424865,
    "2014-10-23": 1.0547375202024818,
    "2014-10-24": 1.05,
    "2014-10-27": 1.05,
    "2014-10-28": 1.05,
    "2014-10-29": 1.05,
    "2014-10-30": 1.05,
    "2014-10-31": 1.05,
    "2014-11-03": 1.05,
    "2014-11-04": 1.05,
    "2014-11-05": 1.0601696970843746,
    "2014-11-06": 1.0601696970843748,
    "2014-11-07": 1.0601696970843746,
    "2014-11-10": 1.0601696970843746,
    "2014-11-11": 1.0601696970843746,
    "2014-11-12": 1.05,
    "2014-11-13": 1.125350156516106,
    "2014-11-14": 1.2130585463751193,
    "2014-11-17": 1.3134853300867453,
    "2014-11-18": 1.42458446305444,
    "2014-11-19": 1.5384639842845118,
    "2014-11-20": 1.57058756323567,
    "2014-11-21": 1.593930497230486,
    "2014-11-24": 1.5917398250416652,
    "2014-11-25": 1.5569058745082427,
    "2014-11-26": 1.5212686541627263,
    "2014-11-27": 1.5212686541627263,
    "2014-11-28": 1.5061733544709264,
    "2014-12-01": 1.4841111289828712,
    "2014-12-02": 1.4445765568716218,
    "2014-12-03": 1.422508340702319,
    "2014-12-04": 1.3891718870703031,
    "2014-12-05": 1.3271151210538474,
    "2014-12-08": 1.2547360694983254,
    "2014-12-09": 1.238678095353897,
    "2014-12-10": 1.2389127732062573,
    "2014-12-11": 1.2470612038700355,
    "2014-12-12": 1.247756554548899,
    "2014-12-15": 1.29217772050947,
    "2014-12-16": 1.3167211790894078,
    "2014-12-17": 1.27477947054509,
    "2014-12-18": 1.2217251926287718,
    "2014-12-19": 1.1907081721908994,
    "2014-12-22": 1.1296769594200766,
    "2014-12-23": 1.0624899355730115,
    "2014-12-24": 1.05,
    "2014-12-25": 1.05,
    "2014-12-26": 1.05,
    "2014-12-29": 1.05,
    "2014-12-30": 1.05,
    "2014-12-31": 1.05,
    "2015-01-01": 1.05,
    "2015-01-02": 1.05,
    "2015-01-05": 1.05,
    "2015-01-06": 1.0546849913055143,
    "2015-01-07": 1.0699546794483492,
    "2015-01-08": 1.074803517232248,
    "2015-01-09": 1.074803517232248,
    "2015-01-12": 1.074803517232248,
    "2015-01-13": 1.070118525926734,
    "2015-01-14": 1.0632122548601388,
    "2015-01-15": 1.0634072371875618,
    "2015-01-16": 1.0823355912323103,
    "2015-01-19": 1.0823355912323103,
    "2015-01-20": 1.1166403608375,
    "2015-01-21": 1.148510583773234,
    "2015-01-22": 1.1549113774676596,
    "2015-01-23": 1.1672619436732974,
    "2015-01-26": 1.1861267495472714,
    "2015-01-27": 1.1937521250548102,
    "2015-01-28": 1.1861712341280293,
    "2015-01-29": 1.1905926746296012,
    "2015-01-30": 1.185711893106885,
    "2015-02-02": 1.1631849342077232,
    "2015-02-03": 1.1212547890949947,
    "2015-02-04": 1.0992698230734264,
    "2015-02-05": 1.0909278821009654,
    "2015-02-06": 1.0784142773067216,
    "2015-02-09": 1.0631480762871612,
    "2015-02-10": 1.0631480762871612,
    "2015-02-11": 1.060843710299776,
    "2015-02-12": 1.05,
    "2015-02-13": 1.05,
    "2015-02-16": 1.05,
    "2015-02-17": 1.05,
    "2015-02-18": 1.05,
    "2015-02-19": 1.05,
    "2015-02-20": 1.05,
    "2015-02-23": 1.05,
    "2015-02-24": 1.05,
    "2015-02-25": 1.05,
    "2015-02-26": 1.05,
    "2015-02-27": 1.05,
    "2015-03-02": 1.05,
    "2015-03-03": 1.05,
    "2015-03-04": 1.05,
    "2015-03-05": 1.05,
    "2015-03-06": 1.05,
    "2015-03-09": 1.05,
    "2015-03-10": 1.05,
    "2015-03-11": 1.05,
    "2015-03-12": 1.05,
    "2015-03-13": 1.05,
    "2015-03-16": 1.05,
    "2015-03-17": 1.05,
    "2015-03-18": 1.05,
    "2015-03-19": 1.05,
    "2015-03-20": 1.05,
    "2015-03-23": 1.05,
    "2015-03-24": 1.05,
    "2015-03-25": 1.05,
    "2015-03-26": 1.05,
    "2015-03-27": 1.05,
    "2015-03-30": 1.05,
    "2015-03-31": 1.05,
    "2015-04-01": 1.05,
    "2015-04-02": 1.05,
    "2015-04-03": 1.05,
    "2015-04-06": 1.05,
    "2015-04-07": 1.061562259086509,
    "2015-04-08": 1.0810757046040176,
    "2015-04-09": 1.0910981390541619,
    "2015-04-10": 1.0985548531438671,
    "2015-04-13": 1.1004474958398263,
    "2015-04-14": 1.0888852367533173,
    "2015-04-15": 1.069371791235809,
    "2015-04-16": 1.0593493567856647,
    "2015-04-17": 1.0826304122318515,
    "2015-04-20": 1.1028372566256845,
    "2015-04-21": 1.1284984111093492,
    "2015-04-22": 1.144113436219805,
    "2015-04-23": 1.1634981737266394,
    "2015-04-24": 1.1493775402888315,
    "2015-04-27": 1.176351639352509,
    "2015-04-28": 1.211167916821197,
    "2015-04-29": 1.241510781691639,
    "2015-04-30": 1.2783736185055172,
    "2015-05-01": 1.3403860564035974,
    "2015-05-04": 1.3815586808635705,
    "2015-05-05": 1.4309710116271934,
    "2015-05-06": 1.4953096286832923,
    "2015-05-07": 1.5180665209489241,
    "2015-05-08": 1.4929449817408327,
    "2015-05-11": 1.423812282001629,
    "2015-05-12": 1.3437815891466136,
    "2015-05-13": 1.2661205077466957,
    "2015-05-14": 1.2040364279766949,
    "2015-05-15": 1.1505283931886214,
    "2015-05-18": 1.1294148823143826,
    "2015-05-19": 1.0995558124534222,
    "2015-05-20": 1.066920386816343,
    "2015-05-21": 1.05,
    "2015-05-22": 1.05,
    "2015-05-25": 1.05,
    "2015-05-26": 1.05,
    "2015-05-27": 1.05,
    "2015-05-28": 1.05,
    "2015-05-29": 1.05,
    "2015-06-01": 1.05,
    "2015-06-02": 1.05,
    "2015-06-03": 1.05,
    "2015-06-04": 1.05,
    "2015-06-05": 1.05,
    "2015-06-08": 1.05,
    "2015-06-09": 1.05,
    "2015-06-10": 1.05,
    "2015-06-11": 1.05,
    "2015-06-12": 1.05,
    "2015-06-15": 1.05,
    "2015-06-16": 1.05,
    "2015-06-17": 1.05,
    "2015-06-18": 1.05,
    "2015-06-19": 1.05,
    "2015-06-22": 1.05,
    "2015-06-23": 1.05,
    "2015-06-24": 1.05,
    "2015-06-25": 1.05,
    "2015-06-26": 1.05,
    "2015-06-29": 1.05,
    "2015-06-30": 1.05,
    "2015-07-01": 1.05,
    "2015-07-02": 1.05,
    "2015-07-03": 1.05,
    "2015-07-06": 1.05,
    "2015-07-07": 1.05,
    "2015-07-08": 1.05,
    "2015-07-09": 1.05,
    "2015-07-10": 1.05,
    "2015-07-13": 1.05,
    "2015-07-14": 1.05,
    "2015-07-15": 1.05,
    "2015-07-16": 1.05,
    "2015-07-17": 1.05,
    "2015-07-20": 1.05,
    "2015-07-21": 1.05,
    "2015-07-22": 1.05,
    "2015-07-23": 1.05,
    "2015-07-24": 1.05,
    "2015-07-27": 1.05,
    "2015-07-28": 1.05,
    "2015-07-29": 1.05,
    "2015-07-30": 1.05,
    "2015-07-31": 1.05,
    "2015-08-03": 1.05,
    "2015-08-04": 1.0500474179019357,
    "2015-08-05": 1.050047417901936,
    "2015-08-06": 1.050047417901936,
    "2015-08-07": 1.050047417901936,
    "2015-08-10": 1.0778855013851378,
    "2015-08-11": 1.0898364704905437,
    "2015-08-12": 1.1122985862725396,
    "2015-08-13": 1.1214276748092602,
    "2015-08-14": 1.1223133061608601,
    "2015-08-17": 1.094475222677658,
    "2015-08-18": 1.0901806369733473,
    "2015-08-19": 1.0677185211913511,
    "2015-08-20": 1.062286500926566,
    "2015-08-21": 1.076758530416169,
    "2015-08-24": 1.10261137746332,
    "2015-08-25": 1.094907576160289,
    "2015-08-26": 1.094907576160289,
    "2015-08-27": 1.0912105078883538,
    "2015-08-28": 1.075852847047151,
    "2015-08-31": 1.05,
    "2015-09-01": 1.05,
    "2015-09-02": 1.0501727994366106,
    "2015-09-03": 1.051814688964169,
    "2015-09-04": 1.051814688964169,
    "2015-09-07": 1.051814688964169,
    "2015-09-08": 1.051814688964169,
    "2015-09-09": 1.051814688964169,
    "2015-09-10": 1.0558828426562532,
    "2015-09-11": 1.0553523722043794,
    "2015-09-14": 1.0610219342296325,
    "2015-09-15": 1.0610219342296323,
    "2015-09-16": 1.0610219342296325,
    "2015-09-17": 1.0567809811009377,
    "2015-09-18": 1.0556695620252532,
    "2015-09-21": 1.05,
    "2015-09-22": 1.05,
    "2015-09-23": 1.05,
    "2015-09-24": 1.05,
    "2015-09-25": 1.05,
    "2015-09-28": 1.05,
    "2015-09-29": 1.05,
    "2015-09-30": 1.05,
    "2015-10-01": 1.05,
    "2015-10-02": 1.05,
    "2015-10-05": 1.05,
    "2015-10-06": 1.05,
    "2015-10-07": 1.05,
    "2015-10-08": 1.05,
    "2015-10-09": 1.05,
    "2015-10-12": 1.05,
    "2015-10-13": 1.05,
    "2015-10-14": 1.05,
    "2015-10-15": 1.05,
    "2015-10-16": 1.05,
    "2015-10-19": 1.05,
    "2015-10-20": 1.05,
    "2015-10-21": 1.0563553486210147,
    "2015-10-22": 1.0655820491254062,
    "2015-10-23": 1.0665206070935092,
    "2015-10-26": 1.0708477243525742,
    "2015-10-27": 1.0897951668420531,
    "2015-10-28": 1.094631799987117,
    "2015-10-29": 1.0865438348921899,
    "2015-10-30": 1.0861850179740884,
    "2015-11-02": 1.0909375004873956,
    "2015-11-03": 1.0906334204691641,
    "2015-11-04": 1.1147945603911948,
    "2015-11-05": 1.1472705468396294,
    "2015-11-06": 1.1564758571163192,
    "2015-11-09": 1.1584065431587844,
    "2015-11-10": 1.1397631806875368,
    "2015-11-11": 1.1091178497786522,
    "2015-11-12": 1.1050991368249088,
    "2015-11-13": 1.1092255361528762,
    "2015-11-16": 1.1093451638270047,
    "2015-11-17": 1.1335020112274399,
    "2015-11-18": 1.1552112616837622,
    "2015-11-19": 1.1548757329573813,
    "2015-11-20": 1.1712761852197193,
    "2015-11-23": 1.2002054742179133,
    "2015-11-24": 1.2208942392388515,
    "2015-11-25": 1.2420861920131334,
    "2015-11-26": 1.2420861920131334,
    "2015-11-27": 1.276497954858593,
    "2015-11-30": 1.360287950046224,
    "2015-12-01": 1.405565450926376,
    "2015-12-02": 1.4579816562768042,
    "2015-12-03": 1.4131576123585516,
    "2015-12-04": 1.349485369335317,
    "2015-12-07": 1.2353834712306897,
    "2015-12-08": 1.1536003649165476,
    "2015-12-09": 1.0578820446597834,
    "2015-12-10": 1.0550970945682074,
    "2015-12-11": 1.0550970945682074,
    "2015-12-14": 1.0550970945682074,
    "2015-12-15": 1.051543497515038,
    "2015-12-16": 1.05,
    "2015-12-17": 1.05,
    "2015-12-18": 1.05,
    "2015-12-21": 1.05,
    "2015-12-22": 1.05,
    "2015-12-23": 1.05,
    "2015-12-24": 1.05,
    "2015-12-25": 1.05,
    "2015-12-28": 1.05,
    "2015-12-29": 1.05,
    "2015-12-30": 1.05,
    "2015-12-31": 1.05,
    "2016-01-01": 1.05,
    "2016-01-04": 1.05,
    "2016-01-05": 1.05,
    "2016-01-06": 1.05,
    "2016-01-07": 1.05,
    "2016-01-08": 1.05,
    "2016-01-11": 1.05,
    "2016-01-12": 1.05,
    "2016-01-13": 1.0602737645512785,
    "2016-01-14": 1.0922036973300941,
    "2016-01-15": 1.1365434174229478,
    "2016-01-18": 1.1365434174229478,
    "2016-01-19": 1.18169744970128,
    "2016-01-20": 1.2410010423261824,
    "2016-01-21": 1.2810220523241203,
    "2016-01-22": 1.2832571113415434,
    "2016-01-25": 1.2759337103279478,
    "2016-01-26": 1.2715538076921857,
    "2016-01-27": 1.2410666593335207,
    "2016-01-28": 1.2164918834096865,
    "2016-01-29": 1.2601281808097589,
    "2016-02-01": 1.2829902458778955,
    "2016-02-02": 1.2827661421339953,
    "2016-02-03": 1.2963885287607142,
    "2016-02-04": 1.3280702694625663,
    "2016-02-05": 1.3141402501614234,
    "2016-02-08": 1.3048420396591858,
    "2016-02-09": 1.321094699445925,
    "2016-02-10": 1.3282354285516889,
    "2016-02-11": 1.3489483742428146,
    "2016-02-12": 1.3108368378967445,
    "2016-02-15": 1.3108368378967445,
    "2016-02-16": 1.2690622408286079,
    "2016-02-17": 1.2156261258564667,
    "2016-02-18": 1.1682586802265793,
    "2016-02-19": 1.1054290821978314,
    "2016-02-22": 1.0853495635892199,
    "2016-02-23": 1.083065676066676,
    "2016-02-24": 1.0891480741705282,
    "2016-02-25": 1.0929642226223604,
    "2016-02-26": 1.0776791356327484,
    "2016-02-29": 1.0719989206922618,
    "2016-03-01": 1.0654772316377852,
    "2016-03-02": 1.0560282628206654,
    "2016-03-03": 1.05,
    "2016-03-04": 1.05,
    "2016-03-07": 1.05,
    "2016-03-08": 1.05,
    "2016-03-09": 1.05,
    "2016-03-10": 1.05,
    "2016-03-11": 1.05,
    "2016-03-14": 1.0509102830176031,
    "2016-03-15": 1.0720115684977587,
    "2016-03-16": 1.0791125680479043,
    "2016-03-17": 1.0798790848946773,
    "2016-03-18": 1.0856276173513586,
    "2016-03-21": 1.0847173343337555,
    "2016-03-22": 1.0640158434939833,
    "2016-03-23": 1.0569148439438378,
    "2016-03-24": 1.0561483270970649,
    "2016-03-25": 1.0561483270970649,
    "2016-03-28": 1.053885275346959,
    "2016-03-29": 1.053885275346959,
    "2016-03-30": 1.0534854807065757,
    "2016-03-31": 1.0603334169256946,
    "2016-04-01": 1.0621706721907587,
    "2016-04-04": 1.0586851914841833,
    "2016-04-05": 1.0691152719058683,
    "2016-04-06": 1.0820507286017018,
    "2016-04-07": 1.0952415601131513,
    "2016-04-08": 1.1260127800050415,
    "2016-04-11": 1.1638856174555319,
    "2016-04-12": 1.1825601369709235,
    "2016-04-13": 1.1939306558922653,
    "2016-04-14": 1.1949911773128838,
    "2016-04-15": 1.17886719215544,
    "2016-04-18": 1.1567260036423102,
    "2016-04-19": 1.1457323676894058,
    "2016-04-20": 1.1340814099170062,
    "2016-04-21": 1.127838953374985,
    "2016-04-22": 1.1361071440458024,
    "2016-04-25": 1.147402655203501,
    "2016-04-26": 1.154804743103249,
    "2016-04-27": 1.1509656465169669,
    "2016-04-28": 1.1433175308969723,
    "2016-04-29": 1.1333008933929172,
    "2016-05-02": 1.1126406041349322,
    "2016-05-03": 1.0904170048627855,
    "2016-05-04": 1.092914201909496,
    "2016-05-05": 1.0961224153057425,
    "2016-05-06": 1.093246883968442,
    "2016-05-09": 1.0959843397383728,
    "2016-05-10": 1.1012927689626946,
    "2016-05-11": 1.1102638109648832,
    "2016-05-12": 1.119628764984236,
    "2016-05-13": 1.118962293484731,
    "2016-05-16": 1.1237703756408859,
    "2016-05-17": 1.131344294949849,
    "2016-05-18": 1.1110601346424565,
    "2016-05-19": 1.0912782502376852,
    "2016-05-20": 1.0800842099082182,
    "2016-05-23": 1.066171801145058,
    "2016-05-24": 1.05,
    "2016-05-25": 1.05,
    "2016-05-26": 1.05,
    "2016-05-27": 1.05,
    "2016-05-30": 1.05,
    "2016-05-31": 1.05,
    "2016-06-01": 1.0526509464146134,
    "2016-06-02": 1.0682043709526945,
    "2016-06-03": 1.0682043709526945,
    "2016-06-06": 1.0682043709526945,
    "2016-06-07": 1.0682043709526945,
    "2016-06-08": 1.065553424538081,
    "2016-06-09": 1.05,
    "2016-06-10": 1.0502118252501491,
    "2016-06-13": 1.0639327237280418,
    "2016-06-14": 1.082544970725561,
    "2016-06-15": 1.10889066893634,
    "2016-06-16": 1.1486240162829966,
    "2016-06-17": 1.2497028745013237,
    "2016-06-20": 1.2980996759491066,
    "2016-06-21": 1.3348499286965179,
    "2016-06-22": 1.3726502328624839,
    "2016-06-23": 1.365332476910796,
    "2016-06-24": 1.2640417934423198,
    "2016-06-27": 1.201924093516644,
    "2016-06-28": 1.1465615937717137,
    "2016-06-29": 1.0824155913949687,
    "2016-06-30": 1.05,
    "2016-07-01": 1.05,
    "2016-07-04": 1.05,
    "2016-07-05": 1.05,
    "2016-07-06": 1.05,
    "2016-07-07": 1.05,
    "2016-07-08": 1.05,
    "2016-07-11": 1.05,
    "2016-07-12": 1.05,
    "2016-07-13": 1.05,
    "2016-07-14": 1.05,
    "2016-07-15": 1.05,
    "2016-07-18": 1.05,
    "2016-07-19": 1.05,
    "2016-07-20": 1.05,
    "2016-07-21": 1.05,
    "2016-07-22": 1.05,
    "2016-07-25": 1.05,
    "2016-07-26": 1.05,
    "2016-07-27": 1.05,
    "2016-07-28": 1.05,
    "2016-07-29": 1.05,
    "2016-08-01": 1.05,
    "2016-08-02": 1.05,
    "2016-08-03": 1.060129936960379,
    "2016-08-04": 1.061149082509108,
    "2016-08-05": 1.0611490825091083,
    "2016-08-08": 1.0611490825091083,
    "2016-08-09": 1.061149082509108,
    "2016-08-10": 1.0510191455487294,
    "2016-08-11": 1.05,
    "2016-08-12": 1.05,
    "2016-08-15": 1.05,
    "2016-08-16": 1.05,
    "2016-08-17": 1.05,
    "2016-08-18": 1.05,
    "2016-08-19": 1.05,
    "2016-08-22": 1.0525476952892556,
    "2016-08-23": 1.0525476952892556,
    "2016-08-24": 1.0525476952892556,
    "2016-08-25": 1.0666166719430197,
    "2016-08-26": 1.0768500720462126,
    "2016-08-29": 1.074302376756957,
    "2016-08-30": 1.074820114960473,
    "2016-08-31": 1.0798918371371626,
    "2016-09-01": 1.07457921262441,
    "2016-09-02": 1.0652659391264625,
    "2016-09-05": 1.0652659391264625,
    "2016-09-06": 1.0673376563037968,
    "2016-09-07": 1.0709624317425128,
    "2016-09-08": 1.0658907095658232,
    "2016-09-09": 1.0606854487791597,
    "2016-09-12": 1.0740781897728229,
    "2016-09-13": 1.097127229262353,
    "2016-09-14": 1.105944857819567,
    "2016-09-15": 1.1193884613908969,
    "2016-09-16": 1.130975135890715,
    "2016-09-19": 1.1352649224071372,
    "2016-09-20": 1.1330135524853258,
    "2016-09-21": 1.1297209718033416,
    "2016-09-22": 1.1162773682320117,
    "2016-09-23": 1.1011396023778457,
    "2016-09-26": 1.082536948262515,
    "2016-09-27": 1.0596675615174616,
    "2016-09-28": 1.057388688209579,
    "2016-09-29": 1.069040589561255,
    "2016-09-30": 1.0752442190056948,
    "2016-10-03": 1.08038243235592,
    "2016-10-04": 1.08038243235592,
    "2016-10-05": 1.085623649859246,
    "2016-10-06": 1.0919350457779469,
    "2016-10-07": 1.1206522585504293,
    "2016-10-10": 1.1658749526485972,
    "2016-10-11": 1.2090840238592748,
    "2016-10-12": 1.268349896442151,
    "2016-10-13": 1.3262209513087524,
    "2016-10-14": 1.350109762214678,
    "2016-10-17": 1.3495154532107343,
    "2016-10-18": 1.3489375421095704,
    "2016-10-19": 1.314240031409994,
    "2016-10-20": 1.2704463025830204,
    "2016-10-21": 1.243030032276223,
    "2016-10-24": 1.2263577562365273,
    "2016-10-25": 1.2252348094937706,
    "2016-10-26": 1.249746325661825,
    "2016-10-27": 1.2656556516474837,
    "2016-10-28": 1.3068411693695619,
    "2016-10-31": 1.359888776971094,
    "2016-11-01": 1.4094452102126183,
    "2016-11-02": 1.4671770426503592,
    "2016-11-03": 1.5636541227538534,
    "2016-11-04": 1.6384997290393202,
    "2016-11-07": 1.6800263834148872,
    "2016-11-08": 1.6842476887644167,
    "2016-11-09": 1.5648060725624162,
    "2016-11-10": 1.4203790431632588,
    "2016-11-11": 1.4203790431632588,
    "2016-11-14": 1.2729545363396642,
    "2016-11-15": 1.1452859519578111,
    "2016-11-16": 1.05,
    "2016-11-17": 1.05,
    "2016-11-18": 1.05,
    "2016-11-21": 1.05,
    "2016-11-22": 1.05,
    "2016-11-23": 1.05,
    "2016-11-24": 1.05,
    "2016-11-25": 1.05,
    "2016-11-28": 1.05,
    "2016-11-29": 1.05,
    "2016-11-30": 1.05,
    "2016-12-01": 1.05,
    "2016-12-02": 1.05,
    "2016-12-05": 1.05,
    "2016-12-06": 1.05,
    "2016-12-07": 1.05,
    "2016-12-08": 1.05,
    "2016-12-09": 1.05,
    "2016-12-12": 1.0569666358811918,
    "2016-12-13": 1.0622208525646275,
    "2016-12-14": 1.09199855191097,
    "2016-12-15": 1.1246055018217063,
    "2016-12-16": 1.1598163580998713,
    "2016-12-19": 1.1668701367847096,
    "2016-12-20": 1.1829123723052426,
    "2016-12-21": 1.1665407439545659,
    "2016-12-22": 1.1501892030866727,
    "2016-12-23": 1.127631520302348,
    "2016-12-26": 1.127631520302348,
    "2016-12-27": 1.1201307341194846,
    "2016-12-28": 1.1037442699802686,
    "2016-12-29": 1.0945512420193422,
    "2016-12-30": 1.090719921683656,
    "2017-01-02": 1.090719921683656,
    "2017-01-03": 1.1269061752102227,
    "2017-01-04": 1.1820161964225964,
    "2017-01-05": 1.2068117595107481,
    "2017-01-06": 1.2200222976376636,
    "2017-01-09": 1.225941639413844,
    "2017-01-10": 1.1959308304621818,
    "2017-01-11": 1.1774632868989419,
    "2017-01-12": 1.1959884479528209,
    "2017-01-13": 1.226805677986819,
    "2017-01-16": 1.226805677986819,
    "2017-01-17": 1.2572624295762747,
    "2017-01-18": 1.2826228120458691,
    "2017-01-19": 1.2749852938289232,
    "2017-01-20": 1.2663706197587534,
    "2017-01-23": 1.2446226870644161,
    "2017-01-24": 1.2054620777016065,
    "2017-01-25": 1.1672311853043709,
    "2017-01-26": 1.1317065974890157,
    "2017-01-27": 1.0920905593524024,
    "2017-01-30": 1.065720620896452,
    "2017-01-31": 1.0561780696946166,
    "2017-02-01": 1.0502199615535133,
    "2017-02-02": 1.0502199615535133,
    "2017-02-03": 1.0502199615535133,
    "2017-02-06": 1.050097021508148,
    "2017-02-07": 1.05,
    "2017-02-08": 1.05,
    "2017-02-09": 1.05,
    "2017-02-10": 1.05,
    "2017-02-13": 1.05,
    "2017-02-14": 1.05,
    "2017-02-15": 1.05,
    "2017-02-16": 1.05,
    "2017-02-17": 1.0636247381908426,
    "2017-02-20": 1.0636247381908426,
    "2017-02-21": 1.0741421062956196,
    "2017-02-22": 1.1021605170006157,
    "2017-02-23": 1.148068613418581,
    "2017-02-24": 1.1948905910253593,
    "2017-02-27": 1.2140621846137951,
    "2017-02-28": 1.2381940426937788,
    "2017-03-01": 1.2101756319887829,
    "2017-03-02": 1.1642675355708176,
    "2017-03-03": 1.117445557964039,
    "2017-03-06": 1.084649226184761,
    "2017-03-07": 1.05,
    "2017-03-08": 1.05,
    "2017-03-09": 1.0564772222883423,
    "2017-03-10": 1.063925116586811,
    "2017-03-13": 1.0678352172442256,
    "2017-03-14": 1.068540314943655,
    "2017-03-15": 1.068540314943655,
    "2017-03-16": 1.0620630926553125,
    "2017-03-17": 1.0546151983568441,
    "2017-03-20": 1.0507050976994292,
    "2017-03-21": 1.05,
    "2017-03-22": 1.05,
    "2017-03-23": 1.05,
    "2017-03-24": 1.05,
    "2017-03-27": 1.05,
    "2017-03-28": 1.05,
    "2017-03-29": 1.05,
    "2017-03-30": 1.0715347024736885,
    "2017-03-31": 1.0952747904275026,
    "2017-04-03": 1.1129148348778468,
    "2017-04-04": 1.1438797395743725,
    "2017-04-05": 1.1729368433687535,
    "2017-04-06": 1.2125112412806431,
    "2017-04-07": 1.2626383767681706,
    "2017-04-10": 1.3184104912854682,
    "2017-04-11": 1.3615054163471185,
    "2017-04-12": 1.4168668301338923,
    "2017-04-13": 1.4685768362052916,
    "2017-04-14": 1.4685768362052916,
    "2017-04-17": 1.489047386595825,
    "2017-04-18": 1.491138433522028,
    "2017-04-19": 1.4857274325416636,
    "2017-04-20": 1.4616053183137399,
    "2017-04-21": 1.4271282842796782,
    "2017-04-24": 1.354149013892571,
    "2017-04-25": 1.2821549519155557,
    "2017-04-26": 1.2137551867228218,
    "2017-04-27": 1.1558506147945102,
    "2017-04-28": 1.0822397373354034,
    "2017-05-01": 1.0608812338906357,
    "2017-05-02": 1.0573720899738062,
    "2017-05-03": 1.0594844827946104,
    "2017-05-04": 1.0570926513696912,
    "2017-05-05": 1.0523614564058819,
    "2017-05-08": 1.052361456405882,
    "2017-05-09": 1.052361456405882,
    "2017-05-10": 1.05,
    "2017-05-11": 1.05,
    "2017-05-12": 1.05,
    "2017-05-15": 1.05,
    "2017-05-16": 1.05,
    "2017-05-17": 1.05,
    "2017-05-18": 1.05,
    "2017-05-19": 1.05,
    "2017-05-22": 1.05,
    "2017-05-23": 1.05,
    "2017-05-24": 1.05,
    "2017-05-25": 1.05,
    "2017-05-26": 1.05,
    "2017-05-29": 1.05,
    "2017-05-30": 1.05,
    "2017-05-31": 1.05,
    "2017-06-01": 1.05,
    "2017-06-02": 1.05,
    "2017-06-05": 1.05,
    "2017-06-06": 1.05,
    "2017-06-07": 1.05,
    "2017-06-08": 1.05,
    "2017-06-09": 1.05,
    "2017-06-12": 1.05,
    "2017-06-13": 1.05,
    "2017-06-14": 1.05,
    "2017-06-15": 1.05,
    "2017-06-16": 1.071187270828781,
    "2017-06-19": 1.0867112242230828,
    "2017-06-20": 1.0963032154089627,
    "2017-06-21": 1.1069525782571916,
    "2017-06-22": 1.1242193755655892,
    "2017-06-23": 1.1215661057666038,
    "2017-06-26": 1.1196685373832544,
    "2017-06-27": 1.1100765461973743,
    "2017-06-28": 1.100319566694686,
    "2017-06-29": 1.0849859841429805,
    "2017-06-30": 1.0671902719139277,
    "2017-07-03": 1.06912016858734,
    "2017-07-04": 1.06912016858734,
    "2017-07-05": 1.0884535714639225,
    "2017-07-06": 1.1165655821752405,
    "2017-07-07": 1.1403409812936833,
    "2017-07-10": 1.1486996484983765,
    "2017-07-11": 1.1407667705689053,
    "2017-07-12": 1.121433367692323,
    "2017-07-13": 1.0924289736354644,
    "2017-07-14": 1.082584307627196,
    "2017-07-17": 1.084417478024204,
    "2017-07-18": 1.0767940742693105,
    "2017-07-19": 1.0767940742693105,
    "2017-07-20": 1.0767940742693105,
    "2017-07-21": 1.0609301264024438,
    "2017-07-24": 1.05,
    "2017-07-25": 1.05,
    "2017-07-26": 1.05,
    "2017-07-27": 1.05,
    "2017-07-28": 1.05,
    "2017-07-31": 1.05,
    "2017-08-01": 1.05,
    "2017-08-02": 1.05,
    "2017-08-03": 1.05,
    "2017-08-04": 1.05,
    "2017-08-07": 1.05,
    "2017-08-08": 1.05,
    "2017-08-09": 1.05,
    "2017-08-10": 1.05,
    "2017-08-11": 1.05,
    "2017-08-14": 1.05,
    "2017-08-15": 1.05,
    "2017-08-16": 1.05,
    "2017-08-17": 1.05,
    "2017-08-18": 1.05,
    "2017-08-21": 1.05,
    "2017-08-22": 1.05,
    "2017-08-23": 1.051577777253992,
    "2017-08-24": 1.0584115097489113,
    "2017-08-25": 1.0615364728690357,
    "2017-08-28": 1.0615364728690355,
    "2017-08-29": 1.068403508944225,
    "2017-08-30": 1.0809266589955346,
    "2017-08-31": 1.086512235160059,
    "2017-09-01": 1.0833872720399345,
    "2017-09-04": 1.0833872720399345,
    "2017-09-05": 1.0833872720399345,
    "2017-09-06": 1.0765202359647452,
    "2017-09-07": 1.0624193086594436,
    "2017-09-08": 1.05,
    "2017-09-11": 1.05,
    "2017-09-12": 1.05,
    "2017-09-13": 1.05,
    "2017-09-14": 1.05,
    "2017-09-15": 1.05,
    "2017-09-18": 1.05,
    "2017-09-19": 1.05,
    "2017-09-20": 1.05,
    "2017-09-21": 1.05,
    "2017-09-22": 1.05,
    "2017-09-25": 1.05,
    "2017-09-26": 1.05,
    "2017-09-27": 1.05,
    "2017-09-28": 1.05,
    "2017-09-29": 1.05,
    "2017-10-02": 1.05,
    "2017-10-03": 1.05,
    "2017-10-04": 1.0671459360297952,
    "2017-10-05": 1.0928281566203535,
    "2017-10-06": 1.1438162252781516,
    "2017-10-09": 1.1916930910888062,
    "2017-10-10": 1.2525027204899262,
    "2017-10-11": 1.3044073063991561,
    "2017-10-12": 1.3365132196746776,
    "2017-10-13": 1.3160842803736632,
    "2017-10-16": 1.29457904509029,
    "2017-10-17": 1.2544429487899624,
    "2017-10-18": 1.202145823177827,
    "2017-10-19": 1.1702160750057766,
    "2017-10-20": 1.154388967523401,
    "2017-10-23": 1.1453985303200387,
    "2017-10-24": 1.1632922915542019,
    "2017-10-25": 1.1852342313541457,
    "2017-10-26": 1.2471040387475583,
    "2017-10-27": 1.2961943724378404,
    "2017-10-30": 1.2968475905688308,
    "2017-10-31": 1.2743850559334713,
    "2017-11-01": 1.2465856311382815,
    "2017-11-02": 1.1588574380508398,
    "2017-11-03": 1.0950350824861494,
    "2017-11-06": 1.0770006710312399,
    "2017-11-07": 1.0608959113316438,
    "2017-11-08": 1.05,
    "2017-11-09": 1.05,
    "2017-11-10": 1.05,
    "2017-11-13": 1.05,
    "2017-11-14": 1.05,
    "2017-11-15": 1.05,
    "2017-11-16": 1.05,
    "2017-11-17": 1.05,
    "2017-11-20": 1.0503013477472707,
    "2017-11-21": 1.0503013477472707,
    "2017-11-22": 1.0503013477472707,
    "2017-11-23": 1.0503013477472707,
    "2017-11-24": 1.0503013477472705,
    "2017-11-27": 1.0503013477472707,
    "2017-11-28": 1.05,
    "2017-11-29": 1.0567776683438774,
    "2017-11-30": 1.0581887605426723,
    "2017-12-01": 1.0581887605426723,
    "2017-12-04": 1.058188760542672,
    "2017-12-05": 1.058188760542672,
    "2017-12-06": 1.0514110921987947,
    "2017-12-07": 1.05,
    "2017-12-08": 1.05,
    "2017-12-11": 1.05,
    "2017-12-12": 1.05,
    "2017-12-13": 1.05,
    "2017-12-14": 1.05,
    "2017-12-15": 1.05,
    "2017-12-18": 1.05,
    "2017-12-19": 1.05,
    "2017-12-20": 1.05,
    "2017-12-21": 1.05,
    "2017-12-22": 1.05,
    "2017-12-25": 1.05,
    "2017-12-26": 1.05,
    "2017-12-27": 1.05,
    "2017-12-28": 1.05,
    "2017-12-29": 1.05,
    "2018-01-01": 1.05,
    "2018-01-02": 1.05,
    "2018-01-03": 1.05,
    "2018-01-04": 1.05,
    "2018-01-05": 1.05,
    "2018-01-08": 1.05,
    "2018-01-09": 1.05,
    "2018-01-10": 1.05,
    "2018-01-11": 1.05,
    "2018-01-12": 1.05,
    "2018-01-15": 1.05,
    "2018-01-16": 1.05,
    "2018-01-17": 1.05,
    "2018-01-18": 1.05,
    "2018-01-19": 1.05,
    "2018-01-22": 1.0665124355278668,
    "2018-01-23": 1.072346486524196,
    "2018-01-24": 1.084665460497407,
    "2018-01-25": 1.085121439300043,
    "2018-01-26": 1.0921905694444927,
    "2018-01-29": 1.1051138104384814,
    "2018-01-30": 1.133175880205569,
    "2018-01-31": 1.164005914550228,
    "2018-02-01": 1.2142209238651902,
    "2018-02-02": 1.2471278021516539,
    "2018-02-05": 1.2255667351920478,
    "2018-02-06": 1.1926391786131811,
    "2018-02-07": 1.1560801798377305,
    "2018-02-08": 1.1275777201902353,
    "2018-02-09": 1.1048328953929236,
    "2018-02-12": 1.1248666305995647,
    "2018-02-13": 1.1447659600824551,
    "2018-02-14": 1.150420492714129,
    "2018-02-15": 1.128251964244026,
    "2018-02-16": 1.1110207806104246,
    "2018-02-19": 1.1110207806104246,
    "2018-02-20": 1.083112435841534,
    "2018-02-21": 1.0622445421740934,
    "2018-02-22": 1.05,
    "2018-02-23": 1.05,
    "2018-02-26": 1.05,
    "2018-02-27": 1.05,
    "2018-02-28": 1.05,
    "2018-03-01": 1.05,
    "2018-03-02": 1.05,
    "2018-03-05": 1.05,
    "2018-03-06": 1.05,
    "2018-03-07": 1.05,
    "2018-03-08": 1.05,
    "2018-03-09": 1.05,
    "2018-03-12": 1.05,
    "2018-03-13": 1.05,
    "2018-03-14": 1.05,
    "2018-03-15": 1.05,
    "2018-03-16": 1.0674732231294535,
    "2018-03-19": 1.0907396115253989,
    "2018-03-20": 1.100396707287275,
    "2018-03-21": 1.100396707287275,
    "2018-03-22": 1.100396707287275,
    "2018-03-23": 1.0829234841578217,
    "2018-03-26": 1.0596570957618763,
    "2018-03-27": 1.05,
    "2018-03-28": 1.0564674978125088,
    "2018-03-29": 1.065401443077097,
    "2018-03-30": 1.065401443077097,
    "2018-04-02": 1.0939079011729869,
    "2018-04-03": 1.1123471981841775,
    "2018-04-04": 1.1324227975974632,
    "2018-04-05": 1.126797159535253,
    "2018-04-06": 1.1178632142706648,
    "2018-04-09": 1.089356756174775,
    "2018-04-10": 1.0709174591635844,
    "2018-04-11": 1.0508418597502989,
    "2018-04-12": 1.05,
    "2018-04-13": 1.05,
    "2018-04-16": 1.05,
    "2018-04-17": 1.05,
    "2018-04-18": 1.05,
    "2018-04-19": 1.05,
    "2018-04-20": 1.05,
    "2018-04-23": 1.05,
    "2018-04-24": 1.05,
    "2018-04-25": 1.05,
    "2018-04-26": 1.0534602139981906,
    "2018-04-27": 1.0534602139981906,
    "2018-04-30": 1.0534602139981906,
    "2018-05-01": 1.0534602139981906,
    "2018-05-02": 1.0534602139981906,
    "2018-05-03": 1.05,
    "2018-05-04": 1.05,
    "2018-05-07": 1.064128590089131,
    "2018-05-08": 1.0811852718056645,
    "2018-05-09": 1.0958174684651778,
    "2018-05-10": 1.0958174684651778,
    "2018-05-11": 1.0982104069038139,
    "2018-05-14": 1.0913297034739085,
    "2018-05-15": 1.074273021757375,
    "2018-05-16": 1.0671896436976502,
    "2018-05-17": 1.0801967780526471,
    "2018-05-18": 1.080133413388649,
    "2018-05-21": 1.0792573731888855,
    "2018-05-22": 1.0850703986151422,
    "2018-05-23": 1.077521580015354,
    "2018-05-24": 1.0645144456603572,
    "2018-05-25": 1.0621848718857192,
    "2018-05-28": 1.0621848718857192,
    "2018-05-29": 1.055813025426257,
    "2018-05-30": 1.05,
    "2018-05-31": 1.05,
    "2018-06-01": 1.05,
    "2018-06-04": 1.05,
    "2018-06-05": 1.05,
    "2018-06-06": 1.05,
    "2018-06-07": 1.05,
    "2018-06-08": 1.05,
    "2018-06-11": 1.05,
    "2018-06-12": 1.05,
    "2018-06-13": 1.05,
    "2018-06-14": 1.05,
    "2018-06-15": 1.05,
    "2018-06-18": 1.05,
    "2018-06-19": 1.05,
    "2018-06-20": 1.05,
    "2018-06-21": 1.05,
    "2018-06-22": 1.05,
    "2018-06-25": 1.05,
    "2018-06-26": 1.05,
    "2018-06-27": 1.05,
    "2018-06-28": 1.05,
    "2018-06-29": 1.05,
    "2018-07-02": 1.059425420858123,
    "2018-07-03": 1.0831793864595276,
    "2018-07-04": 1.0831793864595276,
    "2018-07-05": 1.1091965800508778,
    "2018-07-06": 1.1643577023873475,
    "2018-07-09": 1.219412143790794,
    "2018-07-10": 1.263195739598355,
    "2018-07-11": 1.2947220029487803,
    "2018-07-12": 1.3113703665563328,
    "2018-07-13": 1.3019415185487753,
    "2018-07-16": 1.2855132026949427,
    "2018-07-17": 1.267582569185954,
    "2018-07-18": 1.2386352151010045,
    "2018-07-19": 1.2349959215310458,
    "2018-07-20": 1.2265598736735785,
    "2018-07-23": 1.2055370821490996,
    "2018-07-24": 1.1801861552298898,
    "2018-07-25": 1.1583263417027898,
    "2018-07-26": 1.1201326661208306,
    "2018-07-27": 1.1025823241947725,
    "2018-07-30": 1.1084212125773976,
    "2018-07-31": 1.1061073364026779,
    "2018-08-01": 1.110844730634023,
    "2018-08-02": 1.1221783705531991,
    "2018-08-03": 1.102557874957953,
    "2018-08-06": 1.0791156525501933,
    "2018-08-07": 1.0715020724874267,
    "2018-08-08": 1.0622916169163021,
    "2018-08-09": 1.0501253889501407,
    "2018-08-10": 1.05,
    "2018-08-13": 1.05,
    "2018-08-14": 1.05,
    "2018-08-15": 1.05,
    "2018-08-16": 1.05,
    "2018-08-17": 1.05,
    "2018-08-20": 1.05,
    "2018-08-21": 1.05,
    "2018-08-22": 1.05,
    "2018-08-23": 1.05,
    "2018-08-24": 1.05,
    "2018-08-27": 1.05,
    "2018-08-28": 1.05,
    "2018-08-29": 1.05,
    "2018-08-30": 1.0520367243235484,
    "2018-08-31": 1.055983572869433,
    "2018-09-03": 1.055983572869433,
    "2018-09-04": 1.0559835728694331,
    "2018-09-05": 1.055983572869433,
    "2018-09-06": 1.055983572869433,
    "2018-09-07": 1.0539468485458845,
    "2018-09-10": 1.05,
    "2018-09-11": 1.05,
    "2018-09-12": 1.05,
    "2018-09-13": 1.05,
    "2018-09-14": 1.0590086992732957,
    "2018-09-17": 1.0662075014543162,
    "2018-09-18": 1.0679289481465783,
    "2018-09-19": 1.0966484921863449,
    "2018-09-20": 1.1262980095031987,
    "2018-09-21": 1.1498566124082643,
    "2018-09-24": 1.174442156559144,
    "2018-09-25": 1.2091925653382058,
    "2018-09-26": 1.185818926325289,
    "2018-09-27": 1.1572503724095706,
    "2018-09-28": 1.1278420602452306,
    "2018-10-01": 1.0969810665945556,
    "2018-10-02": 1.0605092111232317,
    "2018-10-03": 1.0551633060963823,
    "2018-10-04": 1.0540823426952468,
    "2018-10-05": 1.0540038936849612,
    "2018-10-08": 1.0683163792575083,
    "2018-10-09": 1.0702315284911477,
    "2018-10-10": 1.0852427420817865,
    "2018-10-11": 1.0852427420817867,
    "2018-10-12": 1.082162201078051,
    "2018-10-15": 1.066926362824278,
    "2018-10-16": 1.065011213590639,
    "2018-10-17": 1.05,
    "2018-10-18": 1.05,
    "2018-10-19": 1.05,
    "2018-10-22": 1.05,
    "2018-10-23": 1.05,
    "2018-10-24": 1.05,
    "2018-10-25": 1.05,
    "2018-10-26": 1.05,
    "2018-10-29": 1.05,
    "2018-10-30": 1.05,
    "2018-10-31": 1.05,
    "2018-11-01": 1.0587883005704892,
    "2018-11-02": 1.062052986791501,
    "2018-11-05": 1.062568097859791,
    "2018-11-06": 1.0625680978597911,
    "2018-11-07": 1.0625680978597911,
    "2018-11-08": 1.053779797289302,
    "2018-11-09": 1.0606744102287426,
    "2018-11-12": 1.0703309066079878,
    "2018-11-13": 1.0703309066079878,
    "2018-11-14": 1.072024537567861,
    "2018-11-15": 1.08634818646309,
    "2018-11-16": 1.0981211023841546,
    "2018-11-19": 1.1100021501413868,
    "2018-11-20": 1.1322405553795194,
    "2018-11-21": 1.1467616035372858,
    "2018-11-22": 1.1467616035372858,
    "2018-11-23": 1.1555070498504914,
    "2018-11-26": 1.1335748347689742,
    "2018-11-27": 1.1270109872484693,
    "2018-11-28": 1.118449429336264,
    "2018-11-29": 1.121053039588461,
    "2018-11-30": 1.1465093846337546,
    "2018-12-03": 1.1833475687596013,
    "2018-12-04": 1.261215064586613,
    "2018-12-05": 1.261215064586613,
    "2018-12-06": 1.3253520434149066,
    "2018-12-07": 1.4330145161296048,
    "2018-12-10": 1.503955058566581,
    "2018-12-11": 1.575887752754054,
    "2018-12-12": 1.5762746737195825,
    "2018-12-13": 1.5764603484622466,
    "2018-12-14": 1.5395660335922634,
    "2018-12-17": 1.5018213152133122,
    "2018-12-18": 1.4927414595979305,
    "2018-12-19": 1.4941983394939793,
    "2018-12-20": 1.5028936760816483,
    "2018-12-21": 1.511589889897201,
    "2018-12-24": 1.5481844081570384,
    "2018-12-25": 1.5481844081570384,
    "2018-12-26": 1.517977369297544,
    "2018-12-27": 1.506610571443853,
    "2018-12-28": 1.495955254025593,
    "2018-12-31": 1.4645053533678776,
    "2019-01-01": 1.4645053533678776,
    "2019-01-02": 1.3974030428357693,
    "2019-01-03": 1.3480622397703332,
    "2019-01-04": 1.2642289332511727,
    "2019-01-07": 1.1881894131848787,
    "2019-01-08": 1.1213566528124894,
    "2019-01-09": 1.0701431807730075,
    "2019-01-10": 1.05,
    "2019-01-11": 1.05,
    "2019-01-14": 1.05,
    "2019-01-15": 1.05,
    "2019-01-16": 1.05,
    "2019-01-17": 1.05,
    "2019-01-18": 1.05,
    "2019-01-21": 1.05,
    "2019-01-22": 1.05,
    "2019-01-23": 1.05,
    "2019-01-24": 1.05,
    "2019-01-25": 1.05,
    "2019-01-28": 1.05,
    "2019-01-29": 1.05,
    "2019-01-30": 1.05,
    "2019-01-31": 1.05,
    "2019-02-01": 1.05,
    "2019-02-04": 1.05,
    "2019-02-05": 1.05,
    "2019-02-06": 1.05,
    "2019-02-07": 1.05,
    "2019-02-08": 1.05,
    "2019-02-11": 1.05,
    "2019-02-12": 1.05,
    "2019-02-13": 1.05,
    "2019-02-14": 1.05,
    "2019-02-15": 1.05,
    "2019-02-18": 1.05,
    "2019-02-19": 1.05,
    "2019-02-20": 1.05,
    "2019-02-21": 1.05,
    "2019-02-22": 1.05,
    "2019-02-25": 1.05,
    "2019-02-26": 1.05,
    "2019-02-27": 1.05,
    "2019-02-28": 1.05,
    "2019-03-01": 1.05,
    "2019-03-04": 1.05,
    "2019-03-05": 1.05,
    "2019-03-06": 1.05,
    "2019-03-07": 1.05,
    "2019-03-08": 1.05,
    "2019-03-11": 1.05,
    "2019-03-12": 1.05,
    "2019-03-13": 1.05,
    "2019-03-14": 1.05,
    "2019-03-15": 1.05,
    "2019-03-18": 1.05,
    "2019-03-19": 1.05,
    "2019-03-20": 1.05,
    "2019-03-21": 1.05,
    "2019-03-22": 1.05,
    "2019-03-25": 1.05,
    "2019-03-26": 1.05,
    "2019-03-27": 1.05,
    "2019-03-28": 1.0630431366726327,
    "2019-03-29": 1.0686053759206395,
    "2019-04-01": 1.0686053759206398,
    "2019-04-02": 1.0686053759206398,
    "2019-04-03": 1.0686053759206395,
    "2019-04-04": 1.0555622392480069,
    "2019-04-05": 1.05,
    "2019-04-08": 1.05,
    "2019-04-09": 1.05,
    "2019-04-10": 1.05,
    "2019-04-11": 1.05,
    "2019-04-12": 1.05,
    "2019-04-15": 1.05,
    "2019-04-16": 1.05,
    "2019-04-17": 1.05,
    "2019-04-18": 1.05,
    "2019-04-19": 1.05,
    "2019-04-22": 1.05,
    "2019-04-23": 1.05,
    "2019-04-24": 1.05,
    "2019-04-25": 1.05,
    "2019-04-26": 1.05,
    "2019-04-29": 1.05,
    "2019-04-30": 1.05,
    "2019-05-01": 1.0506332068419348,
    "2019-05-02": 1.0506332068419348,
    "2019-05-03": 1.0506332068419348,
    "2019-05-06": 1.050633206841935,
    "2019-05-07": 1.0513410135324706,
    "2019-05-08": 1.0507078066905358,
    "2019-05-09": 1.0549479307995822,
    "2019-05-10": 1.0565238269894512,
    "2019-05-13": 1.0683627382818461,
    "2019-05-14": 1.0888333116631448,
    "2019-05-15": 1.1119069004049593,
    "2019-05-16": 1.1357347725531273,
    "2019-05-17": 1.162687789752364,
    "2019-05-20": 1.1750258288719588,
    "2019-05-21": 1.1801388059099351,
    "2019-05-22": 1.1785377785588658,
    "2019-05-23": 1.1622937834640923,
    "2019-05-24": 1.1393516260236907,
    "2019-05-27": 1.1393516260236907,
    "2019-05-28": 1.1179521491747964,
    "2019-05-29": 1.105807519513487,
    "2019-05-30": 1.094093399304204,
    "2019-05-31": 1.1103653858871827,
    "2019-06-03": 1.144522912583425,
    "2019-06-04": 1.1724883963038892,
    "2019-06-05": 1.1952299817219298,
    "2019-06-06": 1.212492189748476,
    "2019-06-07": 1.2042111034271588,
    "2019-06-10": 1.1644668207822122,
    "2019-06-11": 1.1337238634986526,
    "2019-06-12": 1.0998753812063577,
    "2019-06-13": 1.0998963161312274,
    "2019-06-14": 1.1244700804693664,
    "2019-06-17": 1.1626440799290085,
    "2019-06-18": 1.199971368542964,
    "2019-06-19": 1.2224867735213671,
    "2019-06-20": 1.2275857766683993,
    "2019-06-21": 1.196910200743442,
    "2019-06-24": 1.1904657589699348,
    "2019-06-25": 1.1934157850915907,
    "2019-06-26": 1.1949188865644849,
    "2019-06-27": 1.175227320956633,
    "2019-06-28": 1.1754743151883047,
    "2019-07-01": 1.1609309460872281,
    "2019-07-02": 1.1463261589160032,
    "2019-07-03": 1.1518126581702615,
    "2019-07-04": 1.1518126581702615,
    "2019-07-05": 1.139363636498203,
    "2019-07-08": 1.1254035524292474,
    "2019-07-09": 1.1082173638441892,
    "2019-07-10": 1.0825448362798022,
    "2019-07-11": 1.05,
    "2019-07-12": 1.05,
    "2019-07-15": 1.05,
    "2019-07-16": 1.05,
    "2019-07-17": 1.05,
    "2019-07-18": 1.05,
    "2019-07-19": 1.05,
    "2019-07-22": 1.05,
    "2019-07-23": 1.05,
    "2019-07-24": 1.05,
    "2019-07-25": 1.05,
    "2019-07-26": 1.05,
    "2019-07-29": 1.05,
    "2019-07-30": 1.05,
    "2019-07-31": 1.05,
    "2019-08-01": 1.05,
    "2019-08-02": 1.05,
    "2019-08-05": 1.0532408420345891,
    "2019-08-06": 1.0532408420345891,
    "2019-08-07": 1.0600285363825157,
    "2019-08-08": 1.0608544797014439,
    "2019-08-09": 1.0747736844477813,
    "2019-08-12": 1.0781573704467244,
    "2019-08-13": 1.0781573704467244,
    "2019-08-14": 1.0771709808200618,
    "2019-08-15": 1.09177290844959,
    "2019-08-16": 1.0880040244240758,
    "2019-08-19": 1.0813794963905436,
    "2019-08-20": 1.0813794963905436,
    "2019-08-21": 1.07557819166928,
    "2019-08-22": 1.0601503207208236,
    "2019-08-23": 1.05,
    "2019-08-26": 1.05,
    "2019-08-27": 1.05,
    "2019-08-28": 1.05,
    "2019-08-29": 1.05,
    "2019-08-30": 1.05,
    "2019-09-02": 1.05,
    "2019-09-03": 1.05,
    "2019-09-04": 1.0564211029160022,
    "2019-09-05": 1.0564211029160022,
    "2019-09-06": 1.0564211029160024,
    "2019-09-09": 1.0564211029160022,
    "2019-09-10": 1.0564211029160022,
    "2019-09-11": 1.05,
    "2019-09-12": 1.05,
    "2019-09-13": 1.05,
    "2019-09-16": 1.05,
    "2019-09-17": 1.05,
    "2019-09-18": 1.05,
    "2019-09-19": 1.05,
    "2019-09-20": 1.05,
    "2019-09-23": 1.05,
    "2019-09-24": 1.05,
    "2019-09-25": 1.05,
    "2019-09-26": 1.05,
    "2019-09-27": 1.05,
    "2019-09-30": 1.05,
    "2019-10-01": 1.05,
    "2019-10-02": 1.05,
    "2019-10-03": 1.05,
    "2019-10-04": 1.05,
    "2019-10-07": 1.05,
    "2019-10-08": 1.05,
    "2019-10-09": 1.05,
    "2019-10-10": 1.05,
    "2019-10-11": 1.05,
    "2019-10-14": 1.05,
    "2019-10-15": 1.05,
    "2019-10-16": 1.05,
    "2019-10-17": 1.05,
    "2019-10-18": 1.05,
    "2019-10-21": 1.05,
    "2019-10-22": 1.05,
    "2019-10-23": 1.05,
    "2019-10-24": 1.054546749676034,
    "2019-10-25": 1.0583610209888847,
    "2019-10-28": 1.0583610209888847,
    "2019-10-29": 1.0583610209888847,
    "2019-10-30": 1.0583610209888845,
    "2019-10-31": 1.0538142713128509,
    "2019-11-01": 1.05,
    "2019-11-04": 1.05,
    "2019-11-05": 1.05,
    "2019-11-06": 1.05,
    "2019-11-07": 1.05,
    "2019-11-08": 1.05,
    "2019-11-11": 1.05,
    "2019-11-12": 1.05,
    "2019-11-13": 1.05,
    "2019-11-14": 1.05,
    "2019-11-15": 1.05,
    "2019-11-18": 1.05,
    "2019-11-19": 1.05,
    "2019-11-20": 1.05,
    "2019-11-21": 1.05,
    "2019-11-22": 1.05,
    "2019-11-25": 1.05,
    "2019-11-26": 1.05,
    "2019-11-27": 1.05,
    "2019-11-28": 1.05,
    "2019-11-29": 1.05,
    "2019-12-02": 1.05,
    "2019-12-03": 1.05,
    "2019-12-04": 1.05,
    "2019-12-05": 1.05,
    "2019-12-06": 1.05,
    "2019-12-09": 1.05,
    "2019-12-10": 1.05,
    "2019-12-11": 1.05,
    "2019-12-12": 1.05,
    "2019-12-13": 1.05,
    "2019-12-16": 1.05,
    "2019-12-17": 1.05,
    "2019-12-18": 1.05,
    "2019-12-19": 1.05,
    "2019-12-20": 1.05,
    "2019-12-23": 1.05,
    "2019-12-24": 1.05,
    "2019-12-25": 1.05,
    "2019-12-26": 1.05,
    "2019-12-27": 1.05,
    "2019-12-30": 1.05,
    "2019-12-31": 1.05,
    "2020-01-01": 1.05,
    "2020-01-02": 1.05,
    "2020-01-03": 1.05,
    "2020-01-06": 1.05,
    "2020-01-07": 1.05,
    "2020-01-08": 1.05,
    "2020-01-09": 1.05,
    "2020-01-10": 1.05,
    "2020-01-13": 1.05,
    "2020-01-14": 1.05,
    "2020-01-15": 1.05,
    "2020-01-16": 1.05,
    "2020-01-17": 1.05,
    "2020-01-20": 1.05,
    "2020-01-21": 1.05,
    "2020-01-22": 1.05,
    "2020-01-23": 1.05,
    "2020-01-24": 1.05,
    "2020-01-27": 1.0563411785180044,
    "2020-01-28": 1.0563411785180044,
    "2020-01-29": 1.0563411785180041,
    "2020-01-30": 1.0681246960524162,
    "2020-01-31": 1.0905685943087033,
    "2020-02-03": 1.1062343703588362,
    "2020-02-04": 1.1062343703588362,
    "2020-02-05": 1.1062343703588362,
    "2020-02-06": 1.0944508528244241,
    "2020-02-07": 1.0720069545681372,
    "2020-02-10": 1.05,
    "2020-02-11": 1.05,
    "2020-02-12": 1.05,
    "2020-02-13": 1.05,
    "2020-02-14": 1.05,
    "2020-02-17": 1.05,
    "2020-02-18": 1.05,
    "2020-02-19": 1.05,
    "2020-02-20": 1.05,
    "2020-02-21": 1.0581005893180961,
    "2020-02-24": 1.0841903745430845,
    "2020-02-25": 1.1139488575023224,
    "2020-02-26": 1.1537304577523124,
    "2020-02-27": 1.2190183664850898,
    "2020-02-28": 1.2564579393367072,
    "2020-03-02": 1.2645802721746557,
    "2020-03-03": 1.2590809411975001,
    "2020-03-04": 1.2192993409475101,
    "2020-03-05": 1.1890939308551087,
    "2020-03-06": 1.1903855806513708,
    "2020-03-09": 1.2405186844903044,
    "2020-03-10": 1.216259532508222,
    "2020-03-11": 1.216259532508222,
    "2020-03-12": 1.181177033867846,
    "2020-03-13": 1.1343452219018704,
    "2020-03-16": 1.05,
    "2020-03-17": 1.05,
    "2020-03-18": 1.05,
    "2020-03-19": 1.05,
    "2020-03-20": 1.05,
    "2020-03-23": 1.05,
    "2020-03-24": 1.05,
    "2020-03-25": 1.05,
    "2020-03-26": 1.05,
    "2020-03-27": 1.05,
    "2020-03-30": 1.05,
    "2020-03-31": 1.05,
    "2020-04-01": 1.05,
    "2020-04-02": 1.05,
    "2020-04-03": 1.05,
    "2020-04-06": 1.05,
    "2020-04-07": 1.05,
    "2020-04-08": 1.05,
    "2020-04-09": 1.05,
    "2020-04-10": 1.05,
    "2020-04-13": 1.05,
    "2020-04-14": 1.05,
    "2020-04-15": 1.05,
    "2020-04-16": 1.05,
    "2020-04-17": 1.05,
    "2020-04-20": 1.05,
    "2020-04-21": 1.05,
    "2020-04-22": 1.05,
    "2020-04-23": 1.05,
    "2020-04-24": 1.05,
    "2020-04-27": 1.05,
    "2020-04-28": 1.05,
    "2020-04-29": 1.05,
    "2020-04-30": 1.05,
    "2020-05-01": 1.05,
    "2020-05-04": 1.05,
    "2020-05-05": 1.05,
    "2020-05-06": 1.05,
    "2020-05-07": 1.05,
    "2020-05-08": 1.05,
    "2020-05-11": 1.05,
    "2020-05-12": 1.05,
    "2020-05-13": 1.05,
    "2020-05-14": 1.05,
    "2020-05-15": 1.05,
    "2020-05-18": 1.05,
    "2020-05-19": 1.05,
    "2020-05-20": 1.05,
    "2020-05-21": 1.05,
    "2020-05-22": 1.05,
    "2020-05-25": 1.05,
    "2020-05-26": 1.05,
    "2020-05-27": 1.05,
    "2020-05-28": 1.05,
    "2020-05-29": 1.05,
    "2020-06-01": 1.05,
    "2020-06-02": 1.05,
    "2020-06-03": 1.05,
    "2020-06-04": 1.05,
    "2020-06-05": 1.05,
    "2020-06-08": 1.05,
    "2020-06-09": 1.05,
    "2020-06-10": 1.05,
    "2020-06-11": 1.05,
    "2020-06-12": 1.05,
    "2020-06-15": 1.05,
    "2020-06-16": 1.05,
    "2020-06-17": 1.05,
    "2020-06-18": 1.05,
    "2020-06-19": 1.05,
    "2020-06-22": 1.05,
    "2020-06-23": 1.05,
    "2020-06-24": 1.05,
    "2020-06-25": 1.05,
    "2020-06-26": 1.05,
    "2020-06-29": 1.05,
    "2020-06-30": 1.05,
    "2020-07-01": 1.05,
    "2020-07-02": 1.05,
    "2020-07-03": 1.05,
    "2020-07-06": 1.05,
    "2020-07-07": 1.05,
    "2020-07-08": 1.05,
    "2020-07-09": 1.05,
    "2020-07-10": 1.05,
    "2020-07-13": 1.0667384715477586,
    "2020-07-14": 1.1045854209221677,
    "2020-07-15": 1.1216557826013065,
    "2020-07-16": 1.1692391890705232,
    "2020-07-17": 1.2103662441565075,
    "2020-07-20": 1.2570113456540633,
    "2020-07-21": 1.279878527322541,
    "2020-07-22": 1.3181465595772408,
    "2020-07-23": 1.3185765959497389,
    "2020-07-24": 1.320232794549746,
    "2020-07-27": 1.2932465225858638,
    "2020-07-28": 1.2691391655053448,
    "2020-07-29": 1.2526520252814382,
    "2020-07-30": 1.2284963916561746,
    "2020-07-31": 1.2343600477569576,
    "2020-08-03": 1.2385690569992733,
    "2020-08-04": 1.2327116409116408,
    "2020-08-05": 1.2071370605849066,
    "2020-08-06": 1.2017889340292165,
    "2020-08-07": 1.1771060937919426,
    "2020-08-10": 1.1737632917735383,
    "2020-08-11": 1.143013933898803,
    "2020-08-12": 1.1297372605156049,
    "2020-08-13": 1.111227577854844,
    "2020-08-14": 1.0872635083053437,
    "2020-08-17": 1.05,
    "2020-08-18": 1.05,
    "2020-08-19": 1.05,
    "2020-08-20": 1.05,
    "2020-08-21": 1.05,
    "2020-08-24": 1.05,
    "2020-08-25": 1.05,
    "2020-08-26": 1.05,
    "2020-08-27": 1.05,
    "2020-08-28": 1.05,
    "2020-08-31": 1.05,
    "2020-09-01": 1.05,
    "2020-09-02": 1.05,
    "2020-09-03": 1.05,
    "2020-09-04": 1.05,
    "2020-09-07": 1.05,
    "2020-09-08": 1.05,
    "2020-09-09": 1.05,
    "2020-09-10": 1.05,
    "2020-09-11": 1.05,
    "2020-09-14": 1.05,
    "2020-09-15": 1.05,
    "2020-09-16": 1.05,
    "2020-09-17": 1.05,
    "2020-09-18": 1.05,
    "2020-09-21": 1.05,
    "2020-09-22": 1.05,
    "2020-09-23": 1.05,
    "2020-09-24": 1.05,
    "2020-09-25": 1.05,
    "2020-09-28": 1.05,
    "2020-09-29": 1.05,
    "2020-09-30": 1.05,
    "2020-10-01": 1.05,
    "2020-10-02": 1.05,
    "2020-10-05": 1.05,
    "2020-10-06": 1.1673211913480592,
    "2020-10-07": 1.2800293285620268,
    "2020-10-08": 1.3891442448698068,
    "2020-10-09": 1.5063178286869379,
    "2020-10-12": 1.6307571076862128,
    "2020-10-13": 1.58593567753674,
    "2020-10-14": 1.5348601344462978,
    "2020-10-15": 1.4979942600912715,
    "2020-10-16": 1.4603943307652014,
    "2020-10-19": 1.4334935348009563,
    "2020-10-20": 1.4521277365958927,
    "2020-10-21": 1.4752721867600125,
    "2020-10-22": 1.484629714596423,
    "2020-10-23": 1.4815460621118848,
    "2020-10-26": 1.4428141951097815,
    "2020-10-27": 1.3986065463346766,
    "2020-10-28": 1.36200204735499,
    "2020-10-29": 1.3342934355675622,
    "2020-10-30": 1.3126394766800886,
    "2020-11-02": 1.313865769688826,
    "2020-11-03": 1.3555656285617554,
    "2020-11-04": 1.3073930832537966,
    "2020-11-05": 1.25349512525206,
    "2020-11-06": 1.1986590821330114,
    "2020-11-09": 1.1386261730913472,
    "2020-11-10": 1.05,
    "2020-11-11": 1.05,
    "2020-11-12": 1.05,
    "2020-11-13": 1.05,
    "2020-11-16": 1.05,
    "2020-11-17": 1.05,
    "2020-11-18": 1.05,
    "2020-11-19": 1.05,
    "2020-11-20": 1.05,
    "2020-11-23": 1.05,
    "2020-11-24": 1.05,
    "2020-11-25": 1.05,
    "2020-11-26": 1.05,
    "2020-11-27": 1.05,
    "2020-11-30": 1.05,
    "2020-12-01": 1.05,
    "2020-12-02": 1.05,
    "2020-12-03": 1.05,
    "2020-12-04": 1.05,
    "2020-12-07": 1.05,
    "2020-12-08": 1.05,
    "2020-12-09": 1.05,
    "2020-12-10": 1.05,
    "2020-12-11": 1.05,
    "2020-12-14": 1.05,
    "2020-12-15": 1.05,
    "2020-12-16": 1.05,
    "2020-12-17": 1.05,
    "2020-12-18": 1.05,
    "2020-12-21": 1.05,
    "2020-12-22": 1.05,
    "2020-12-23": 1.05,
    "2020-12-24": 1.05,
    "2020-12-25": 1.05,
    "2020-12-28": 1.05,
    "2020-12-29": 1.05,
    "2020-12-30": 1.05,
    "2020-12-31": 1.0958664219969527,
    "2021-01-01": 1.0958664219969527,
    "2021-01-04": 1.1474959520332644,
    "2021-01-05": 1.1887169664487989,
    "2021-01-06": 1.1887169664487989,
    "2021-01-07": 1.1887169664487989,
    "2021-01-08": 1.142850544451846,
    "2021-01-11": 1.096869642043534,
    "2021-01-12": 1.086536904540347,
    "2021-01-13": 1.086536904540347,
    "2021-01-14": 1.0865369045403468,
    "2021-01-15": 1.086536904540347,
    "2021-01-18": 1.086536904540347,
    "2021-01-19": 1.0808882769123476,
    "2021-01-20": 1.05,
    "2021-01-21": 1.05,
    "2021-01-22": 1.05,
    "2021-01-25": 1.05,
    "2021-01-26": 1.05,
    "2021-01-27": 1.05,
    "2021-01-28": 1.05,
    "2021-01-29": 1.05,
    "2021-02-01": 1.05,
    "2021-02-02": 1.05,
    "2021-02-03": 1.05,
    "2021-02-04": 1.05,
    "2021-02-05": 1.05,
    "2021-02-08": 1.0510791385102027,
    "2021-02-09": 1.0510791385102025,
    "2021-02-10": 1.0510791385102025,
    "2021-02-11": 1.0510791385102025,
    "2021-02-12": 1.0569241672559697,
    "2021-02-15": 1.0569241672559697,
    "2021-02-16": 1.0596009455726492,
    "2021-02-17": 1.0683958292201992,
    "2021-02-18": 1.0759932549491782,
    "2021-02-19": 1.0895777262160462,
    "2021-02-22": 1.1041016753340471,
    "2021-02-23": 1.1207192889684012,
    "2021-02-24": 1.1576051770168312,
    "2021-02-25": 1.1805305922302307,
    "2021-02-26": 1.18720547793633,
    "2021-03-01": 1.166836500072562,
    "2021-03-02": 1.1464629696113258,
    "2021-03-03": 1.1007821979153458,
    "2021-03-04": 1.0702593569729673,
    "2021-03-05": 1.05,
    "2021-03-08": 1.05,
    "2021-03-09": 1.05,
    "2021-03-10": 1.05,
    "2021-03-11": 1.05,
    "2021-03-12": 1.05,
    "2021-03-15": 1.05,
    "2021-03-16": 1.05,
    "2021-03-17": 1.05,
    "2021-03-18": 1.05,
    "2021-03-19": 1.05,
    "2021-03-22": 1.05,
    "2021-03-23": 1.05,
    "2021-03-24": 1.05,
    "2021-03-25": 1.05,
    "2021-03-26": 1.05,
    "2021-03-29": 1.05,
    "2021-03-30": 1.05,
    "2021-03-31": 1.05,
    "2021-04-01": 1.05,
    "2021-04-02": 1.05,
    "2021-04-05": 1.05,
    "2021-04-06": 1.05,
    "2021-04-07": 1.05,
    "2021-04-08": 1.05,
    "2021-04-09": 1.05,
    "2021-04-12": 1.05,
    "2021-04-13": 1.05,
    "2021-04-14": 1.05,
    "2021-04-15": 1.05,
    "2021-04-16": 1.05,
    "2021-04-19": 1.05,
    "2021-04-20": 1.05,
    "2021-04-21": 1.05,
    "2021-04-22": 1.05,
    "2021-04-23": 1.05,
    "2021-04-26": 1.05,
    "2021-04-27": 1.05,
    "2021-04-28": 1.05,
    "2021-04-29": 1.05,
    "2021-04-30": 1.05,
    "2021-05-03": 1.05,
    "2021-05-04": 1.05,
    "2021-05-05": 1.05,
    "2021-05-06": 1.05,
    "2021-05-07": 1.05,
    "2021-05-10": 1.05,
    "2021-05-11": 1.05,
    "2021-05-12": 1.05,
    "2021-05-13": 1.05,
    "2021-05-14": 1.0796273033151595,
    "2021-05-17": 1.1373416187296823,
    "2021-05-18": 1.1943493241820866,
    "2021-05-19": 1.2562670847598851,
    "2021-05-20": 1.2843761570764436,
    "2021-05-21": 1.2819791334842496,
    "2021-05-24": 1.248753859455632,
    "2021-05-25": 1.198875578540661,
    "2021-05-26": 1.1571131506069123,
    "2021-05-27": 1.1428291326842603,
    "2021-05-28": 1.121773842738839,
    "2021-05-31": 1.121773842738839,
    "2021-06-01": 1.1084487750083472,
    "2021-06-02": 1.1133681304369474,
    "2021-06-03": 1.0954677894089557,
    "2021-06-04": 1.081642735015049,
    "2021-06-07": 1.0754677452375052,
    "2021-06-08": 1.0643037715820918,
    "2021-06-09": 1.0522549916160582,
    "2021-06-10": 1.05,
    "2021-06-11": 1.05,
    "2021-06-14": 1.05,
    "2021-06-15": 1.05,
    "2021-06-16": 1.05,
    "2021-06-17": 1.05,
    "2021-06-18": 1.05,
    "2021-06-21": 1.05,
    "2021-06-22": 1.05,
    "2021-06-23": 1.05,
    "2021-06-24": 1.05,
    "2021-06-25": 1.05,
    "2021-06-28": 1.05,
    "2021-06-29": 1.05,
    "2021-06-30": 1.05,
    "2021-07-01": 1.05,
    "2021-07-02": 1.05,
    "2021-07-05": 1.05,
    "2021-07-06": 1.05,
    "2021-07-07": 1.05,
    "2021-07-08": 1.05,
    "2021-07-09": 1.05,
    "2021-07-12": 1.05,
    "2021-07-13": 1.05,
    "2021-07-14": 1.05,
    "2021-07-15": 1.05,
    "2021-07-16": 1.05,
    "2021-07-19": 1.05,
    "2021-07-20": 1.05,
    "2021-07-21": 1.05,
    "2021-07-22": 1.05,
    "2021-07-23": 1.05,
    "2021-07-26": 1.05,
    "2021-07-27": 1.05,
    "2021-07-28": 1.05,
    "2021-07-29": 1.05,
    "2021-07-30": 1.05,
    "2021-08-02": 1.05,
    "2021-08-03": 1.05,
    "2021-08-04": 1.05,
    "2021-08-05": 1.05,
    "2021-08-06": 1.05,
    "2021-08-09": 1.05,
    "2021-08-10": 1.05,
    "2021-08-11": 1.05,
    "2021-08-12": 1.05,
    "2021-08-13": 1.05,
    "2021-08-16": 1.05,
    "2021-08-17": 1.05,
    "2021-08-18": 1.05,
    "2021-08-19": 1.05,
    "2021-08-20": 1.0507595616355991,
    "2021-08-23": 1.061522870144325,
    "2021-08-24": 1.0701478501437176,
    "2021-08-25": 1.0822546673215154,
    "2021-08-26": 1.092582560785916,
    "2021-08-27": 1.091822999150317,
    "2021-08-30": 1.0810596906415912,
    "2021-08-31": 1.0855699866327035,
    "2021-09-01": 1.0809299045666105,
    "2021-09-02": 1.0706020111022099,
    "2021-09-03": 1.0706020111022099,
    "2021-09-06": 1.0706020111022099,
    "2021-09-07": 1.089194219375026,
    "2021-09-08": 1.0897297357081641,
    "2021-09-09": 1.0822630005964595,
    "2021-09-10": 1.0822630005964593,
    "2021-09-13": 1.0822630005964593,
    "2021-09-14": 1.065261579982264,
    "2021-09-15": 1.0679635045807832,
    "2021-09-16": 1.0798315465117407,
    "2021-09-17": 1.0948800839891935,
    "2021-09-20": 1.1077220426021124,
    "2021-09-21": 1.1179446518572889,
    "2021-09-22": 1.1015719349351263,
    "2021-09-23": 1.0897038930041687,
    "2021-09-24": 1.074655355526716,
    "2021-09-27": 1.061813396913797,
    "2021-09-28": 1.0558184432252373,
    "2021-09-29": 1.0624314733595555,
    "2021-09-30": 1.066477483499345,
    "2021-10-01": 1.0664774834993447,
    "2021-10-04": 1.0664774834993447,
    "2021-10-05": 1.0606590402741074,
    "2021-10-06": 1.0540460101397895,
    "2021-10-07": 1.05,
    "2021-10-08": 1.05,
    "2021-10-11": 1.050294305201151,
    "2021-10-12": 1.05680102732287,
    "2021-10-13": 1.062797161698966,
    "2021-10-14": 1.0627971616989662,
    "2021-10-15": 1.0627971616989662,
    "2021-10-18": 1.0918117316413554,
    "2021-10-19": 1.1247613311336313,
    "2021-10-20": 1.1470532099820505,
    "2021-10-21": 1.18668080922356,
    "2021-10-22": 1.2493319892521282,
    "2021-10-25": 1.2752393796066923,
    "2021-10-26": 1.280706197043962,
    "2021-10-27": 1.277485321492349,
    "2021-10-28": 1.2633678207553545,
    "2021-10-29": 1.2377555725010807,
    "2021-11-01": 1.2409898129769994,
    "2021-11-02": 1.225588484674907,
    "2021-11-03": 1.2364555659616059,
    "2021-11-04": 1.2120316259535617,
    "2021-11-05": 1.1760017906284184,
    "2021-11-08": 1.1283478314885342,
    "2021-11-09": 1.1059033935198788,
    "2021-11-10": 1.0699691745602775,
    "2021-11-11": 1.0688830160638065,
    "2021-11-12": 1.0678739196146556,
    "2021-11-15": 1.0570773727805165,
    "2021-11-16": 1.05,
    "2021-11-17": 1.05,
    "2021-11-18": 1.05,
    "2021-11-19": 1.05,
    "2021-11-22": 1.05,
    "2021-11-23": 1.05,
    "2021-11-24": 1.05,
    "2021-11-25": 1.05,
    "2021-11-26": 1.05,
    "2021-11-29": 1.05,
    "2021-11-30": 1.05,
    "2021-12-01": 1.05,
    "2021-12-02": 1.05,
    "2021-12-03": 1.05,
    "2021-12-06": 1.05,
    "2021-12-07": 1.05,
    "2021-12-08": 1.05,
    "2021-12-09": 1.05,
    "2021-12-10": 1.05,
    "2021-12-13": 1.05,
    "2021-12-14": 1.05,
    "2021-12-15": 1.05,
    "2021-12-16": 1.05,
    "2021-12-17": 1.05,
    "2021-12-20": 1.05,
    "2021-12-21": 1.05,
    "2021-12-22": 1.05,
    "2021-12-23": 1.05,
    "2021-12-24": 1.05,
    "2021-12-27": 1.05,
    "2021-12-28": 1.05,
    "2021-12-29": 1.05,
    "2021-12-30": 1.0539374552537908,
    "2021-12-31": 1.0567342990568882,
    "2022-01-03": 1.0567342990568882,
    "2022-01-04": 1.0786508281657796,
    "2022-01-05": 1.1043021612803152,
    "2022-01-06": 1.1261300924732418,
    "2022-01-07": 1.1386333468861285,
    "2022-01-10": 1.16851938524155,
    "2022-01-11": 1.1685524116304047,
    "2022-01-12": 1.172208300549432,
    "2022-01-13": 1.166553655232531,
    "2022-01-14": 1.1764227075602978,
    "2022-01-17": 1.1764227075602978,
    "2022-01-18": 1.1884052076886706,
    "2022-01-19": 1.1892261561724664,
    "2022-01-20": 1.1844906619832805,
    "2022-01-21": 1.168342147370155,
    "2022-01-24": 1.1573874439110061,
    "2022-01-25": 1.1301490565756722,
    "2022-01-26": 1.1210257300932356,
    "2022-01-27": 1.102839196980304,
    "2022-01-28": 1.1063582263972203,
    "2022-01-31": 1.1025600248551126,
    "2022-02-01": 1.0991143052931047,
    "2022-02-02": 1.1211380811927465,
    "2022-02-03": 1.1472298183131133,
    "2022-02-04": 1.1558828977059266,
    "2022-02-07": 1.1604452708283453,
    "2022-02-08": 1.1520951380865718,
    "2022-02-09": 1.116424184687825,
    "2022-02-10": 1.0857077036745935,
    "2022-02-11": 1.0720770736879155,
    "2022-02-14": 1.077993990185518,
    "2022-02-15": 1.0790892980305038,
    "2022-02-16": 1.0926737094085524,
    "2022-02-17": 1.103285124730112,
    "2022-02-18": 1.1052117406546256,
    "2022-02-21": 1.1052117406546256,
    "2022-02-22": 1.1084134411101514,
    "2022-02-23": 1.1279965058088353,
    "2022-02-24": 1.1542859051746341,
    "2022-02-25": 1.1709274354123917,
    "2022-02-28": 1.171538377198772,
    "2022-03-01": 1.1649440529677635,
    "2022-03-02": 1.1414313815794146,
    "2022-03-03": 1.1015575708355674,
    "2022-03-04": 1.0732168143002676,
    "2022-03-07": 1.0848911709283855,
    "2022-03-08": 1.0697163990846184,
    "2022-03-09": 1.0697163990846186,
    "2022-03-10": 1.0697163990846184,
    "2022-03-11": 1.0690437592220203,
    "2022-03-14": 1.0523281395432664,
    "2022-03-15": 1.05,
    "2022-03-16": 1.05,
    "2022-03-17": 1.05,
    "2022-03-18": 1.05,
    "2022-03-21": 1.05,
    "2022-03-22": 1.05,
    "2022-03-23": 1.05,
    "2022-03-24": 1.05,
    "2022-03-25": 1.05,
    "2022-03-28": 1.05,
    "2022-03-29": 1.05,
    "2022-03-30": 1.05,
    "2022-03-31": 1.05,
    "2022-04-01": 1.05,
    "2022-04-04": 1.05,
    "2022-04-05": 1.05,
    "2022-04-06": 1.06205400148116,
    "2022-04-07": 1.0737399575190132,
    "2022-04-08": 1.0895161601874126,
    "2022-04-11": 1.1152861783986858,
    "2022-04-12": 1.1226433365462025,
    "2022-04-13": 1.1106660128137673,
    "2022-04-14": 1.0989800567759138,
    "2022-04-15": 1.0989800567759138,
    "2022-04-18": 1.0832038541075144,
    "2022-04-19": 1.0622862047637645,
    "2022-04-20": 1.0727091713652355,
    "2022-04-21": 1.0985928850050346,
    "2022-04-22": 1.1258172236994766,
    "2022-04-25": 1.146132646371495,
    "2022-04-26": 1.1723609765549459,
    "2022-04-27": 1.1871123550718135,
    "2022-04-28": 1.203251394968779,
    "2022-04-29": 1.2242656160817111,
    "2022-05-02": 1.260862448702185,
    "2022-05-03": 1.2636874614816695,
    "2022-05-04": 1.2502615318809052,
    "2022-05-05": 1.2270881964985514,
    "2022-05-06": 1.1950471946333026,
    "2022-05-09": 1.1564900649129886,
    "2022-05-10": 1.1339640598361413,
    "2022-05-11": 1.1148584861710502,
    "2022-05-12": 1.0959323902679146,
    "2022-05-13": 1.0797348323257896,
    "2022-05-16": 1.0613797067536117,
    "2022-05-17": 1.05,
    "2022-05-18": 1.05,
    "2022-05-19": 1.05,
    "2022-05-20": 1.05,
    "2022-05-23": 1.05,
    "2022-05-24": 1.05,
    "2022-05-25": 1.05,
    "2022-05-26": 1.05,
    "2022-05-27": 1.05,
    "2022-05-30": 1.05,
    "2022-05-31": 1.05,
    "2022-06-01": 1.05,
    "2022-06-02": 1.05,
    "2022-06-03": 1.05,
    "2022-06-06": 1.05,
    "2022-06-07": 1.05,
    "2022-06-08": 1.05,
    "2022-06-09": 1.05,
    "2022-06-10": 1.05,
    "2022-06-13": 1.0576136895323893,
    "2022-06-14": 1.0730143897030995,
    "2022-06-15": 1.0730143897030993,
    "2022-06-16": 1.0730143897030993,
    "2022-06-17": 1.0730143897030993,
    "2022-06-20": 1.0730143897030993,
    "2022-06-21": 1.06540070017071,
    "2022-06-22": 1.05,
    "2022-06-23": 1.05,
    "2022-06-24": 1.05,
    "2022-06-27": 1.05,
    "2022-06-28": 1.05,
    "2022-06-29": 1.05,
    "2022-06-30": 1.05,
    "2022-07-01": 1.05,
    "2022-07-04": 1.05,
    "2022-07-05": 1.05,
    "2022-07-06": 1.05,
    "2022-07-07": 1.05,
    "2022-07-08": 1.05,
    "2022-07-11": 1.05,
    "2022-07-12": 1.05,
    "2022-07-13": 1.05,
    "2022-07-14": 1.05,
    "2022-07-15": 1.05,
    "2022-07-18": 1.05,
    "2022-07-19": 1.05,
    "2022-07-20": 1.05,
    "2022-07-21": 1.05,
    "2022-07-22": 1.05,
    "2022-07-25": 1.05,
    "2022-07-26": 1.05,
    "2022-07-27": 1.05,
    "2022-07-28": 1.05,
    "2022-07-29": 1.05,
    "2022-08-01": 1.0514380624564572,
    "2022-08-02": 1.0518227631538362,
    "2022-08-03": 1.0518227631538362,
    "2022-08-04": 1.0518227631538362,
    "2022-08-05": 1.0518227631538362,
    "2022-08-08": 1.0503847006973792,
    "2022-08-09": 1.0517693362424099,
    "2022-08-10": 1.0517693362424096,
    "2022-08-11": 1.0517693362424099,
    "2022-08-12": 1.0517693362424099,
    "2022-08-15": 1.0517693362424099,
    "2022-08-16": 1.05,
    "2022-08-17": 1.05,
    "2022-08-18": 1.05,
    "2022-08-19": 1.05,
    "2022-08-22": 1.0815927806612238,
    "2022-08-23": 1.110290949587252,
    "2022-08-24": 1.138449863441767,
    "2022-08-25": 1.149617921071262,
    "2022-08-26": 1.159852949950249,
    "2022-08-29": 1.1517975990549323,
    "2022-08-30": 1.1457626205512368,
    "2022-08-31": 1.1521939938916683,
    "2022-09-01": 1.1628246130740618,
    "2022-09-02": 1.1636571251472234,
    "2022-09-05": 1.1636571251472234,
    "2022-09-06": 1.1806413111685348,
    "2022-09-07": 1.1977118181179507,
    "2022-09-08": 1.1860837450075556,
    "2022-09-09": 1.1839556369788933,
    "2022-09-12": 1.2034171225817736,
    "2022-09-13": 1.2092037922001424,
    "2022-09-14": 1.2186423655366605,
    "2022-09-15": 1.2417190716238324,
    "2022-09-16": 1.2700881042162993,
    "2022-09-19": 1.3085973494232923,
    "2022-09-20": 1.340607694283489,
    "2022-09-21": 1.3479817479166964,
    "2022-09-22": 1.3209674945157908,
    "2022-09-23": 1.299947498996284,
    "2022-09-26": 1.2762149365266202,
    "2022-09-27": 1.2498051938803028,
    "2022-09-28": 1.1932588695388284,
    "2022-09-29": 1.1742342027680106,
    "2022-09-30": 1.147214596911825,
    "2022-10-03": 1.1019088876194667,
    "2022-10-04": 1.05,
    "2022-10-05": 1.05,
    "2022-10-06": 1.05,
    "2022-10-07": 1.05,
    "2022-10-10": 1.05,
    "2022-10-11": 1.05,
    "2022-10-12": 1.05,
    "2022-10-13": 1.05,
    "2022-10-14": 1.05,
    "2022-10-17": 1.05,
    "2022-10-18": 1.05,
    "2022-10-19": 1.05,
    "2022-10-20": 1.05,
    "2022-10-21": 1.05,
    "2022-10-24": 1.05,
    "2022-10-25": 1.05,
    "2022-10-26": 1.05,
    "2022-10-27": 1.0696146925669499,
    "2022-10-28": 1.0895172041472674,
    "2022-10-31": 1.113781557135761,
    "2022-11-01": 1.1686546100517705,
    "2022-11-02": 1.2141418319933497,
    "2022-11-03": 1.2495475601488315,
    "2022-11-04": 1.2729232671064088,
    "2022-11-07": 1.2915908410891377,
    "2022-11-08": 1.2614376106917562,
    "2022-11-09": 1.2388882511430634,
    "2022-11-10": 1.1838678304206316,
    "2022-11-11": 1.1405896118827372,
    "2022-11-14": 1.0976576849115145,
    "2022-11-15": 1.0729378623928862,
    "2022-11-16": 1.05,
    "2022-11-17": 1.05,
    "2022-11-18": 1.05,
    "2022-11-21": 1.05,
    "2022-11-22": 1.05,
    "2022-11-23": 1.05,
    "2022-11-24": 1.05,
    "2022-11-25": 1.05,
    "2022-11-28": 1.05,
    "2022-11-29": 1.05,
    "2022-11-30": 1.05,
    "2022-12-01": 1.05,
    "2022-12-02": 1.05,
    "2022-12-05": 1.05,
    "2022-12-06": 1.05,
    "2022-12-07": 1.05,
    "2022-12-08": 1.05,
    "2022-12-09": 1.05,
    "2022-12-12": 1.0817997999846098,
    "2022-12-13": 1.0925691089181773,
    "2022-12-14": 1.0925691089181773,
    "2022-12-15": 1.0925691089181773,
    "2022-12-16": 1.0925691089181775,
    "2022-12-19": 1.0607693089335677,
    "2022-12-20": 1.05,
    "2022-12-21": 1.05,
    "2022-12-22": 1.05,
    "2022-12-23": 1.05,
    "2022-12-26": 1.05,
    "2022-12-27": 1.05,
    "2022-12-28": 1.05,
    "2022-12-29": 1.05,
    "2022-12-30": 1.05,
    "2023-01-02": 1.05,
    "2023-01-03": 1.0640552746341023,
    "2023-01-04": 1.0640552746341023,
    "2023-01-05": 1.0640552746341023,
    "2023-01-06": 1.0640552746341023,
    "2023-01-09": 1.0640552746341023,
    "2023-01-10": 1.05,
    "2023-01-11": 1.05,
    "2023-01-12": 1.05,
    "2023-01-13": 1.05,
    "2023-01-16": 1.05,
    "2023-01-17": 1.05,
    "2023-01-18": 1.05,
    "2023-01-19": 1.05,
    "2023-01-20": 1.05,
    "2023-01-23": 1.05,
    "2023-01-24": 1.05,
    "2023-01-25": 1.05,
    "2023-01-26": 1.05,
    "2023-01-27": 1.05,
    "2023-01-30": 1.05,
    "2023-01-31": 1.05,
    "2023-02-01": 1.05,
    "2023-02-02": 1.05,
    "2023-02-03": 1.05,
    "2023-02-06": 1.05,
    "2023-02-07": 1.05,
    "2023-02-08": 1.05,
    "2023-02-09": 1.05,
    "2023-02-10": 1.05,
    "2023-02-13": 1.05,
    "2023-02-14": 1.05,
    "2023-02-15": 1.05,
    "2023-02-16": 1.085783303538345,
    "2023-02-17": 1.117117694102323,
    "2023-02-20": 1.117117694102323,
    "2023-02-21": 1.156694337965804,
    "2023-02-22": 1.1894266643741265,
    "2023-02-23": 1.2233046502935578,
    "2023-02-24": 1.2299281001039628,
    "2023-02-27": 1.2315421896044918,
    "2023-02-28": 1.2311262234336844,
    "2023-03-01": 1.2276345935542456,
    "2023-03-02": 1.2274949511310251,
    "2023-03-03": 1.2262157244590375,
    "2023-03-06": 1.2473109232894273,
    "2023-03-07": 1.2963705066035587,
    "2023-03-08": 1.3638225866318217,
    "2023-03-09": 1.4164728904275168,
    "2023-03-10": 1.3972734815702241,
    "2023-03-13": 1.389189934682524,
    "2023-03-14": 1.3312692501346555,
    "2023-03-15": 1.2518155006170109,
    "2023-03-16": 1.1756918898333542,
    "2023-03-17": 1.1643213391773481,
    "2023-03-20": 1.12782650000048,
    "2023-03-21": 1.097526923541544,
    "2023-03-22": 1.080287896502042,
    "2023-03-23": 1.0700228599937927,
    "2023-03-24": 1.0594652928303288,
    "2023-03-27": 1.05,
    "2023-03-28": 1.05,
    "2023-03-29": 1.05,
    "2023-03-30": 1.05,
    "2023-03-31": 1.05,
    "2023-04-03": 1.05,
    "2023-04-04": 1.05,
    "2023-04-05": 1.05,
    "2023-04-06": 1.05,
    "2023-04-07": 1.05,
    "2023-04-10": 1.05,
    "2023-04-11": 1.05,
    "2023-04-12": 1.05,
    "2023-04-13": 1.05,
    "2023-04-14": 1.05,
    "2023-04-17": 1.05,
    "2023-04-18": 1.05,
    "2023-04-19": 1.0544831645172643,
    "2023-04-20": 1.0657951369346197,
    "2023-04-21": 1.0912479235508667,
    "2023-04-24": 1.1368697745223482,
    "2023-04-25": 1.1822774091298973,
    "2023-04-26": 1.2445257946973007,
    "2023-04-27": 1.2689870922306494,
    "2023-04-28": 1.2716137969029,
    "2023-05-01": 1.2452438995911916,
    "2023-05-02": 1.219271732234216,
    "2023-05-03": 1.1695811476546722,
    "2023-05-04": 1.167904966327172,
    "2023-05-05": 1.150208262478629,
    "2023-05-08": 1.1453105156972694,
    "2023-05-09": 1.1533370380842398,
    "2023-05-10": 1.1417528305451203,
    "2023-05-11": 1.1160079241455771,
    "2023-05-12": 1.1056251367056222,
    "2023-05-15": 1.0985447078532151,
    "2023-05-16": 1.0856954450626732,
    "2023-05-17": 1.0802386870966685,
    "2023-05-18": 1.071886504873008,
    "2023-05-19": 1.0796668178613709,
    "2023-05-22": 1.088258655527688,
    "2023-05-23": 1.0917977243919883,
    "2023-05-24": 1.1320839380868342,
    "2023-05-25": 1.1779259342209394,
    "2023-05-26": 1.2356291340548864,
    "2023-05-29": 1.2356291340548864,
    "2023-05-30": 1.267386860433136,
    "2023-05-31": 1.2984602403704562,
    "2023-06-01": 1.3157904069512238,
    "2023-06-02": 1.314177096274716,
    "2023-06-05": 1.2968730812235982,
    "2023-06-06": 1.300131629695199,
    "2023-06-07": 1.3126541471782283,
    "2023-06-08": 1.2922293818632358,
    "2023-06-09": 1.303429034479235,
    "2023-06-12": 1.3278774081244453,
    "2023-06-13": 1.3452312919909781,
    "2023-06-14": 1.335508791392558,
    "2023-06-15": 1.3240017592817257,
    "2023-06-16": 1.2870034531583348,
    "2023-06-19": 1.2870034531583348,
    "2023-06-20": 1.2354651235496166,
    "2023-06-21": 1.205652671755253,
    "2023-06-22": 1.1763579887698836,
    "2023-06-23": 1.1636861131640328,
    "2023-06-26": 1.1651236605491069,
    "2023-06-27": 1.1796094691615864,
    "2023-06-28": 1.1657823180439748,
    "2023-06-29": 1.165473908086565,
    "2023-06-30": 1.174066127335712,
    "2023-07-03": 1.1772346437624321,
    "2023-07-04": 1.1772346437624321,
    "2023-07-05": 1.163249617846679,
    "2023-07-06": 1.1988273010849486,
    "2023-07-07": 1.2282015316311092,
    "2023-07-10": 1.2640632683431305,
    "2023-07-11": 1.2982974520965511,
    "2023-07-12": 1.283751264481087,
    "2023-07-13": 1.2235774097460868,
    "2023-07-14": 1.1717810796094739,
    "2023-07-17": 1.1143144164043675,
    "2023-07-18": 1.0585753745741262,
    "2023-07-19": 1.0515312376851793,
    "2023-07-20": 1.0515312376851793,
    "2023-07-21": 1.0515312376851793,
    "2023-07-24": 1.0515312376851793,
    "2023-07-25": 1.05,
    "2023-07-26": 1.05,
    "2023-07-27": 1.05,
    "2023-07-28": 1.05,
    "2023-07-31": 1.05,
    "2023-08-01": 1.05,
    "2023-08-02": 1.05,
    "2023-08-03": 1.0529168748412776,
    "2023-08-04": 1.0529168748412776,
    "2023-08-07": 1.0529168748412778,
    "2023-08-08": 1.0529168748412778,
    "2023-08-09": 1.0529168748412778,
    "2023-08-10": 1.05,
    "2023-08-11": 1.05,
    "2023-08-14": 1.0555804145055907,
    "2023-08-15": 1.0660027048531895,
    "2023-08-16": 1.0688419076352416,
    "2023-08-17": 1.08407286810191,
    "2023-08-18": 1.1000415142162163,
    "2023-08-21": 1.125090835119338,
    "2023-08-22": 1.1361648862405078,
    "2023-08-23": 1.1333256834584557,
    "2023-08-24": 1.118094722991787,
    "2023-08-25": 1.102126076877481,
    "2023-08-28": 1.0721551568936594,
    "2023-08-29": 1.050658815424891,
    "2023-08-30": 1.050658815424891,
    "2023-08-31": 1.050658815424891,
    "2023-09-01": 1.050658815424891,
    "2023-09-04": 1.050658815424891,
    "2023-09-05": 1.0671944949276568,
    "2023-09-06": 1.0877248542709057,
    "2023-09-07": 1.09902303916727,
    "2023-09-08": 1.1092289047177863,
    "2023-09-11": 1.1302875337018086,
    "2023-09-12": 1.139370235578237,
    "2023-09-13": 1.1374357054495534,
    "2023-09-14": 1.133832699657742,
    "2023-09-15": 1.1305349294689815,
    "2023-09-18": 1.1266591335670462,
    "2023-09-19": 1.1255248230615638,
    "2023-09-20": 1.1378112086792709,
    "2023-09-21": 1.1501859503956784,
    "2023-09-22": 1.1865132839796506,
    "2023-09-25": 1.2330962110440826,
    "2023-09-26": 1.2729067523762179,
    "2023-09-27": 1.315776566857272,
    "2023-09-28": 1.3792393209246832,
    "2023-09-29": 1.4110515310722842,
    "2023-10-02": 1.4434486696331632,
    "2023-10-03": 1.4926673254062948,
    "2023-10-04": 1.4932875538128818,
    "2023-10-05": 1.4850620214363033,
    "2023-10-06": 1.4792662078012935,
    "2023-10-09": 1.4536348481845556,
    "2023-10-10": 1.383269317850788,
    "2023-10-11": 1.3306553830371004,
    "2023-10-12": 1.266057217777763,
    "2023-10-13": 1.200851800193996,
    "2023-10-16": 1.1336865182128317,
    "2023-10-17": 1.0898799651427296,
    "2023-10-18": 1.0681216422365032,
    "2023-10-19": 1.057412664984048,
    "2023-10-20": 1.0557869824424966,
    "2023-10-23": 1.0524207253330011,
    "2023-10-24": 1.052420725333001,
    "2023-10-25": 1.052420725333001,
    "2023-10-26": 1.0524207253330011,
    "2023-10-27": 1.05,
    "2023-10-30": 1.05,
    "2023-10-31": 1.05,
    "2023-11-01": 1.05,
    "2023-11-02": 1.05,
    "2023-11-03": 1.05,
    "2023-11-06": 1.05,
    "2023-11-07": 1.05,
    "2023-11-08": 1.05,
    "2023-11-09": 1.05,
    "2023-11-10": 1.05,
    "2023-11-13": 1.05,
    "2023-11-14": 1.05,
    "2023-11-15": 1.05,
    "2023-11-16": 1.05,
    "2023-11-17": 1.05,
    "2023-11-20": 1.05,
    "2023-11-21": 1.05,
    "2023-11-22": 1.05,
    "2023-11-23": 1.05,
    "2023-11-24": 1.05,
    "2023-11-27": 1.05,
    "2023-11-28": 1.05,
    "2023-11-29": 1.05,
    "2023-11-30": 1.05,
    "2023-12-01": 1.05,
    "2023-12-04": 1.05,
    "2023-12-05": 1.05,
    "2023-12-06": 1.05,
    "2023-12-07": 1.0529136894461328,
    "2023-12-08": 1.0529136894461328,
    "2023-12-11": 1.0529136894461328,
    "2023-12-12": 1.0529136894461328,
    "2023-12-13": 1.0529136894461328,
    "2023-12-14": 1.05,
    "2023-12-15": 1.05,
    "2023-12-18": 1.05,
    "2023-12-19": 1.05,
    "2023-12-20": 1.05,
    "2023-12-21": 1.05,
    "2023-12-22": 1.05,
    "2023-12-25": 1.05,
    "2023-12-26": 1.05,
    "2023-12-27": 1.05,
    "2023-12-28": 1.05,
    "2023-12-29": 1.05,
    "2024-01-01": 1.05,
    "2024-01-02": 1.058461587344778,
    "2024-01-03": 1.075026315263831,
    "2024-01-04": 1.081018379303768,
    "2024-01-05": 1.0866806997230847,
    "2024-01-08": 1.0961158888892604,
    "2024-01-09": 1.0914923416585587,
    "2024-01-10": 1.087924095127128,
    "2024-01-11": 1.0819320310871912,
    "2024-01-12": 1.0808416875980569,
    "2024-01-15": 1.0808416875980569,
    "2024-01-16": 1.1124599439495562,
    "2024-01-17": 1.1543847131393137,
    "2024-01-18": 1.1888012223605098,
    "2024-01-19": 1.2312843726424654,
    "2024-01-22": 1.26914616038449,
    "2024-01-23": 1.276272321249397,
    "2024-01-24": 1.2729233920799594,
    "2024-01-25": 1.2608866130785297,
    "2024-01-26": 1.2416027316016711,
    "2024-01-29": 1.2488663034556766,
    "2024-01-30": 1.2463487455302702,
    "2024-01-31": 1.2355287303059095,
    "2024-02-01": 1.2378114350488967,
    "2024-02-02": 1.2146121662437994,
    "2024-02-05": 1.1649148297175869,
    "2024-02-06": 1.1192527812604116,
    "2024-02-07": 1.087658916350376,
    "2024-02-08": 1.05,
    "2024-02-09": 1.05,
    "2024-02-12": 1.05,
    "2024-02-13": 1.05,
    "2024-02-14": 1.05,
    "2024-02-15": 1.05,
    "2024-02-16": 1.05,
    "2024-02-19": 1.05,
    "2024-02-20": 1.05,
    "2024-02-21": 1.05,
    "2024-02-22": 1.05,
    "2024-02-23": 1.05,
    "2024-02-26": 1.05,
    "2024-02-27": 1.05,
    "2024-02-28": 1.05,
    "2024-02-29": 1.05,
    "2024-03-01": 1.05,
    "2024-03-04": 1.05,
    "2024-03-05": 1.062168636386958,
    "2024-03-06": 1.0894369763101213,
    "2024-03-07": 1.1274238690439944,
    "2024-03-08": 1.1555190927699923,
    "2024-03-11": 1.205437819543122,
    "2024-03-12": 1.224820003017562,
    "2024-03-13": 1.216393688250744,
    "2024-03-14": 1.2232855516074916,
    "2024-03-15": 1.2445407683115852,
    "2024-03-18": 1.2568917597266236,
    "2024-03-19": 1.2820560005332902,
    "2024-03-20": 1.3000505372866824,
    "2024-03-21": 1.2943904333795977,
    "2024-03-22": 1.2781011538325318,
    "2024-03-25": 1.259920340711575,
    "2024-03-26": 1.2470150374432127,
    "2024-03-27": 1.2397002700531432,
    "2024-03-28": 1.2336298299317003,
    "2024-03-29": 1.2336298299317003,
    "2024-04-01": 1.217610475035593,
    "2024-04-02": 1.208295624477519,
    "2024-04-03": 1.204284770643626,
    "2024-04-04": 1.2241416551770428,
    "2024-04-05": 1.2311785949577705,
    "2024-04-08": 1.2668778821699445,
    "2024-04-09": 1.269991306109203,
    "2024-04-10": 1.2364532283465546,
    "2024-04-11": 1.1930682292525145,
    "2024-04-12": 1.1557793776638638,
    "2024-04-15": 1.1158470356836234,
    "2024-04-16": 1.0877710615897755,
    "2024-04-17": 1.0815102357866144,
    "2024-04-18": 1.0755165558275703,
    "2024-04-19": 1.0726202555733995,
    "2024-04-22": 1.0598115043545477,
    "2024-04-23": 1.05,
    "2024-04-24": 1.05,
    "2024-04-25": 1.05,
    "2024-04-26": 1.05,
    "2024-04-29": 1.05,
    "2024-04-30": 1.05,
    "2024-05-01": 1.05,
    "2024-05-02": 1.05,
    "2024-05-03": 1.05,
    "2024-05-06": 1.05,
    "2024-05-07": 1.05,
    "2024-05-08": 1.05,
    "2024-05-09": 1.05,
    "2024-05-10": 1.05,
    "2024-05-13": 1.072877539478159,
    "2024-05-14": 1.1367680782801854,
    "2024-05-15": 1.1579779858631638,
    "2024-05-16": 1.180524360197419,
    "2024-05-17": 1.2024548679493932,
    "2024-05-20": 1.1978329407081236,
    "2024-05-21": 1.1452206102272364,
    "2024-05-22": 1.1266102269205283,
    "2024-05-23": 1.119533459685579,
    "2024-05-24": 1.1166338183389517,
    "2024-05-27": 1.1166338183389517,
    "2024-05-28": 1.1178517873709857,
    "2024-05-29": 1.1254193399780064,
    "2024-05-30": 1.1382600279807462,
    "2024-05-31": 1.1530957763392087,
    "2024-06-03": 1.1465248429068067,
    "2024-06-04": 1.1554523457546204,
    "2024-06-05": 1.1674041019546648,
    "2024-06-06": 1.179966437943783,
    "2024-06-07": 1.1496610824860147,
    "2024-06-10": 1.1372011495130696,
    "2024-06-11": 1.1088000653963328,
    "2024-06-12": 1.0780025482681281,
    "2024-06-13": 1.05,
    "2024-06-14": 1.05,
    "2024-06-17": 1.05,
    "2024-06-18": 1.05,
    "2024-06-19": 1.05,
    "2024-06-20": 1.05,
    "2024-06-21": 1.05,
    "2024-06-24": 1.05,
    "2024-06-25": 1.05,
    "2024-06-26": 1.05,
    "2024-06-27": 1.05,
    "2024-06-28": 1.05,
    "2024-07-01": 1.05,
    "2024-07-02": 1.05,
    "2024-07-03": 1.05,
    "2024-07-04": 1.05,
    "2024-07-05": 1.05,
    "2024-07-08": 1.05,
    "2024-07-09": 1.05,
    "2024-07-10": 1.05,
    "2024-07-11": 1.05,
    "2024-07-12": 1.05,
    "2024-07-15": 1.05,
    "2024-07-16": 1.05,
    "2024-07-17": 1.05,
    "2024-07-18": 1.05,
    "2024-07-19": 1.05,
    "2024-07-22": 1.0511727055329099,
    "2024-07-23": 1.0529681703838054,
    "2024-07-24": 1.0529681703838054,
    "2024-07-25": 1.058877390132435,
    "2024-07-26": 1.0678424135785445,
    "2024-07-29": 1.0780403558286333,
    "2024-07-30": 1.0973958240813086,
    "2024-07-31": 1.174874430293915,
    "2024-08-01": 1.2149038676474562,
    "2024-08-02": 1.2308137653525475,
    "2024-08-05": 1.268553466642006,
    "2024-08-06": 1.2685739922000667,
    "2024-08-07": 1.1910953859874602,
    "2024-08-08": 1.1451567288852895,
    "2024-08-09": 1.1202818077340893,
    "2024-08-12": 1.0832154495737218,
    "2024-08-13": 1.076156981221199,
    "2024-08-14": 1.0761569812211993,
    "2024-08-15": 1.0761569812211993,
    "2024-08-16": 1.076156981221199,
    "2024-08-19": 1.064112990309109,
    "2024-08-20": 1.05,
    "2024-08-21": 1.05,
    "2024-08-22": 1.05,
    "2024-08-23": 1.05,
    "2024-08-26": 1.05,
    "2024-08-27": 1.05,
    "2024-08-28": 1.05,
    "2024-08-29": 1.05,
    "2024-08-30": 1.05,
    "2024-09-02": 1.05,
    "2024-09-03": 1.101021397498799,
    "2024-09-04": 1.151140038039725,
    "2024-09-05": 1.2139878419861074,
    "2024-09-06": 1.279051606370997,
    "2024-09-09": 1.3424832450483162,
    "2024-09-10": 1.3656463141523472,
    "2024-09-11": 1.3776931612469514,
    "2024-09-12": 1.3641860039523697,
    "2024-09-13": 1.355530521850937,
    "2024-09-16": 1.4018161710914323,
    "2024-09-17": 1.4348260109565387,
    "2024-09-18": 1.458880037017442,
    "2024-09-19": 1.4705594510999773,
    "2024-09-20": 1.4751094174456774,
    "2024-09-23": 1.4699524785941964,
    "2024-09-24": 1.4804491231777646,
    "2024-09-25": 1.4938285647358374,
    "2024-09-26": 1.5281301426870812,
    "2024-09-27": 1.5630491023685966,
    "2024-09-30": 1.54938388032827,
    "2024-10-01": 1.5507950447688963,
    "2024-10-02": 1.5722267275596562,
    "2024-10-03": 1.6043157071306016,
    "2024-10-04": 1.5748617801059746,
    "2024-10-07": 1.6176574138073323,
    "2024-10-08": 1.6340735305616,
    "2024-10-09": 1.665348397649785,
    "2024-10-10": 1.6735250340462913,
    "2024-10-11": 1.7399555362074988,
    "2024-10-14": 1.7509049505199397,
    "2024-10-15": 1.7662491005657208,
    "2024-10-16": 1.7425160246854863,
    "2024-10-17": 1.7239184558663774,
    "2024-10-18": 1.7097140472551569,
    "2024-10-21": 1.6744158860193683,
    "2024-10-22": 1.6340298807616471,
    "2024-10-23": 1.6168500939034978,
    "2024-10-24": 1.5968152109422038,
    "2024-10-25": 1.5772708015676724,
    "2024-10-28": 1.5843566498143278,
    "2024-10-29": 1.592921791603325,
    "2024-10-30": 1.6183416236308532,
    "2024-10-31": 1.6708971303884745,
    "2024-11-01": 1.7100432080026207,
    "2024-11-04": 1.7730198042983816,
    "2024-11-05": 1.8157732606960884,
    "2024-11-06": 1.7549155204478732,
    "2024-11-07": 1.6279792918075024,
    "2024-11-08": 1.4996578403466911,
    "2024-11-11": 1.3319772325246864,
    "2024-11-12": 1.1813825636390984,
    "2024-11-13": 1.1121963888235853,
    "2024-11-14": 1.0896223078337068,
    "2024-11-15": 1.0917854273737362,
    "2024-11-18": 1.0921809378983243,
    "2024-11-19": 1.1127976737700105,
    "2024-11-20": 1.141151506603506,
    "2024-11-21": 1.1663348677312382,
    "2024-11-22": 1.1763122182805543,
    "2024-11-25": 1.1641928572315368,
    "2024-11-26": 1.1323758150242766,
    "2024-11-27": 1.0972534228319137,
    "2024-11-28": 1.0972534228319137,
    "2024-11-29": 1.0720700617041814,
    "2024-12-02": 1.05,
    "2024-12-03": 1.05,
    "2024-12-04": 1.05,
    "2024-12-05": 1.05,
    "2024-12-06": 1.05,
    "2024-12-09": 1.05,
    "2024-12-10": 1.0535403658392506,
    "2024-12-11": 1.0535403658392506,
    "2024-12-12": 1.060139529203664,
    "2024-12-13": 1.0619431721867207,
    "2024-12-16": 1.0787986718474145,
    "2024-12-17": 1.0910166270206116,
    "2024-12-18": 1.0910166270206116,
    "2024-12-19": 1.0844174636561978,
    "2024-12-20": 1.0826138206731415,
    "2024-12-23": 1.0657583210124475,
    "2024-12-24": 1.05,
    "2024-12-25": 1.05,
    "2024-12-26": 1.0858831037252747,
    "2024-12-27": 1.120371566078758,
    "2024-12-30": 1.159982036891426,
    "2024-12-31": 1.2164307858435504,
    "2025-01-01": 1.2164307858435504,
    "2025-01-02": 1.2663334674545352,
    "2025-01-03": 1.2716362823488025,
    "2025-01-06": 1.3030234621929062,
    "2025-01-07": 1.3384284462767646,
    "2025-01-08": 1.3609362849324935,
    "2025-01-09": 1.3609362849324935,
    "2025-01-10": 1.3687040689111982,
    "2025-01-13": 1.4045322564670806,
    "2025-01-14": 1.4076749238790942,
    "2025-01-15": 1.332659468982568,
    "2025-01-16": 1.2537028813747146,
    "2025-01-17": 1.1972904264922903,
    "2025-01-20": 1.1972904264922903,
    "2025-01-21": 1.121684334994172,
    "2025-01-22": 1.0558586027751486,
    "2025-01-23": 1.0635082320690445,
    "2025-01-24": 1.0715475214145196,
    "2025-01-27": 1.0897194966795642,
    "2025-01-28": 1.1011008786178078,
    "2025-01-29": 1.105814464197938,
    "2025-01-30": 1.1061475949221427,
    "2025-01-31": 1.1125585295012899,
    "2025-02-03": 1.1136932128902264,
    "2025-02-04": 1.1104295368737072,
    "2025-02-05": 1.1098154550031984,
    "2025-02-06": 1.1077416882416478,
    "2025-02-07": 1.10172209379375,
    "2025-02-10": 1.0811574244325035,
    "2025-02-11": 1.0734911751914464,
    "2025-02-12": 1.0661990940912474,
    "2025-02-13": 1.0602901008346977,
    "2025-02-14": 1.0518594713579732,
    "2025-02-17": 1.0518594713579732,
    "2025-02-18": 1.0518594713579732,
    "2025-02-19": 1.05,
    "2025-02-20": 1.05,
    "2025-02-21": 1.05,
    "2025-02-24": 1.05,
    "2025-02-25": 1.05,
    "2025-02-26": 1.0527588852628547,
    "2025-02-27": 1.0527588852628547,
    "2025-02-28": 1.0667066139505863,
    "2025-03-03": 1.0823867283185182,
    "2025-03-04": 1.1105522570590172,
    "2025-03-05": 1.1181620438583857,
    "2025-03-06": 1.1370798876795918,
    "2025-03-07": 1.141227962278765,
    "2025-03-10": 1.1468776313097577,
    "2025-03-11": 1.138346215854358,
    "2025-03-12": 1.1279775437921349,
    "2025-03-13": 1.1125778578744623,
    "2025-03-14": 1.1030270535682,
    "2025-03-17": 1.1048411380562067,
    "2025-03-18": 1.1043567781180605,
    "2025-03-19": 1.1252682302578454,
    "2025-03-20": 1.1534931901089411,
    "2025-03-21": 1.1723530638358626,
    "2025-03-24": 1.1728345245249774,
    "2025-03-25": 1.1735409116914925,
    "2025-03-26": 1.185907909717364,
    "2025-03-27": 1.1921685332191925,
    "2025-03-28": 1.1871588802174922,
    "2025-03-31": 1.2032085072332566,
    "2025-04-01": 1.2188120368872852,
    "2025-04-02": 1.2233490360707993,
    "2025-04-03": 1.2084665568792434,
    "2025-04-04": 1.2338112390392963,
    "2025-04-07": 1.2211594335468168,
    "2025-04-08": 1.2151971373633177,
    "2025-04-09": 1.1773816880141474,
    "2025-04-10": 1.1542604259492455,
    "2025-04-11": 1.1139254164434313,
    "2025-04-14": 1.0869022663441008,
    "2025-04-15": 1.057404892360103,
    "2025-04-16": 1.0574048923601027,
    "2025-04-17": 1.0574048923601027,
    "2025-04-18": 1.0574048923601027,
    "2025-04-21": 1.05,
    "2025-04-22": 1.05,
    "2025-04-23": 1.05,
    "2025-04-24": 1.05,
    "2025-04-25": 1.05,
    "2025-04-28": 1.05,
    "2025-04-29": 1.05,
    "2025-04-30": 1.05,
    "2025-05-01": 1.05,
    "2025-05-02": 1.05,
    "2025-05-05": 1.05,
    "2025-05-06": 1.05,
    "2025-05-07": 1.05,
    "2025-05-08": 1.05,
    "2025-05-09": 1.05,
    "2025-05-12": 1.05,
    "2025-05-13": 1.05,
    "2025-05-14": 1.0794162101026115,
    "2025-05-15": 1.095778191130626,
    "2025-05-16": 1.1117502874762297,
    "2025-05-19": 1.1474648906212317,
    "2025-05-20": 1.1874504567713928,
    "2025-05-21": 1.1764941574884689,
    "2025-05-22": 1.1683235868870145,
    "2025-05-23": 1.1734665775187423,
    "2025-05-26": 1.1734665775187423,
    "2025-05-27": 1.1437402882053995,
    "2025-05-28": 1.1037547220552382,
    "2025-05-29": 1.0852948112355507,
    "2025-05-30": 1.0771034008089901,
    "2025-06-02": 1.0598952520932134,
    "2025-06-03": 1.0799473972012035,
    "2025-06-04": 1.0819244906358894,
    "2025-06-05": 1.0819244906358894,
    "2025-06-06": 1.0819244906358894,
    "2025-06-09": 1.078017552374335,
    "2025-06-10": 1.0519770934346862,
    "2025-06-11": 1.05,
    "2025-06-12": 1.05,
    "2025-06-13": 1.05,
    "2025-06-16": 1.05,
    "2025-06-17": 1.05,
    "2025-06-18": 1.05,
    "2025-06-19": 1.05,
    "2025-06-20": 1.05,
    "2025-06-23": 1.0663776362219066,
    "2025-06-24": 1.078431115714403,
    "2025-06-25": 1.0890589467842435,
    "2025-06-26": 1.102436177535153,
    "2025-06-27": 1.1145593834629346,
    "2025-06-30": 1.1161909520823599,
    "2025-07-01": 1.1282363100544952,
    "2025-07-02": 1.1373684070786632,
    "2025-07-03": 1.123991176327754,
    "2025-07-04": 1.123991176327754,
    "2025-07-07": 1.1301921981219099,
    "2025-07-08": 1.1302730796262341,
    "2025-07-09": 1.1403175173004958,
    "2025-07-10": 1.1477520804417753,
    "2025-07-11": 1.1669023764446422,
    "2025-07-14": 1.1927229292070387,
    "2025-07-15": 1.213791808362442,
    "2025-07-16": 1.228697054854631,
    "2025-07-17": 1.2413718997637482,
    "2025-07-18": 1.264349338378295,
    "2025-07-21": 1.2596796829101906,
    "2025-07-22": 1.266850140237096,
    "2025-07-23": 1.2543805065420053,
    "2025-07-24": 1.2442157122334303,
    "2025-07-25": 1.2277050812927208,
    "2025-07-28": 1.2229115167593296,
    "2025-07-29": 1.1910475812210604,
    "2025-07-30": 1.1644174554699191,
    "2025-07-31": 1.138540482489244,
    "2025-08-01": 1.1129233788125399,
    "2025-08-04": 1.0782418183297013,
    "2025-08-05": 1.0662383917525442,
    "2025-08-06": 1.0562896295676936,
    "2025-08-07": 1.0524620607125388,
    "2025-08-08": 1.0524620607125388,
    "2025-08-11": 1.052562848560743,
    "2025-08-12": 1.0501007878482043,
    "2025-08-13": 1.0501007878482045,
    "2025-08-14": 1.0501007878482045,
    "2025-08-15": 1.0501007878482043,
    "2025-08-18": 1.05,
    "2025-08-19": 1.05,
    "2025-08-20": 1.05,
    "2025-08-21": 1.0517545809109357,
    "2025-08-22": 1.0517545809109357,
    "2025-08-25": 1.0517545809109357,
    "2025-08-26": 1.0517545809109357,
    "2025-08-27": 1.056979902057709,
    "2025-08-28": 1.0658564877330328,
    "2025-08-29": 1.0798408343999575,
    "2025-09-01": 1.0798408343999575,
    "2025-09-02": 1.1834816718804464,
    "2025-09-03": 1.2777327700772303,
    "2025-09-04": 1.3556855375487895,
    "2025-09-05": 1.3902537400251531,
    "2025-09-08": 1.4175657541894346,
    "2025-09-09": 1.3539229658539438,
    "2025-09-10": 1.2883378079210697,
    "2025-09-11": 1.223492495829482,
    "2025-09-12": 1.1859134430422094,
    "2025-09-15": 1.1764394746603482,
    "2025-09-16": 1.1781530681753565,
    "2025-09-17": 1.1746741420683333,
    "2025-09-18": 1.1665698806816642,
    "2025-09-19": 1.1650876690796008,
    "2025-09-22": 1.1545522833672037,
    "2025-09-23": 1.147102638484408,
    "2025-09-24": 1.1518916115105893,
    "2025-09-25": 1.1697243877295196,
    "2025-09-26": 1.1881065089302414,
    "2025-09-29": 1.1910063238537703,
    "2025-09-30": 1.19006681331715,
    "2025-10-01": 1.1863196947230583,
    "2025-10-02": 1.190767630158211,
    "2025-10-03": 1.1917409106409669,
    "2025-10-06": 1.239320052418702,
    "2025-10-07": 1.2900425243900688,
    "2025-10-08": 1.350044191799841,
    "2025-10-09": 1.4155815219365118,
    "2025-10-10": 1.4554521848226014,
    "2025-10-13": 1.4618729137939204,
    "2025-10-14": 1.4494077650403538,
    "2025-10-15": 1.4151649525286127,
    "2025-10-16": 1.3804594931169334,
    "2025-10-17": 1.3686910656100202,
    "2025-10-20": 1.3463234054350852,
    "2025-10-21": 1.330005180136564,
    "2025-10-22": 1.3285107937616245,
    "2025-10-23": 1.3065595011625537,
    "2025-10-24": 1.2808742110644082,
    "2025-10-27": 1.2479536722061817,
    "2025-10-28": 1.223570965810866,
    "2025-10-29": 1.1730776286987976,
    "2025-10-30": 1.1357144138756001,
    "2025-10-31": 1.1118661435896722,
    "2025-11-03": 1.0983888534631865,
    "2025-11-04": 1.086556842713385,
    "2025-11-05": 1.106072055745631,
    "2025-11-06": 1.10812303808707,
    "2025-11-07": 1.1174064829413093,
    "2025-11-10": 1.173041214589849,
    "2025-11-11": 1.2188447395525963,
    "2025-11-12": 1.2602731530926001,
    "2025-11-13": 1.3016345065679933,
    "2025-11-14": 1.3614004718628738,
    "2025-11-17": 1.3801360148221913,
    "2025-11-18": 1.404077102506826,
    "2025-11-19": 1.441791056008246,
    "2025-11-20": 1.4639731014004838,
    "2025-11-21": 1.4570168251068014,
    "2025-11-24": 1.4285573200740698,
    "2025-11-25": 1.3750260218142643,
    "2025-11-26": 1.3093542001553098,
    "2025-11-27": 1.3093542001553098,
    "2025-11-28": 1.2889399357136055,
    "2025-12-01": 1.2549493100947469,
    "2025-12-02": 1.226783978261806,
    "2025-12-03": 1.204130335684518,
    "2025-12-04": 1.1723642588247158,
    "2025-12-05": 1.1436682883124707,
    "2025-12-08": 1.1658046303987932,
    "2025-12-09": 1.2037277882206738,
    "2025-12-10": 1.2291187017057204,
    "2025-12-11": 1.2653685860056947,
    "2025-12-12": 1.2874316118154479,
    "2025-12-15": 1.299369992463475,
    "2025-12-16": 1.2771217580465088,
    "2025-12-17": 1.266653426417746,
    "2025-12-18": 1.2407560127082284,
    "2025-12-19": 1.202482465614463,
    "2025-12-22": 1.1514903056440418,
    "2025-12-23": 1.1330208405083104,
    "2025-12-24": 1.1205867134224081,
    "2025-12-25": 1.1205867134224081,
    "2025-12-26": 1.129376625707017,
    "2025-12-29": 1.1450220212975482,
    "2025-12-30": 1.168896816927209,
    "2025-12-31": 1.2178257693194112,
    "2026-01-01": 1.2178257693194112,
    "2026-01-02": 1.2672201739323978,
    "2026-01-05": 1.3254739748374231,
    "2026-01-06": 1.410434788880458,
    "2026-01-07": 1.4673824114020138,
    "2026-01-08": 1.4845254271630686,
    "2026-01-09": 1.4867888628014676,
    "2026-01-12": 1.483205278781682,
    "2026-01-13": 1.4463965394566998,
    "2026-01-14": 1.4250327148917799,
    "2026-01-15": 1.3958557694451401,
    "2026-01-16": 1.3681205373626542,
    "2026-01-19": 1.3681205373626542,
    "2026-01-20": 1.3443261118924448,
    "2026-01-21": 1.2961158656299427,
    "2026-01-22": 1.2335506181190665,
    "2026-01-23": 1.1834411578835566,
    "2026-01-26": 1.1494576924969468,
    "2026-01-27": 1.1112692302389344,
    "2026-01-28": 1.1067700145358956,
    "2026-01-29": 1.121712347656001,
    "2026-01-30": 1.1316646782554285,
    "2026-02-02": 1.1478182746638854,
    "2026-02-03": 1.1509329416038334,
    "2026-02-04": 1.1555350748796773,
    "2026-02-05": 1.1426225597558002,
    "2026-02-06": 1.1501145097977863,
    "2026-02-09": 1.1639269354453412,
    "2026-02-10": 1.176241262026614,
    "2026-02-11": 1.1710505556884758,
    "2026-02-12": 1.1785577014742423,
    "2026-02-13": 1.1717694197361075,
    "2026-02-16": 1.1717694197361075,
    "2026-02-17": 1.1568197264524385,
    "2026-02-18": 1.1539189434526085,
    "2026-02-19": 1.212581412419646,
    "2026-02-20": 1.2537157283844516,
    "2026-02-23": 1.2937895196095242,
    "2026-02-24": 1.3232932464254659,
    "2026-02-25": 1.3343032599028244,
    "2026-02-26": 1.3025099152265107,
    "2026-02-27": 1.3165819413256585,
    "2026-03-02": 1.2826831164714314,
    "2026-03-03": 1.276874631680339,
    "2026-03-04": 1.261812323083063,
    "2026-03-05": 1.2364560382451262,
    "2026-03-06": 1.1963634811506727,
    "2026-03-09": 1.2139584686236975,
    "2026-03-10": 1.204821386111337,
    "2026-03-11": 1.2058323268643307,
    "2026-03-12": 1.2581654059962069,
    "2026-03-13": 1.279147518433253,
    "2026-03-16": 1.2754314520565706,
    "2026-03-17": 1.2728698186827578,
    "2026-03-18": 1.2808688982615357,
    "2026-03-19": 1.254018595911747,
    "2026-03-20": 1.2742234823976957,
    "2026-03-23": 1.268929470976707,
    "2026-03-24": 1.2943996151020412,
    "2026-03-25": 1.2920768842436086,
    "2026-03-26": 1.3082345691117472,
    "2026-03-27": 1.286712467366844,
    "2026-03-30": 1.2809174023964396,
    "2026-03-31": 1.2317868084654462,
    "2026-04-01": 1.2035742100036362,
    "2026-04-02": 1.145621258551886,
    "2026-04-03": 1.145621258551886,
    "2026-04-06": 1.0891650141945042,
    "2026-04-07": 1.0583865424830714,
    "2026-04-08": 1.05,
    "2026-04-09": 1.05,
    "2026-04-10": 1.05,
    "2026-04-13": 1.05,
    "2026-04-14": 1.05,
    "2026-04-15": 1.05,
    "2026-04-16": 1.05,
    "2026-04-17": 1.05,
    "2026-04-20": 1.05,
    "2026-04-21": 1.05,
    "2026-04-22": 1.05,
    "2026-04-23": 1.05,
    "2026-04-24": 1.05,
    "2026-04-27": 1.080501843483694,
    "2026-04-28": 1.1133367780098369,
    "2026-04-29": 1.1997492893715063,
    "2026-04-30": 1.2785936026982505,
    "2026-05-01": 1.3492875050436541,
    "2026-05-04": 1.3926303630041756,
    "2026-05-05": 1.4243200282687916,
    "2026-05-06": 1.3629233592843704,
    "2026-05-07": 1.32276873974391,
    "2026-05-08": 1.2692770791882126,
    "2026-05-11": 1.218755460269994,
    "2026-05-12": 1.17477005134649,
    "2026-05-13": 1.175280703301362,
    "2026-05-14": 1.1575942598832025,
    "2026-05-15": 1.1456615210161378,
    "2026-05-18": 1.1622890960335102,
    "2026-05-19": 1.1781918588443157,
    "2026-05-20": 1.1526653645121956,
    "2026-05-21": 1.1316621141440708,
    "2026-05-22": 1.1316621141440708
  }
}

Writing dealer_markup.json


In [ ]:
%%writefile pure_volatility_trading.py
import warnings
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model

warnings.filterwarnings("ignore")

GARCH_SIGNAL_PERCENTILE: int = 90
STATIC_DEALER_MARKUP: float = 1.00
MIN_OBS_FOR_GARCH: int = 100
DEALER_PRICING_WINDOW: int = 90
INVESTMENT_CAPITAL: float = 100000.0
BASE_OU_KAPPA: float = 25.0
FALLEN_ANGEL_OU_KAPPA: float = 40.0

def get_trade_size():
    trade_size = input("Enter the Hedge Fund trade size (dollars), Note: round it to the nearest integer: ")
    while not trade_size.isdigit() or int(trade_size) < 0:
        trade_size = input("Enter the Hedge Fund trade size, Note: round it to the nearest integer: ")
    return int(trade_size) / 1000000.0

def get_hold_days():
    hold_days = input("Enter the number of hold days (positive integer): ")
    while not hold_days.isdigit() or int(hold_days) <= 0:
        hold_days = input("Please enter a valid positive integer for hold days: ")
    return int(hold_days)

def prepare_pristine_data(raw_filepath: str) -> pd.DataFrame:
    if not os.path.exists(raw_filepath):
        raise FileNotFoundError(f"Could not find {raw_filepath}")

    with open(raw_filepath, "r") as f:
        data = json.load(f)

    df_raw = pd.DataFrame(data)
    df_raw.index = pd.to_datetime(df_raw.index)
    df_raw = df_raw.sort_index().resample("B").last().ffill()

    # --- THE FIX: Unconditional conversion to Basis Points ---
    df_raw = df_raw * 100.0

    return df_raw

def get_treasury_rates(df_raw: pd.DataFrame) -> pd.Series:
    try:
        start = df_raw.index.min().strftime("%Y-%m-%d")
        end = df_raw.index.max().strftime("%Y-%m-%d")
        raw = yf.download("^IRX", start=start, end=end, progress=False)
        rates = raw["Close"] / 100.0
        return rates.squeeze() if isinstance(rates, pd.DataFrame) else rates
    except Exception:
        return pd.Series(0.04, index=df_raw.index)

def get_dealer_volatility(series: pd.Series, window: int = DEALER_PRICING_WINDOW) -> pd.Series:
    returns = series.diff().dropna()
    dealer_vol = returns.rolling(window).std().shift(1)
    return dealer_vol.reindex(series.index).ffill().bfill()

def fit_garch_volatility(series: pd.Series, label: str = "", min_obs: int = MIN_OBS_FOR_GARCH) -> pd.Series:
    returns = series.diff().dropna()
    if len(returns) < min_obs: return returns.rolling(DEALER_PRICING_WINDOW).std().reindex(series.index).ffill().bfill()
    try:
        mdl = arch_model(returns, vol="GARCH", p=1, q=1, mean="AR", lags=1, dist="Normal", rescale=True)
        res = mdl.fit(disp="off", show_warning=False)
        params = res.params
        alpha, beta = params.get('alpha[1]', 0), params.get('beta[1]', 0)
        if alpha + beta >= 1.0: raise ValueError(f"GARCH non-stationary for {label}")
        return res.conditional_volatility.reindex(series.index).ffill().bfill()
    except Exception:
        return returns.rolling(DEALER_PRICING_WINDOW).std().reindex(series.index).ffill().bfill()

def calc_ou_straddle(dealer_vol_series: pd.Series, r_series: pd.Series, markup_series: pd.Series, T_days: int, kappa: float) -> pd.Series:
    T = T_days / 252.0
    annualized_normal_vol = dealer_vol_series * np.sqrt(252)
    KAPPA_FLOOR = 1e-6
    if kappa < KAPPA_FLOOR: ou_variance = (annualized_normal_vol ** 2) * T
    else: ou_variance = (annualized_normal_vol ** 2 / (2 * kappa)) * (1 - np.exp(-2 * kappa * T))
    safe_ou_variance = np.where(ou_variance > 0, ou_variance, 1e-12)
    effective_vol = np.sqrt(safe_ou_variance / T)
    priced_vol = effective_vol * markup_series
    discount_factor = np.exp(-r_series * T)
    sqrt_2_over_pi = np.sqrt(2 / np.pi)
    return discount_factor * priced_vol * np.sqrt(T) * sqrt_2_over_pi

def calculate_dynamic_pb_markup(retail_markup_series: pd.Series, daily_vol_bps: pd.Series, trade_size_millions: float = 50.0) -> pd.Series:
    base_discount = 0.05
    volume_discount = np.log10(max(1.0, trade_size_millions)) * 0.05
    annualized_vol_bps = daily_vol_bps * np.sqrt(252)
    safe_vol_threshold = 100.0
    illiquidity_penalty = np.maximum(0.0, (annualized_vol_bps - safe_vol_threshold) / 1000.0)
    total_discount = base_discount + volume_discount - illiquidity_penalty
    total_discount = np.clip(total_discount, 0.0, 0.25)
    return retail_markup_series * (1.0 - total_discount)

def calculate_dynamic_execution_friction(base_spread_bps: float, macro_vol_proxy: pd.Series, shock_percentile: float = 90.0, growth_rate: float = 0.08) -> pd.Series:
    threshold_vol = np.percentile(macro_vol_proxy.dropna(), shock_percentile)
    excess_vol = np.maximum(0, macro_vol_proxy - threshold_vol)
    dynamic_friction_bps = base_spread_bps * np.exp(growth_rate * excess_vol)
    return pd.Series(dynamic_friction_bps, index=macro_vol_proxy.index)

def analyze_strategy(df_raw: pd.DataFrame, r_daily: pd.Series, retail_markup: pd.Series, percentile_override: float = 90.0, kappa_override: float = 25.0, trade_size: float = 50.0, hold_days: int = 5) -> list[dict]:
    skip_cols = ["Risk_Free", "EUR_Risk_Free_10Y"]
    assets = [col for col in df_raw.columns if col not in skip_cols]
    results = []
    ts = trade_size

    for asset in assets:
        print(f"\n{'─' * 60}\n  Processing Pure Volatility: {asset}")

        if asset == "Fallen_Angel" or asset in ["B", "CCC"]:
            current_kappa = FALLEN_ANGEL_OU_KAPPA if kappa_override == 25.0 else kappa_override
            asset_base_retail_markup = retail_markup * 1.20
            print(f"  [Distressed Rules Applied] Kappa: {current_kappa} | Liquidity Premium: +20%")
        else:
            current_kappa = kappa_override
            asset_base_retail_markup = retail_markup
            print(f"  [Standard Rules Applied] Kappa: {current_kappa}")

        asset_garch_vol = fit_garch_volatility(df_raw[asset], label=f"{asset} index")
        vol_threshold = float(np.percentile(asset_garch_vol.dropna(), percentile_override))

        shock_days = asset_garch_vol >= vol_threshold
        actual_freq = shock_days.sum() / max(len(shock_days), 1)

        asset_dealer_vol = get_dealer_volatility(df_raw[asset])
        asset_hf_markup = calculate_dynamic_pb_markup(retail_markup_series=asset_base_retail_markup, daily_vol_bps=asset_dealer_vol, trade_size_millions= ts)

        gross_hd = (df_raw[asset].shift(-hold_days) - df_raw[asset]).abs()

        valid_shock_gross = gross_hd[shock_days].dropna()
        valid_norm_gross = gross_hd[~shock_days].dropna()
        gross_shock = float(valid_shock_gross.mean()) if not valid_shock_gross.empty else 0.0
        gross_normal = float(valid_norm_gross.mean()) if not valid_norm_gross.empty else 0.0

        hf_straddle_cost = calc_ou_straddle(asset_dealer_vol, r_daily, asset_hf_markup, T_days=hold_days, kappa=current_kappa)

        valid_hf_shock_fee = hf_straddle_cost[shock_days].dropna()
        valid_hf_norm_fee = hf_straddle_cost[~shock_days].dropna()
        hf_fee_shock = float(valid_hf_shock_fee.mean()) if not valid_hf_shock_fee.empty else 0.0
        hf_fee_normal = float(valid_hf_norm_fee.mean()) if not valid_hf_norm_fee.empty else 0.0

        friction_penalty = calculate_dynamic_execution_friction(base_spread_bps=1.0, macro_vol_proxy=asset_dealer_vol, shock_percentile=90.0)

        valid_friction = friction_penalty[shock_days].dropna()
        avg_shock_penalty = float(valid_friction.mean()) if not valid_friction.empty else 0.0

        hf_net_shock = gross_shock - hf_fee_shock - avg_shock_penalty
        hf_net_normal = gross_normal - hf_fee_normal
        hf_mult = hf_net_shock / abs(hf_net_normal) if hf_net_normal else 0.0

        ret_straddle_cost = calc_ou_straddle(asset_dealer_vol, r_daily, asset_base_retail_markup, T_days=hold_days, kappa=current_kappa)

        valid_ret_shock_fee = ret_straddle_cost[shock_days].dropna()
        valid_ret_norm_fee = ret_straddle_cost[~shock_days].dropna()
        ret_fee_shock = float(valid_ret_shock_fee.mean()) if not valid_ret_shock_fee.empty else 0.0
        ret_fee_normal = float(valid_ret_norm_fee.mean()) if not valid_ret_norm_fee.empty else 0.0

        ret_gross_shock = gross_shock
        ret_gross_normal = gross_normal

        ret_net_shock = ret_gross_shock - ret_fee_shock - avg_shock_penalty
        ret_net_normal = ret_gross_normal - ret_fee_normal
        ret_mult = ret_net_shock / abs(ret_net_normal) if ret_net_normal else 0.0

        results.append({
            "Asset": asset, "Freq": actual_freq, "VolThreshold": vol_threshold,
            "HF_Gross_Shock": gross_shock, "HF_Gross_Normal": gross_normal,
            "HF_Fee_Shock": hf_fee_shock, "HF_Fee_Normal": hf_fee_normal,
            "HF_Net_Shock": hf_net_shock, "HF_Net_Normal": hf_net_normal, "HF_Mult": hf_mult,
            "Ret_Gross_Shock": ret_gross_shock, "Ret_Gross_Normal": ret_gross_normal,
            "Ret_Fee_Shock": ret_fee_shock, "Ret_Fee_Normal": ret_fee_normal,
            "Ret_Net_Shock": ret_net_shock, "Ret_Net_Normal": ret_net_normal, "Ret_Mult": ret_mult,
        })
    return results

def plot_summary_table(results: list[dict], investment: float = INVESTMENT_CAPITAL, hold_days = 5) -> None:
    columns = [
        "Asset", "Market\nEnvironment",
        "Gross\nReturn (%)", f"Gross Payout\n(${investment / 1000:.0f}k)",
        "HF Net\nReturn (%)", f"HF Net Profit\n(${investment / 1000:.0f}k)",
        "Retail Net\nReturn (%)", f"Retail Net Profit\n(${investment / 1000:.0f}k)"
    ]

    def fmt_pct(val): return "N/A" if pd.isna(val) else f"{val / 100.0:+.2f}%"
    def fmt_usd(val): return "N/A" if pd.isna(val) else f"${val * (investment / 10000.0):+,.0f}"
    def get_color(val):
        if pd.isna(val): return "#FFFFFF"
        return "#E6F4EA" if val > 0 else "#FCE8E6"

    cell_text = []
    colors = []

    for r in results:
        freq_shock = r['Freq'] * 100
        freq_norm = (1 - r['Freq']) * 100
        asset_label = f"★ {r['Asset']}" if r['Asset'] == "Fallen_Angel" else r['Asset']

        row_shock = [
            asset_label, f"Shock (Top {freq_shock:.0f}%)",
            fmt_pct(r['HF_Gross_Shock']), fmt_usd(r['HF_Gross_Shock']),
            fmt_pct(r['HF_Net_Shock']), fmt_usd(r['HF_Net_Shock']),
            fmt_pct(r['Ret_Net_Shock']), fmt_usd(r['Ret_Net_Shock'])
        ]
        cell_text.append(row_shock)
        colors.append([
            "w", "#f8d7da", "#F0F8FF", "#F0F8FF",
            get_color(r['HF_Net_Shock']), get_color(r['HF_Net_Shock']),
            get_color(r['Ret_Net_Shock']), get_color(r['Ret_Net_Shock'])
        ])

        row_norm = [
            "", f"Normal (Bottom {freq_norm:.0f}%)",
            fmt_pct(r['HF_Gross_Normal']), fmt_usd(r['HF_Gross_Normal']),
            fmt_pct(r['HF_Net_Normal']), fmt_usd(r['HF_Net_Normal']),
            fmt_pct(r['Ret_Net_Normal']), fmt_usd(r['Ret_Net_Normal'])
        ]
        cell_text.append(row_norm)
        colors.append([
            "w", "#e2e3e5", "#F0F8FF", "#F0F8FF",
            get_color(r['HF_Net_Normal']), get_color(r['HF_Net_Normal']),
            get_color(r['Ret_Net_Normal']), get_color(r['Ret_Net_Normal'])
        ])

    fig, ax = plt.subplots(figsize=(14, max(8, len(results) * 1.2)))
    ax.axis('off')
    ax.axis('tight')

    table = ax.table(cellText=cell_text, cellColours=colors, colLabels=columns, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.8)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('#1D3557')
        elif col == 0 and row % 2 != 0:
            cell.set_text_props(weight='bold')
            if "★" in cell.get_text().get_text():
                cell.set_text_props(color='#D4AF37')
        elif col == 1:
            cell.set_text_props(style='italic')

    plt.title(f"Pure Volatility Returns per ${investment:,.0f} Invested" + ' and held for ' + str(hold_days) + ' days', fontweight="bold", fontsize=15, pad=20)
    plt.tight_layout()
    plt.savefig("pure_vol_summary_table.png", dpi=150, bbox_inches="tight")
    plt.show()

def plot_scenario(results: list[dict], mode: str, hold_days = 5) -> None:
    assets = [r.get("Asset", r.get("Pair")) for r in results]
    x, width = np.arange(len(assets)), 0.35

    fig, ax = plt.subplots(figsize=(16, 7))

    normal_fee_vals = None
    if mode == "Gross":
        normal_vals = [r.get("HF_Gross_Normal", 0) if not np.isnan(r.get("HF_Gross_Normal", 0)) else 0 for r in results]
        shock_vals = [r.get("HF_Gross_Shock", 0) if not np.isnan(r.get("HF_Gross_Shock", 0)) else 0 for r in results]
        title, filename = "TIER 1: Theoretical Gross Payout", f"pure_{mode.lower()}_payout.png"
        shock_colors, normal_colors = ["#E63946"] * len(shock_vals), ["#457B9D"] * len(normal_vals)
    elif mode == "HedgeFund":
        normal_vals = [r.get("HF_Net_Normal", 0) if not np.isnan(r.get("HF_Net_Normal", 0)) else 0 for r in results]
        shock_vals = [r.get("HF_Net_Shock", 0) if not np.isnan(r.get("HF_Net_Shock", 0)) else 0 for r in results]
        title, filename = "TIER 2: Hedge Fund Net Returns", f"pure_{mode.lower()}_returns.png"
        shock_colors = ["#2A9D8F" if v > 0 else "#E63946" for v in shock_vals]
        normal_colors = ["#A8DADC" if v > 0 else "#F4A261" for v in normal_vals]
        normal_fee_vals = [r.get("HF_Fee_Normal", 0) for r in results]
    elif mode == "Retail":
        normal_vals = [r.get("Ret_Net_Normal", 0) if not np.isnan(r.get("Ret_Net_Normal", 0)) else 0 for r in results]
        shock_vals = [r.get("Ret_Net_Shock", 0) if not np.isnan(r.get("Ret_Net_Shock", 0)) else 0 for r in results]
        title, filename = "TIER 3: Retail Net Returns", f"pure_{mode.lower()}_returns.png"
        shock_colors = ["#2A9D8F" if v > 0 else "#E63946" for v in shock_vals]
        normal_colors = ["#A8DADC" if v > 0 else "#F4A261" for v in normal_vals]
        normal_fee_vals = [r.get("Ret_Fee_Normal", 0) for r in results]

    ax.bar(x - width / 2, normal_vals, width, label="Normal Day", color=normal_colors, alpha=0.6)
    ax.bar(x + width / 2, shock_vals, width, label="Post-Shock Trade", color=shock_colors)

    all_plotted_vals = [v for v in normal_vals + shock_vals if not np.isnan(v)]
    if all_plotted_vals:
        min_y = min(all_plotted_vals)
        max_y = max(all_plotted_vals)
        max_abs = max(abs(min_y), abs(max_y))
        if max_abs < 10.0: max_abs = 10.0
        limit = max_abs * 1.40
        ax.set_ylim(-limit, limit)

    ax.set_ylabel("Net Profit (bps)", fontweight="bold")
    ax.set_title(title + ' after being held for ' + str(hold_days) + ' days', fontsize=15, fontweight="bold", pad=20)
    ax.set_xticks(x)
    labels = ["★ " + a if "Fallen_Angel" in a else a for a in assets]
    ax.set_xticklabels(labels, fontweight="bold", fontsize=11)

    ax.axhline(0, color="black", linewidth=1.5, zorder=3)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="upper left")

    for i, r in enumerate(results):
        fee_shock = r.get('HF_Fee_Shock') if mode == "HedgeFund" else r.get('Ret_Fee_Shock')
        y_val = shock_vals[i]
        va_algn = "bottom" if y_val >= 0 else "top"
        pixel_offset = 10 if y_val >= 0 else -10

        if mode == "Gross":
            normal_divisor = max(abs(r.get('HF_Gross_Normal', 1)), 0.0001)
            text = f"Gross\nMult: {(r['HF_Gross_Shock'] / normal_divisor):.2f}x" if r.get('HF_Gross_Normal') else ""
        else:
            mult = r['HF_Mult'] if mode == "HedgeFund" else r['Ret_Mult']
            text = f"Fee (Shock): {fee_shock:.1f} bps\nMult: {mult:.2f}x"

        ax.annotate(text, xy=(x[i] + width / 2, y_val), xytext=(4, pixel_offset), textcoords="offset points", ha="left", va=va_algn, fontsize=8, bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", pad=1.5))

        if normal_fee_vals is not None and mode != "Gross":
            norm_fee = normal_fee_vals[i]
            if not np.isnan(norm_fee):
                y_norm = normal_vals[i]
                va_algn_norm = "bottom" if y_norm >= 0 else "top"
                pixel_offset_norm = 10 if y_norm >= 0 else -10
                norm_text = f"Fee (Normal): {norm_fee:.1f} bps"
                ax.annotate(norm_text, xy=(x[i] - width / 2, y_norm), xytext=(-4, pixel_offset_norm), textcoords="offset points", ha="right", va=va_algn_norm, fontsize=8, bbox=dict(facecolor="white", alpha=0.85, edgecolor="none", pad=1.5))

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()

Writing pure_volatility_trading.py


In [ ]:
%%writefile expected_loss_by_grade.json
{
  "AAA": 0.00536,
  "AA": 0.00536,
  "A": 0.0055476,
  "BBB": 0.005829,
  "BB": 0.0151548,
  "B": 0.0181332,
  "CCC": 0.0199144
}

Writing expected_loss_by_grade.json


In [ ]:
import os
import importlib
import main
import data_engine
import ml_engine
import main_alpha

# Ensure API Key is bound
try:
    from google.colab import userdata
    os.environ["FRED_API_KEY"] = userdata.get('FRED_API_KEY')
except Exception:
    os.environ["FRED_API_KEY"] = "ccb67ba570ac152dc7930a216488320d"

# Flush memory
importlib.reload(data_engine)
importlib.reload(ml_engine)
importlib.reload(main_alpha)
importlib.reload(main)

# Run
main.main()

  GLOBAL CREDIT VOLATILITY & AI ENGINE (CLOUD INSTANCE)
  [1] USA Public Corporate Bonds (FRED) - FULL PIPELINE
  [2] EUR Public Corporate Bonds (ETFs) - MATRIX ONLY
  [3] Global Private Credit Pipeline (AlphaCredit Modules 1 & 2)
  [4] Emerging Markets (USD & Local) - MATRIX ONLY
  [5] Liquid ML Credit Scorecard (XGBoost + SHAP Matrix)
  [6] Private Credit Super-Learner & Risk Sim (Modules 3 & 4)
---------------------------------------------------------------------------
  Select Market Architecture [1/2/3/4/5/6]: 6

  [ROUTING] Booting Institutional DeepHazard Neural Cox Pipeline...
    -> Neural Cox Forward Pass Complete. Hazards calculated for 500 deals.

  [ALPHACREDIT.AI] Booting Module 4: Front-End UI DataFrames & Risk Sim...
    -> Live risk-free: 4.1800% | Credit spread: 2.3200% | Implied LGD: 29.9355% | Synthetic Recovery: 70.1%
    -> Synthetic Market-Implied Recovery Rate Calculated: 70.1%
  [Discount Rate] 5Y Treasury (DGS5): 4.1800% + OAS (HY): 274 bps = 6.9200% (source: 